## Simulation Functions

In [1]:
def normalize_boxscore_variables(df):
    df_ = df.copy()
    # FGM > [36, 45]
    df_[['home_FGM', 'away_FGM']] = (df_[['home_FGM', 'away_FGM']] - 36)/(45 - 36)
    # FGA > [82, 95]
    df_[['home_FGA', 'away_FGA']] = (df_[['home_FGA', 'away_FGA']] - 82)/(95 - 82)
    # FGP > [0.42, 0.51]
    df_[['home_FGP', 'away_FGP']] = (df_[['home_FGP', 'away_FGP']] - 0.42)/(0.51 - 0.42)
    # 3FGM > [8, 17]    
    df_[['home_3FGM', 'away_3FGM']] = (df_[['home_3FGM', 'away_3FGM']] - 8)/(17 - 8)
    # 3FGA > [25, 44]
    df_[['home_3FGA', 'away_3FGA']] = (df_[['home_3FGA', 'away_3FGA']] - 25)/(44 - 25)
    # 3FGP > [0.3, 0.4]
    df_[['home_3FGP', 'away_3FGP']] = (df_[['home_3FGP', 'away_3FGP']] - 0.3)/(0.4 - 0.3)
    # FTM > [13, 23]    
    df_[['home_FTM', 'away_FTM']] = (df_[['home_FTM', 'away_FTM']] - 13)/(23 - 13)
    # FTA > [18, 28]
    df_[['home_FTA', 'away_FTA']] = (df_[['home_FTA', 'away_FTA']] - 18)/(28 - 18)
    # FTP > [0.7, 0.85]    
    df_[['home_FTP', 'away_FTP']] = (df_[['home_FTP', 'away_FTP']] - 0.7)/(0.85 - 0.7)
    # RebOff > [8, 14]    
    df_[['home_RebOff', 'away_RebOff']] = (df_[['home_RebOff', 'away_RebOff']] - 8)/(14 - 8)
    # RebDef > [30, 40]    
    df_[['home_RebDef', 'away_RebDef']] = (df_[['home_RebDef', 'away_RebDef']] - 30)/(40 - 30)
    # RebTot > [40 , 50] 
    df_[['home_RebTot', 'away_RebTot']] = (df_[['home_RebTot', 'away_RebTot']] - 40)/(50 - 40)
    # AST > [20, 30]    
    df_[["home_AST", "away_AST"]] = (df_[["home_AST", "away_AST"]] - 20)/(30 - 20)
    # STL > [5, 10]
    df_[["home_STL", "away_STL"]] = (df_[["home_STL", "away_STL"]] - 5)/(10 - 5)
    # BLK > [3, 7]    
    df_[["home_BLK", "away_BLK"]] = (df_[["home_BLK", "away_BLK"]] - 3)/(7 - 3)
    # TO > [10, 17]    
    df_[["home_TO", "away_TO"]] = (df_[["home_TO", "away_TO"]] - 10)/(17 - 10)
    # FP > [17, 23]    
    df_[["home_FP", "away_FP"]] = (df_[["home_FP", "away_FP"]] - 17)/(23 - 17)
    # Points > [100, 124]
    df_[["home_points", "away_points"]] = (df_[["home_points", "away_points"]] - 100)/(124 - 100)
    # PMP > [-10, 10]
    df_[["home_PMP", "away_PMP"]] = (df_[["home_PMP", "away_PMP"]] - (-10))/(10 - (-10))
    return df_

In [2]:
def put_categorical_training_outcomes(df):
    h, a = 'home_outcome', 'away_outcome'
    return df\
    .assign(ctgc = 0)\
    .assign(ctgc = lambda x: [2 if i == 0 and j >= 0.5 and k >= 0.5 else i for i, j, k in zip(x['ctgc'], x[h], x[a])])\
    .assign(ctgc = lambda x: [3 if i == 0 and j >= 0.5 and k < 0.5 else i for i, j, k in zip(x['ctgc'], x[h], x[a])])\
    .assign(ctgc = lambda x: [4 if i == 0 and j < 0.5 and k >= 0.5 else i for i, j, k in zip(x['ctgc'], x[h], x[a])])\
    .assign(ctgc = lambda x: [5 if i == 0 and j < 0.5 and k < 0.5 else i for i, j, k in zip(x['ctgc'], x[h], x[a])])\
    .assign(ctgc = lambda x: [1 if i >= 0.4 and i <= 0.6 and j >= 0.4 and j <= 0.6 else k for i, j, k in zip(x[h], x[a], x['ctgc'])])

In [ ]:
class NGramModel_ExtraInfo(nn.Module):
    """
    A Neural Network designed for N-Gram modeling that incorporates additional 
    numerical input features into the model architecture alongside the standard 
    token embeddings, with optional Layer Normalization and Dropout.

    Attributes:
        embedding (nn.Embedding): Embedding layer to convert token indices into 
        dense vectors.
        flatten (nn.Flatten): Flattens the output of the embedding layer to 
        prepare for linear layers.
        network (nn.Sequential): A sequence of linear and activation layers 
        culminating in the output layer.

    Parameters:
        context_size (int): The number of tokens used for context in embeddings.
        vocab_size (int): The size of the vocabulary in the embedding layer.
        embedding_dim (int): The dimensionality of the embedding vectors.
        numerical_input_dim (int): The dimensionality of the numerical feature vectors.
        num_hidden_neurons (int or list): The number of neurons in each hidden layer.
                                          If a list, specifies neurons for each layer.
        num_layers (int): The total number of hidden layers in the model.
        activation (callable): The activation function used in the network.
        dropout_rate (float): Dropout probability (0.0 means no dropout).
        use_layer_norm (bool): Whether to apply Layer Normalization in the layers.
    """
    def __init__(self, context_size, vocab_size, embedding_dim, numerical_input_dim, 
                 num_hidden_neurons, num_layers, activation = nn.Tanh, dropout_rate = 0.0, use_layer_norm = False):
        super(NGramModel_ExtraInfo, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        # Initialize the network layers
        layers = self._init_layers(context_size, vocab_size, embedding_dim, numerical_input_dim, 
                                   num_hidden_neurons, num_layers, activation, dropout_rate, use_layer_norm)

        # Use nn.Sequential to manage the network layers
        self.network = nn.Sequential(*layers)

    def _init_layers(self, context_size, vocab_size, embedding_dim, numerical_input_dim, 
                     num_hidden_neurons, num_layers, activation, dropout_rate, use_layer_norm):
        """
        Initializes the layers of the network based on the provided specifications.
        """
        layers = []
        input_size = embedding_dim * context_size + numerical_input_dim

        # Handle num_hidden_neurons as a list or single value
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Append the first layer with optional LayerNorm and Dropout
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation())
        layers.append(nn.Dropout(dropout_rate))

        # Append subsequent hidden layers with optional LayerNorm and Dropout
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]
            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))            
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))

        # Output layer
        layers.append(nn.Linear(num_hidden_neurons[-1], vocab_size))
        return layers

    def forward(self, train_token, train_numerical):
        """
        Forward pass of the model.

        Parameters:
            train_token (torch.Tensor): Tensor of token indices.
            train_numerical (torch.Tensor): Tensor of numerical features.

        Returns:
            torch.Tensor: The output logits from the final layer of the network.
        """
        # Embed and flatten token inputs
        embedded = self.embedding(train_token)
        flattened = self.flatten(embedded)

        # Concatenate embedded tokens with numerical inputs
        concatenated = torch.cat((flattened, train_numerical), dim=1)

        # Pass through the network and return output
        return self.network(concatenated)
    
class NGramModel_ExtraInfo_State(nn.Module):
    """
    Versão do NGramModel_ExtraInfo que inclui um embedding adicional
    para um token de estado (por exemplo, combinação tempo × diferença de pontos).

    Estrutura:
        - Embedding para tokens (sequência de contexto).
        - Embedding para estado (state_id, ex.: 0..224).
        - Flatten dos embeddings de tokens.
        - Concatenação: [tokens_flattened, state_embed, numerical_features]
        - MLP (network) até o output vocab_size.
    """
    def __init__(
        self,
        context_size,
        vocab_size,
        embedding_dim,
        numerical_input_dim,
        state_vocab_size,
        state_embedding_dim,
        num_hidden_neurons,
        num_layers,
        activation = nn.Tanh,
        dropout_rate = 0.0,
        use_layer_norm = False,
    ):
        super(NGramModel_ExtraInfo_State, self).__init__()

        # Embedding dos tokens (jogadas)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        # Embedding do estado (ex.: 225 estados tempo×diferença)
        self.state_embedding = nn.Embedding(state_vocab_size, state_embedding_dim)

        # Inicializa a MLP principal
        layers = self._init_layers(
            context_size = context_size,
            vocab_size = vocab_size,
            embedding_dim = embedding_dim,
            numerical_input_dim = numerical_input_dim,
            state_embedding_dim = state_embedding_dim,
            num_hidden_neurons = num_hidden_neurons,
            num_layers = num_layers,
            activation = activation,
            dropout_rate = dropout_rate,
            use_layer_norm = use_layer_norm,
        )

        self.network = nn.Sequential(*layers)

    def _init_layers(
        self,
        context_size,
        vocab_size,
        embedding_dim,
        numerical_input_dim,
        state_embedding_dim,
        num_hidden_neurons,
        num_layers,
        activation,
        dropout_rate,
        use_layer_norm,
    ):
        """
        Inicializa as camadas da MLP principal, levando em conta:
        - embeddings dos tokens (context_size * embedding_dim)
        - embedding do estado (state_embedding_dim)
        - variáveis numéricas (numerical_input_dim)
        """
        layers = []

        # Tamanho total de input da MLP:
        # [tokens_flattened, state_embed, numerical_features]
        input_size = (
            embedding_dim * context_size
            + state_embedding_dim
            + numerical_input_dim
        )

        # num_hidden_neurons pode ser int ou lista
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Primeira camada
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation())
        layers.append(nn.Dropout(dropout_rate))

        # Demais camadas ocultas
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))

        # Camada de saída
        layers.append(nn.Linear(num_hidden_neurons[-1], vocab_size))

        return layers

    def forward(self, train_token, train_numerical, train_state):
        """
        Forward pass.

        Parâmetros:
            train_token    : Tensor (batch_size, context_size) com índices dos tokens.
            train_numerical: Tensor (batch_size, numerical_input_dim) com features numéricas.
            train_state    : Tensor (batch_size,) com índices de estado (0..state_vocab_size-1).

        Retorna:
            logits (batch_size, vocab_size)
        """
        # Embedding dos tokens de contexto: (B, C, D)
        embedded_tokens = self.embedding(train_token)
        # Flatten: (B, C*D)
        flattened_tokens = self.flatten(embedded_tokens)

        # Embedding do estado: (B, state_embedding_dim)
        state_emb = self.state_embedding(train_state)

        # Concatena: [tokens_flattened, state_embedding, numerical_features]
        concatenated = torch.cat(
            (flattened_tokens, state_emb, train_numerical),
            dim=1
        )

        # Passa pela MLP principal
        return self.network(concatenated)

class NGramModel_ExtraInfo_TimeDiffTower(nn.Module):
    """
    Versão estendida da NGramModel_ExtraInfo que inclui:
        - Torre MLP pequena para variáveis de tempo/diferença de pontos
        - Concatenação com embeddings e variáveis numéricas tradicionais

    Estrutura:
        embedding → flatten
        time_diff_features → time_diff_tower
        numerical_features → entram direto
        concat → MLP principal
    """

    def __init__(
        self,
        context_size,
        vocab_size,
        embedding_dim,
        numerical_input_dim,     # variáveis numéricas comuns
        time_diff_input_dim,     # ex: 2 (tempo_restante, diff_pontos)
        time_tower_hidden_dim,   # tamanho da torre
        time_tower_out_dim,      # saída da torre
        num_hidden_neurons,
        num_layers,
        activation = nn.Tanh,
        dropout_rate = 0.0,
        use_layer_norm = False,
    ):
        super().__init__()

        # -------- EMBEDDING + FLATTEN --------
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        # -------- TORRE PARA TEMPO E DIFERENÇA --------
        self.time_tower = nn.Sequential(
            nn.Linear(time_diff_input_dim, time_tower_hidden_dim),
            activation(),
            nn.Dropout(dropout_rate),
            nn.Linear(time_tower_hidden_dim, time_tower_out_dim),
            activation(),
            nn.Dropout(dropout_rate),
        )

        # -------- MLP PRINCIPAL (igual ao ExtraInfo original) --------
        layers = self._init_layers(
            context_size = context_size,
            vocab_size = vocab_size,
            embedding_dim = embedding_dim,
            numerical_input_dim = numerical_input_dim + time_tower_out_dim,
            num_hidden_neurons = num_hidden_neurons,
            num_layers = num_layers,
            activation = activation,
            dropout_rate = dropout_rate,
            use_layer_norm = use_layer_norm,
        )

        self.network = nn.Sequential(*layers)

    def _init_layers(
        self,
        context_size,
        vocab_size,
        embedding_dim,
        numerical_input_dim,
        num_hidden_neurons,
        num_layers,
        activation,
        dropout_rate,
        use_layer_norm,
    ):
        layers = []
        input_size = embedding_dim * context_size + numerical_input_dim

        # num_hidden_neurons pode ser lista ou int
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Primeira camada
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation())
        layers.append(nn.Dropout(dropout_rate))

        # Camadas seguintes
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))

        # Camada de saída
        layers.append(nn.Linear(num_hidden_neurons[-1], vocab_size))

        return layers

    def forward(self, train_token, time_diff_features, train_numerical):
        """
        train_token         -> (B, context_size)
        time_diff_features  -> (B, time_diff_input_dim) ex.: [remaining_time, score_diff]
        train_numerical     -> (B, num_numéricas)
        """

        # Embeddings
        embedded = self.embedding(train_token)
        flattened = self.flatten(embedded)

        # Torre dos estados de tempo e diferença
        time_repr = self.time_tower(time_diff_features)

        # Concat final
        concatenated = torch.cat((flattened, time_repr, train_numerical), dim=1)

        return self.network(concatenated)

class NGramModel_ExtraInfo_MultiTowers(nn.Module):
    """
    N-Gram Neural Network com:
      - 1 torre para tokens (embedding + MLP pequena de 2 camadas);
      - N torres para grupos de variáveis numéricas (cada uma MLP de 2 camadas);
      - 1 MLP de fusão em cima das representações.

    Uso:
      - forward(token_ids, numerical_groups)
      - numerical_groups é SEMPRE uma lista (ou tupla) de tensores.
        Ex: [X_time_diff, X_score_context, X_flags]
        Cada tensor com shape (B, d_i).

    Parâmetros:
        context_size (int): Número de tokens de contexto.
        vocab_size (int): Tamanho do vocabulário.
        embedding_dim (int): Dimensão do embedding dos tokens.

        numeric_input_dims (list[int]):
            Dimensão de entrada de cada grupo numérico.
            Ex: [2, 5, 10] → 3 torres com essas dimensões.

        token_tower_dim (int): Dimensão da saída da torre de tokens
                               (também usada como hidden interno).

        numeric_tower_dims (int ou list[int]):
            - int  → mesma dimensão de saída para todas as torres numéricas
                     (também usada como hidden interno).
            - list → dimensão de saída específica por torre.

        num_hidden_neurons (int ou list[int]): Neurônios da MLP de fusão.
        num_layers (int): Número de camadas ocultas da MLP de fusão.
        activation (callable): Função de ativação (ex.: nn.GELU).
        dropout_rate (float): Probabilidade de dropout.
        use_layer_norm (bool): Se True, usa LayerNorm na MLP de fusão (pre-LN).
    """

    def __init__(
        self,
        context_size: int,
        vocab_size: int,
        embedding_dim: int,
        numeric_input_dims: List[int],
        token_tower_dim: int,
        numeric_tower_dims,
        num_hidden_neurons,
        num_layers: int,
        activation = nn.GELU,
        dropout_rate: float = 0.0,
        use_layer_norm: bool = False,
    ):
        super().__init__()

        self.context_size = context_size
        self.embedding_dim = embedding_dim
        self.token_tower_dim = token_tower_dim

        # -------- Normaliza numeric_input_dims --------
        if not isinstance(numeric_input_dims, (list, tuple)) or len(numeric_input_dims) == 0:
            raise ValueError("numeric_input_dims deve ser uma lista não vazia de inteiros.")
        self.numeric_input_dims = list(numeric_input_dims)
        self.num_numeric_towers = len(self.numeric_input_dims)

        # -------- Normaliza numeric_tower_dims --------
        if isinstance(numeric_tower_dims, int):
            self.numeric_tower_dims = [numeric_tower_dims] * self.num_numeric_towers
        else:
            self.numeric_tower_dims = list(numeric_tower_dims)
            if len(self.numeric_tower_dims) != self.num_numeric_towers:
                raise ValueError(
                    "numeric_tower_dims deve ser int ou lista com mesmo comprimento de numeric_input_dims."
                )

        # -------- Torre de tokens (2 camadas) --------
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        token_input_dim = context_size * embedding_dim
        self.token_tower = nn.Sequential(
            nn.Linear(token_input_dim, token_tower_dim),
            activation(),
            nn.Dropout(dropout_rate),
            nn.Linear(token_tower_dim, token_tower_dim),
            activation(),
            nn.Dropout(dropout_rate),
        )

        # -------- Torres numéricas (cada uma com 2 camadas) --------
        self.numeric_towers = nn.ModuleList()
        for in_dim, out_dim in zip(self.numeric_input_dims, self.numeric_tower_dims):
            tower = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                activation(),
                nn.Dropout(dropout_rate),
                nn.Linear(out_dim, out_dim),
                activation(),
                nn.Dropout(dropout_rate),
            )
            self.numeric_towers.append(tower)

        # -------- MLP de fusão --------
        fusion_layers = self._init_fusion_layers(
            vocab_size = vocab_size,
            token_tower_dim = token_tower_dim,
            numeric_tower_dims = self.numeric_tower_dims,
            num_hidden_neurons = num_hidden_neurons,
            num_layers = num_layers,
            activation = activation,
            dropout_rate = dropout_rate,
            use_layer_norm = use_layer_norm,
        )
        self.network = nn.Sequential(*fusion_layers)

    def _init_fusion_layers(
        self,
        vocab_size: int,
        token_tower_dim: int,
        numeric_tower_dims: List[int],
        num_hidden_neurons,
        num_layers: int,
        activation,
        dropout_rate: float,
        use_layer_norm: bool,
    ):
        layers = []
        input_size = token_tower_dim + sum(numeric_tower_dims)

        # num_hidden_neurons pode ser int ou lista
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        else:
            num_hidden_neurons = list(num_hidden_neurons)
            if len(num_hidden_neurons) != num_layers:
                raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Primeira camada
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation())
        layers.append(nn.Dropout(dropout_rate))

        # Camadas seguintes
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))

        # Saída
        layers.append(nn.Linear(num_hidden_neurons[-1], vocab_size))
        return layers

    def forward(self, token_ids: torch.Tensor, numerical_groups: List[torch.Tensor]):
        """
        token_ids:        (B, context_size)
        numerical_groups: lista de tensores, cada um (B, d_i),
                          com len(numerical_groups) == num_numeric_towers
        """
        if not isinstance(numerical_groups, (list, tuple)):
            raise TypeError("numerical_groups deve ser uma lista/tupla de tensores.")

        if len(numerical_groups) != self.num_numeric_towers:
            raise ValueError(
                f"Foram passados {len(numerical_groups)} grupos numéricos, "
                f"mas o modelo espera {self.num_numeric_towers} torres."
            )

        # Torre de tokens
        embedded = self.embedding(token_ids)          # (B, C, D)
        flattened = self.flatten(embedded)           # (B, C*D)
        token_repr = self.token_tower(flattened)     # (B, token_tower_dim)

        # Torres numéricas
        numeric_reprs = []
        for g, tower in zip(numerical_groups, self.numeric_towers):
            numeric_reprs.append(tower(g))           # cada um (B, out_dim_i)

        # Fusão
        fused = torch.cat([token_repr] + numeric_reprs, dim=1)  # (B, token_dim + sum(num_dims))
        logits = self.network(fused)                            # (B, vocab_size)
        return logits



In [ ]:
class SpentTimeModel(nn.Module):
    """
    A neural network designed to predict time spent based on contextual token 
    information and numerical inputs, with customizable layers, activation functions, 
    optional Layer Normalization, and Dropout.

    Parameters:
        context_size (int): Number of tokens in the context window.
        vocab_size (int): Size of the vocabulary for the embedding layer.
        time_size (int): The output dimension representing time categories or continuous values.
        embedding_dim (int): Dimensionality of the embedding vectors.
        numerical_input_dim (int): Dimensionality of the numerical input features.
        num_hidden_neurons (int or list): Number of neurons in each hidden layer. 
                                          If a list, specifies neurons for each layer.
        num_layers (int): Number of hidden layers in the network.
        activation_function (callable): Activation function to be used in hidden layers.
        dropout_rate (float): Dropout probability (0.0 means no dropout).
        use_layer_norm (bool): Whether to apply Layer Normalization in the layers.
    """

    def __init__(self, context_size, vocab_size, time_size, embedding_dim, numerical_input_dim,
                 num_hidden_neurons, num_layers, activation_function = nn.Tanh, dropout_rate = 0.0, use_layer_norm = False):
        super(SpentTimeModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim = 1)

        # Building the network dynamically based on the number of layers
        layers = []
        input_features = embedding_dim * context_size + numerical_input_dim

        # Handle num_hidden_neurons as a list or single value
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Append the first layer with optional LayerNorm and Dropout
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_features))
        layers.append(nn.Linear(input_features, num_hidden_neurons[0]))
        layers.append(activation_function())
        layers.append(nn.Dropout(dropout_rate))

        # Append additional hidden layers with optional LayerNorm and Dropout
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation_function())
            layers.append(nn.Dropout(dropout_rate))

        # Output layer to predict the time spent
        layers.append(nn.Linear(num_hidden_neurons[-1], time_size))

        # Combine all layers into a Sequential module
        self.network = nn.Sequential(*layers)

    def forward(self, train_token, train_numerical):
        """
        Defines the forward pass of the model using token indices and numerical features.

        Parameters:
            train_token (torch.Tensor): Tensor of token indices.
            train_numerical (torch.Tensor): Tensor of numerical features.

        Returns:
            torch.Tensor: The output tensor representing predicted time values.
        """
        # Embed and flatten token inputs
        embedded = self.embedding(train_token)
        flattened = self.flatten(embedded)

        # Concatenate embedded tokens with numerical inputs
        concatenated = torch.cat((flattened, train_numerical), dim=1)

        # Pass through the network and return output
        return self.network(concatenated)

class SpentTimeModel_ExtraInfo_State(nn.Module):
    """
    Modelo para prever tempo gasto (ou categorias de tempo) usando:
      - contexto de tokens,
      - variáveis numéricas adicionais,
      - um token de estado (ex.: combinação tempo × diferença de pontos).

    Estrutura:
        embedding         : Embedding para tokens.
        state_embedding   : Embedding para o token de estado.
        flatten           : Achata os embeddings dos tokens.
        network           : MLP sobre [tokens_flat, state_emb, numéricos].

    Parâmetros:
        context_size (int): Número de tokens na janela de contexto.
        vocab_size (int): Tamanho do vocabulário de tokens.
        time_size (int): Dimensão de saída (número de classes de tempo ou 1 se contínuo).
        embedding_dim (int): Dimensão do embedding dos tokens.
        numerical_input_dim (int): Dimensão das variáveis numéricas.
        state_vocab_size (int): Tamanho do vocabulário de estados.
        state_embedding_dim (int): Dimensão do embedding de estado.
        num_hidden_neurons (int ou list): Neurônios em cada camada oculta.
        num_layers (int): Número de camadas ocultas.
        activation_function (callable): Função de ativação (ex.: nn.GELU, nn.Tanh).
        dropout_rate (float): Probabilidade de dropout.
        use_layer_norm (bool): Se True, aplica LayerNorm em estilo pre-LN.
    """

    def __init__(
        self,
        context_size,
        vocab_size,
        time_size,
        embedding_dim,
        numerical_input_dim,
        state_vocab_size,
        state_embedding_dim,
        num_hidden_neurons,
        num_layers,
        activation_function = nn.Tanh,
        dropout_rate = 0.0,
        use_layer_norm = False,
    ):
        super(SpentTimeModel_ExtraInfo_State, self).__init__()

        # Embeddings
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.state_embedding = nn.Embedding(state_vocab_size, state_embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        # Tamanho total de entrada: tokens_flat + state_emb + numéricos
        input_features = (
            embedding_dim * context_size
            + state_embedding_dim
            + numerical_input_dim
        )

        # Tratar num_hidden_neurons como int ou lista
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        layers = []

        # Primeira camada oculta (pre-LN opcional)
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_features))
        layers.append(nn.Linear(input_features, num_hidden_neurons[0]))
        layers.append(activation_function())
        layers.append(nn.Dropout(dropout_rate))

        # Camadas ocultas subsequentes
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation_function())
            layers.append(nn.Dropout(dropout_rate))

        # Camada de saída para prever o "tempo"
        layers.append(nn.Linear(num_hidden_neurons[-1], time_size))

        self.network = nn.Sequential(*layers)

    def forward(self, train_token, train_numerical, train_state):
        """
        Forward pass do modelo.

        Parâmetros:
            train_token    (Tensor): (batch_size, context_size) índices de tokens.
            train_numerical(Tensor): (batch_size, numerical_input_dim) variáveis numéricas.
            train_state    (Tensor): (batch_size,) índices de estado.

        Retorna:
            Tensor (batch_size, time_size): predição de tempo.
        """
        # Embedding dos tokens
        embedded_tokens = self.embedding(train_token)     # (B, C, D)
        flattened_tokens = self.flatten(embedded_tokens)  # (B, C*D)

        # Embedding do estado
        state_emb = self.state_embedding(train_state)     # (B, state_embedding_dim)

        # Concatena tudo: [tokens_flat, state_emb, numéricos]
        concatenated = torch.cat(
            (flattened_tokens, state_emb, train_numerical),
            dim=1
        )

        return self.network(concatenated)
    
class SpentTimeModel_TimeDiffTower(nn.Module):
    """
    Versão estendida da SpentTimeModel que inclui:
      - torre MLP pequena para variáveis de tempo/diferença (ex: remaining_time, score_diff)
      - concatenação dessa representação com:
          * embeddings dos tokens
          * variáveis numéricas tradicionais

    Estrutura:
        train_token → embedding → flatten
        time_diff_features → time_tower
        train_numerical → entra direto
        concat = [flattened, time_repr, train_numerical]
        → network (MLP principal) → time_size (classes/saídas de tempo)
    """

    def __init__(
        self,
        context_size: int,
        vocab_size: int,
        time_size: int,
        embedding_dim: int,
        numerical_input_dim: int,      # variáveis numéricas comuns
        time_diff_input_dim: int,      # ex: 2 (remaining_time, score_diff)
        time_tower_hidden_dim: int,
        time_tower_out_dim: int,
        num_hidden_neurons,
        num_layers: int,
        activation_function = nn.Tanh,
        dropout_rate: float = 0.0,
        use_layer_norm: bool = False,
    ):
        super(SpentTimeModel_TimeDiffTower, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim = 1)

        # -------- Torre para tempo/diferença --------
        # (sem LayerNorm aqui, como combinamos)
        self.time_tower = nn.Sequential(
            nn.Linear(time_diff_input_dim, time_tower_hidden_dim),
            activation_function(),
            nn.Dropout(dropout_rate),
            nn.Linear(time_tower_hidden_dim, time_tower_out_dim),
            activation_function(),
            nn.Dropout(dropout_rate),
        )

        # -------- MLP principal (igual à SpentTimeModel, mas com input aumentado) --------
        # input_features = embedding*context + numéricas “normais” + representação de tempo
        input_features = embedding_dim * context_size + numerical_input_dim + time_tower_out_dim

        layers = []
        # num_hidden_neurons pode ser int ou lista
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Primeira camada
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_features))
        layers.append(nn.Linear(input_features, num_hidden_neurons[0]))
        layers.append(activation_function())
        layers.append(nn.Dropout(dropout_rate))

        # Camadas ocultas adicionais
        for i in range(1, num_layers):
            in_dim  = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation_function())
            layers.append(nn.Dropout(dropout_rate))

        # Saída para tempo (classes / bins de tempo)
        layers.append(nn.Linear(num_hidden_neurons[-1], time_size))

        self.network = nn.Sequential(*layers)

    def forward(
        self,
        train_token: torch.Tensor,         # (B, context_size)
        time_diff_features: torch.Tensor,  # (B, time_diff_input_dim)
        train_numerical: torch.Tensor,     # (B, numerical_input_dim)
    ) -> torch.Tensor:
        
        """
        Forward pass usando:
          - tokens
          - features de tempo/diferença
          - demais variáveis numéricas
        """

        # Embedding dos tokens
        embedded = self.embedding(train_token)
        flattened = self.flatten(embedded)

        # Torre de tempo/diferença
        time_repr = self.time_tower(time_diff_features)

        # Concatena tudo
        concatenated = torch.cat((flattened, time_repr, train_numerical), dim = 1)

        # MLP principal
        return self.network(concatenated)
    
class SpentTimeModel_MultiTowers(nn.Module):
    """
    Modelo de tempo gasto com múltiplas torres numéricas (MultiTowers),
    prevendo time_size.

    Atualização: torres (tokens e numéricas) agora têm DUAS camadas,
    no padrão do seu TimeDiffTower:
      Linear(in -> tower_dim) -> act -> drop -> Linear(tower_dim -> tower_dim) -> act -> drop
    """

    def __init__(
        self,
        context_size: int,
        vocab_size: int,
        time_size: int,
        embedding_dim: int,
        numeric_input_dims: List[int],
        token_tower_dim: int,
        numeric_tower_dims,
        num_hidden_neurons,
        num_layers: int,
        activation_function = nn.GELU,
        dropout_rate: float = 0.0,
        use_layer_norm: bool = False,
    ):
        super().__init__()

        self.context_size = context_size
        self.embedding_dim = embedding_dim
        self.token_tower_dim = token_tower_dim
        self.time_size = time_size

        # -------- Normalização das dims numéricas --------
        if not isinstance(numeric_input_dims, (list, tuple)) or len(numeric_input_dims) == 0:
            raise ValueError("numeric_input_dims deve ser uma lista não vazia de inteiros.")
        self.numeric_input_dims = list(numeric_input_dims)
        self.num_numeric_towers = len(self.numeric_input_dims)

        # numeric_tower_dims: int ou lista
        if isinstance(numeric_tower_dims, int):
            self.numeric_tower_dims = [numeric_tower_dims] * self.num_numeric_towers
        else:
            self.numeric_tower_dims = list(numeric_tower_dims)
            if len(self.numeric_tower_dims) != self.num_numeric_towers:
                raise ValueError(
                    "numeric_tower_dims deve ser int ou lista com mesmo comprimento de numeric_input_dims."
                )

        # -------- Torre de tokens --------
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        token_input_dim = context_size * embedding_dim

        self.token_tower = nn.Sequential(
            nn.Linear(token_input_dim, token_tower_dim),
            activation_function(),
            nn.Dropout(dropout_rate),
            nn.Linear(token_tower_dim, token_tower_dim),
            activation_function(),
            nn.Dropout(dropout_rate),
        )

        # -------- Torres numéricas --------
        self.numeric_towers = nn.ModuleList()
        for in_dim, out_dim in zip(self.numeric_input_dims, self.numeric_tower_dims):
            tower = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                activation_function(),
                nn.Dropout(dropout_rate),
                nn.Linear(out_dim, out_dim),
                activation_function(),
                nn.Dropout(dropout_rate),
            )
            self.numeric_towers.append(tower)

        # -------- MLP de fusão --------
        fusion_layers = self._init_fusion_layers(
            time_size=time_size,
            token_tower_dim=token_tower_dim,
            numeric_tower_dims=self.numeric_tower_dims,
            num_hidden_neurons=num_hidden_neurons,
            num_layers=num_layers,
            activation_function=activation_function,
            dropout_rate=dropout_rate,
            use_layer_norm=use_layer_norm,
        )
        self.network = nn.Sequential(*fusion_layers)

    def _init_fusion_layers(
        self,
        time_size: int,
        token_tower_dim: int,
        numeric_tower_dims: List[int],
        num_hidden_neurons,
        num_layers: int,
        activation_function,
        dropout_rate: float,
        use_layer_norm: bool,
    ):
        layers = []
        input_size = token_tower_dim + sum(numeric_tower_dims)

        # num_hidden_neurons: int ou lista
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        else:
            num_hidden_neurons = list(num_hidden_neurons)
            if len(num_hidden_neurons) != num_layers:
                raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Primeira camada
        if use_layer_norm:
            layers.append(nn.LayerNorm(input_size))
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        layers.append(activation_function())
        layers.append(nn.Dropout(dropout_rate))

        # Camadas seguintes
        for i in range(1, num_layers):
            in_dim = num_hidden_neurons[i - 1]
            out_dim = num_hidden_neurons[i]

            if use_layer_norm:
                layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(activation_function())
            layers.append(nn.Dropout(dropout_rate))

        # Saída
        layers.append(nn.Linear(num_hidden_neurons[-1], time_size))
        return layers

    def forward(self, token_ids: torch.Tensor, numerical_groups: List[torch.Tensor]):
        if not isinstance(numerical_groups, (list, tuple)):
            raise TypeError("numerical_groups deve ser uma lista/tupla de tensores.")

        if len(numerical_groups) != self.num_numeric_towers:
            raise ValueError(
                f"Foram passados {len(numerical_groups)} grupos numéricos, "
                f"mas o modelo espera {self.num_numeric_towers} torres."
            )

        embedded = self.embedding(token_ids)          # (B, C, D)
        flattened = self.flatten(embedded)           # (B, C*D)
        token_repr = self.token_tower(flattened)     # (B, token_tower_dim)

        numeric_reprs = []
        for g, tower in zip(numerical_groups, self.numeric_towers):
            numeric_reprs.append(tower(g))           # (B, out_dim_i)

        fused = torch.cat([token_repr] + numeric_reprs, dim=1)  # (B, token_dim + sum(out_dims))
        logits = self.network(fused)                              # (B, time_size)
        return logits


In [ ]:
class GameStatus:
    def __init__(self, num_plays, remaining_time, limhteam, limvteam, limhtimeout, limvtimeout,
                 home_points, away_points, ball_possession, period, keep_generating, sorted_token,
                 prob_token, sorted_time, prob_time, token_vector = None, numerical_vector = None):

        self.remaining_time  = np.asarray(remaining_time)
        self.num_plays       = np.asarray(num_plays)
        self.limhteam        = np.asarray(limhteam)
        self.limvteam        = np.asarray(limvteam)
        self.limhtimeout     = np.asarray(limhtimeout)
        self.limvtimeout     = np.asarray(limvtimeout)
        self.home_points     = np.asarray(home_points)
        self.away_points     = np.asarray(away_points)
        self.ball_possession = np.asarray(ball_possession)
        self.period          = np.asarray(period)
        self.keep_generating = np.asarray(keep_generating)
        self.sorted_token    = np.asarray(sorted_token)
        self.prob_token      = np.asarray(prob_token)
        self.sorted_time     = np.asarray(sorted_time)
        self.prob_time       = np.asarray(prob_time)
 
        # validação rápida
        n = len(self.remaining_time)
        list_items = [self.limhteam, self.limvteam, self.limhtimeout, self.limvtimeout, 
                      self.home_points, self.away_points, self.ball_possession, self.period, 
                      self.keep_generating, self.num_plays, self.sorted_token, self.prob_token, 
                      self.sorted_time, self.prob_time]
        if not all(len(arr) == n for arr in list_items):
            raise ValueError("All arrays must have the same size.")
        
        # 2D variables (opcional)
        if token_vector is not None:
            token_vector = np.asarray(token_vector)
            if token_vector.shape[0] != n:
                raise ValueError("token_vector must have n_sims rows.")
        self.token_vector = token_vector  # pode ser None

        if numerical_vector is not None:
            numerical_vector = np.asarray(numerical_vector)
            if numerical_vector.shape[0] != n:
                raise ValueError("numerical_vector must have n_sims rows.")
        self.numerical_vector = numerical_vector  # pode ser None

    @property
    def n_sims(self):
        return self.remaining_time.shape[0]
    
    def copy(self):
        return GameStatus(
            self.num_plays.copy(),
            self.remaining_time.copy(),
            self.limhteam.copy(),
            self.limvteam.copy(),
            self.limhtimeout.copy(),
            self.limvtimeout.copy(),
            self.home_points.copy(),
            self.away_points.copy(),
            self.ball_possession.copy(),
            self.period.copy(),
            self.keep_generating.copy(),
            self.sorted_token.copy(),
            self.prob_token.copy(),
            self.sorted_time.copy(),
            self.prob_time.copy(),
            None if self.token_vector is None else self.token_vector.copy(),
            None if self.numerical_vector is None else self.numerical_vector.copy(),
        )

    @classmethod
    def from_initial(cls, n_sims, window, n_features):
        return cls(
            num_plays       = np.zeros(n_sims, dtype = int),
            remaining_time  = np.full(n_sims, 7200, dtype = int),
            limhteam        = np.full(n_sims, 4, dtype = int),
            limvteam        = np.full(n_sims, 4, dtype = int),
            limhtimeout     = np.full(n_sims, 7, dtype = int),
            limvtimeout     = np.full(n_sims, 7, dtype = int),
            home_points     = np.zeros(n_sims, dtype = int),
            away_points     = np.zeros(n_sims, dtype = int),
            ball_possession = np.zeros(n_sims, dtype = int),
            period          = np.zeros(n_sims, dtype = int),
            keep_generating = np.ones(n_sims, dtype = int),
            sorted_token    = np.zeros(n_sims, dtype = int),
            prob_token      = np.zeros(n_sims, dtype = float),
            sorted_time     = np.zeros(n_sims, dtype = int),
            prob_time       = np.zeros(n_sims, dtype = float),
            token_vector    = np.zeros((n_sims, window), dtype = int),
            numerical_vector= np.zeros((n_sims, n_features), dtype = float),
        )

    def select(self, idx):
        idx = np.asarray(idx)
        token_vec = None
        if self.token_vector is not None:
            token_vec = self.token_vector[idx]

        num_vec = None
        if self.numerical_vector is not None:
            num_vec = self.numerical_vector[idx]
        return GameStatus(
            self.num_plays,
            self.remaining_time[idx],
            self.limhteam[idx],
            self.limvteam[idx],
            self.limhtimeout[idx],
            self.limvtimeout[idx],
            self.home_points[idx],
            self.away_points[idx],
            self.ball_possession[idx],
            self.period[idx],
            self.keep_generating[idx],
            self.sorted_token[idx],
            self.prob_token[idx],
            self.sorted_time[idx],
            self.prob_time[idx],
            token_vec,
            num_vec,
        )
    
    def to_dict(self, idx = None):
        """
        Transform the class in a dictionary with current status of GameStatus.

        Parâmetros
        ----------
        idx:
            - None: return all simulations.
            - int: return only a unique dictionary with scalar values.
            - lista/array: retunrn arrays only for the selected index.

        Return
        -------
        dict
        """
        # Se nenhum índice for passado → retorna tudo
        if idx is None:
            return {
                "num_plays":       self.num_plays.copy(),
                "period":          self.period.copy(),
                "remaining_time":  self.remaining_time.copy(),
                "limhteam":        self.limhteam.copy(),
                "limvteam":        self.limvteam.copy(),
                "limhtimeout":     self.limhtimeout.copy(),
                "limvtimeout":     self.limvtimeout.copy(),
                "home_points":     self.home_points.copy(),
                "away_points":     self.away_points.copy(),
                "ball_possession": self.ball_possession.copy(),
                "keep_generating": self.keep_generating.copy(),
                "sorted_token":    self.sorted_token.copy(),
                "prob_token":      self.prob_token.copy(),
                "sorted_time":     self.sorted_time.copy(),
                "token_vector":    None if self.token_vector is None else [self.token_vector.copy()],
                "numerical_vector":None if self.numerical_vector is None else [self.numerical_vector.copy()]
            }

        # Converte índices para array
        idx = np.asarray(idx)

        # Caso idx seja um único inteiro
        if idx.ndim == 0:
            return {
                "num_plays":       int(self.num_plays[idx]),
                "period":          int(self.period[idx]),
                "remaining_time":  int(self.remaining_time[idx]),
                "limhteam":        int(self.limhteam[idx]),
                "limvteam":        int(self.limvteam[idx]),
                "limhtimeout":     int(self.limhtimeout[idx]),
                "limvtimeout":     int(self.limvtimeout[idx]),
                "home_points":     int(self.home_points[idx]),
                "away_points":     int(self.away_points[idx]),
                "ball_possession": int(self.ball_possession[idx]),
                "keep_generating": int(self.keep_generating[idx]),
                "sorted_token":    int(self.sorted_token[idx]),
                "prob_token":      float(self.prob_token[idx]),
                "sorted_time":     int(self.sorted_time[idx]),
                "token_vector":    None if self.token_vector is None else [self.token_vector[idx].copy()],
                "numerical_vector":None if self.numerical_vector is None else [self.numerical_vector[idx].copy()]
            }

        # Caso idx seja lista/array → seleciona subconjunto das sims
        return {
            "num_plays":       self.num_plays[idx].copy(),
            "period":          self.period[idx].copy(),
            "remaining_time":  self.remaining_time[idx].copy(),
            "limhteam":        self.limhteam[idx].copy(),
            "limvteam":        self.limvteam[idx].copy(),
            "limhtimeout":     self.limhtimeout[idx].copy(),
            "limvtimeout":     self.limvtimeout[idx].copy(),
            "home_points":     self.home_points[idx].copy(),
            "away_points":     self.away_points[idx].copy(),
            "ball_possession": self.ball_possession[idx].copy(),
            "keep_generating": self.keep_generating[idx].copy(),
            "sorted_token":    self.sorted_token[idx].copy(),
            "prob_token":      self.prob_token[idx].copy(),
            "sorted_time":     self.sorted_time[idx].copy(),
            "token_vector":    None if self.token_vector is None else [self.token_vector[idx].copy()],
            "numerical_vector":None if self.numerical_vector is None else [self.numerical_vector[idx].copy()]
        }
        
    def update_indices(self, idx, **kwargs):
        """
        Update the features 1D (and 2D) only for selected index.
        Ex:
            gs.update_indices([1,2,3], remaining_time = [100,200,150])
        """
        idx = np.asarray(idx)

        for key, value in kwargs.items():
            arr = getattr(self, key)
            if arr is None:
                raise ValueError(f"Feature '{key}' is None in GameStatus.")

            value = np.asarray(value)

            # Campo 1D
            if arr.ndim == 1:
                if value.ndim == 0:
                    arr[idx] = value
                else:
                    arr[idx] = value

            # Campo 2D (token_vector, numerical_vector)
            elif arr.ndim == 2:
                # arr[idx, :] recebe value
                if value.ndim == 1:
                    # mesmo valor em todas as linhas selecionadas
                    arr[idx, :] = value
                else:
                    # value tem que ter shape (len(idx), n_cols)
                    arr[idx, :] = value

            else:
                raise ValueError(f"Feature '{key}' has ndim = {arr.ndim} not available.")

            setattr(self, key, arr)


In [ ]:
class GameLog:
    def __init__(self, n_sims: int):
        # a list object for each simulation
        self.period          = [[] for _ in range(n_sims)]
        self.remaining_time  = [[] for _ in range(n_sims)]
        self.token           = [[] for _ in range(n_sims)]
        self.prob            = [[] for _ in range(n_sims)]
        self.time            = [[] for _ in range(n_sims)]
        self.home_points     = [[] for _ in range(n_sims)]
        self.away_points     = [[] for _ in range(n_sims)]
        self.limhteam        = [[] for _ in range(n_sims)]
        self.limvteam        = [[] for _ in range(n_sims)]
        self.limhtimeout     = [[] for _ in range(n_sims)]
        self.limvtimeout     = [[] for _ in range(n_sims)]
        self.ball_possession = [[] for _ in range(n_sims)]
        self.keep_generating = [[] for _ in range(n_sims)]
        self.n_sims = n_sims

    def _to_iter(self, x, n):
        """
        Convert escalar / array / list into an iterable object with size n.
        - If x is a scalar: repeat n times.
        - if x is an array/list: just convert to np.array and check the size.
        """
        if np.isscalar(x):
            return [x] * n
        x = np.asarray(x)
        if x.shape[0] != n:
            raise ValueError("Unmatched size in add_events.")
        return x

    def add_events(self, indices, game_status):
        """
        Add a game "step" in game log for a simulations subset.

        indices: list/array of simulations índex that have play.
        The orthers arguments could be:
          - scalars (apply for all indices), or
          - arrays/lists with the same size of indices
        """
        indices = np.asarray(indices)
        n = len(indices)

        period          = self._to_iter(game_status.period[indices],          n)
        remaining_time  = self._to_iter(game_status.remaining_time[indices],  n)
        token           = self._to_iter(game_status.sorted_token[indices],    n)
        prob            = self._to_iter(game_status.prob_token[indices],      n)
        time            = self._to_iter(game_status.sorted_time[indices],     n)
        home_points     = self._to_iter(game_status.home_points[indices],     n)
        away_points     = self._to_iter(game_status.away_points[indices],     n)
        limhteam        = self._to_iter(game_status.limhteam[indices],        n)
        limvteam        = self._to_iter(game_status.limvteam[indices],        n)
        limhtimeout     = self._to_iter(game_status.limhtimeout[indices],     n)
        limvtimeout     = self._to_iter(game_status.limvtimeout[indices],     n)
        ball_possession = self._to_iter(game_status.ball_possession[indices], n)
        keep_generating = self._to_iter(game_status.keep_generating[indices], n)

        fields = [("period", period, int),
                  ("remaining_time", remaining_time, int),
                  ("token", token, int),
                  ("prob", prob, float),
                  ("time", time, int),
                  ("home_points", home_points, int),
                  ("away_points", away_points, int),
                  ("limhteam", limhteam, int),
                  ("limvteam", limvteam, int),
                  ("limhtimeout", limhtimeout, int),
                  ("limvtimeout", limvtimeout, int),
                  ("ball_possession", ball_possession, int),
                  ("keep_generating", keep_generating, int)]

        for i, *vals in zip(indices, *[col for _, col, _ in fields]):
            for (attr, _, cast), v in zip(fields, vals):
                getattr(self, attr)[i].append(cast(v))            

    def to_dataframe(self):

        rows = []

        for sim_id in range(self.n_sims):
            n_steps = len(self.period[sim_id])

            for step in range(n_steps):
                rows.append({
                    "sim_id": sim_id,
                    "step": step,
                    "period": self.period[sim_id][step],
                    "remaining_time": self.remaining_time[sim_id][step],
                    "token": self.token[sim_id][step],
                    "prob": self.prob[sim_id][step],
                    "time": self.time[sim_id][step],
                    "home_points": self.home_points[sim_id][step],
                    "away_points": self.away_points[sim_id][step],
                    "limhteam": self.limhteam[sim_id][step],
                    "limvteam": self.limvteam[sim_id][step],
                    "limhtimeout": self.limhtimeout[sim_id][step],
                    "limvtimeout": self.limvtimeout[sim_id][step],
                    "ball_possession": self.ball_possession[sim_id][step],
                    "keep_generating": self.keep_generating[sim_id][step],
                })

        return pd.DataFrame(rows)

In [ ]:
def define_special_tokens(config):
    if config == 1:

        # tokens que encerram o periodo
        end_period_tokens = [91, 92, 93, 94, 95, 97, 98, 99, 154, 155, 156, 157, 158, 159, 84, 85, 86, 87, 88, 89, 90, 85, 86, 87, 88, 89, 90, 84]
        # tokens que demandam rebote obrigatorio apos eles
        missed_shot_tokens = [ 18,  19,  34,  35, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 118, 119, 128, 129, 
                              140, 141, 144, 145, 162, 163, 166, 167, 172, 173, 174, 179, 180, 193, 194, 197, 201, 202, 
                              205, 206, 211, 212, 215, 216, 221, 222, 225, 226, 229, 230, 233, 234, 237, 238, 245, 246, 
                              247, 387, 388, 403, 404, 409, 410, 415, 416, 417, 418, 431, 432, 433, 450, 451, 460, 461, 
                              529, 530, 531, 538, 539, 540, 541, 556, 557, 558]
        # special_tokens = [ 96,  111, 112, 114, 117, 121, 
        #                   122, 124, 127, 131, 132, 134, 137, 142, 143, 147, 148, 150, 153, 239, 240, 243, 244, 257, 
        #                   258, 280, 281, 333, 334]
        # tokens que sao arremessos errados que sao bloqueados e dai necessitam o sorteio de um token de rebote 
        # antes do sorteio de um token de encerramento
        timeout_home_team = [6, 27, 45, 47, 65, 572, 574, 576, 577, 580, 581, 582, 586, 587, 588, 592, 593, 597, 599]
        timeout_away_team = [5,  7, 26, 44, 46,  64, 573, 575, 578, 579, 583, 584, 585, 589, 590, 591, 594, 595, 596, 598, 600]

        shot_blocked_tokens = [104, 105, 108, 109, 162, 163]
        # define a posse para a equipe mandante no periodo 1
        ball_possession_home_p1 = [86]
        # define a posse para a equipe visitante no periodo 1
        ball_possession_away_p1 = [87]
        # define a posse para nenuma das equipes no periodo 1
        ball_possession_undefined_p1 = [85]
        # define a posse de bola para equipe mandante depois de um primeiro token sem definir no primeiro periodo
        ball_possession_undefined_home_p1 = [66, 68, 70, 72, 78, 86, 511, 518]
        # define a posse de bola para equipe visitante depois de um primeiro token sem definir no primeiro periodo
        ball_possession_undefined_away_p1 = [67, 69, 71, 73, 79, 87, 512, 517]
        # define a posse para nenhuma das equipes no overtime
        ball_possession_undefined_pot = [84]
        # define a posse de bola para equipe mandante depois de um primeiro token sem definir no overtime
        ball_possession_undefined_home_pot = [66, 68]
        # define a posse de bola para equipe visitante depois de um primeiro token sem definir no overtime
        ball_possession_undefined_away_pot = [67, 69]

        home_pts_1 = [164, 168, 172, 177, 181, 189, 191, 195, 199, 203, 209, 213, 217, 219, 223, 227, 231, 235, 268, 
                      352, 358, 384, 388, 396, 402, 406, 410, 417, 418, 431, 436, 437, 445, 447, 451, 459, 500, 525, 
                      527, 534, 536, 547, 549, 550, 552, 554, 563, 565]

        home_pts_2 = [  1,   3,   5,   7,   8,  10,  12,  14,  18,  20,  24,  26,  28,  30,  34,  38,  42,  44,  46,  
                       48,  50,  54,  56, 170, 183, 241, 243, 254, 270, 386, 391, 392, 408, 413, 414, 423, 424, 425, 
                      427, 439, 449, 454, 455, 502, 533]

        home_pts_3 = [16,  22,  32,  36,  40,  52,  58,  60,  62,  64, 185, 441]

        away_pts_1 = [165, 169, 173, 178, 182, 190, 192, 196, 200, 204, 210, 214, 218, 220, 224, 228, 232, 236, 269, 
                      351, 357, 383, 387, 395, 401, 405, 409, 415, 416, 419, 430, 434, 435, 444, 446, 450, 458, 499, 
                      526, 528, 535, 537, 546, 548, 551, 553, 555, 564, 566]

        away_pts_2 = [  2,   4,   6,   9,  11,  13,  15,  19,  21,  25,  27,  29,  31,  35,  39,  43,  45,  47,  49,  
                       51,  55,  57, 171, 184, 242, 244, 255, 271, 385, 389, 390, 407, 411, 412, 420, 421, 422, 426, 
                       438, 448, 452, 453, 501, 532]
        
        away_pts_3 = [17, 23, 33, 37, 41, 53, 59, 61, 63, 65, 186, 440]
        
        home_fouls = [ 15,  17,  19,  21,  23,  31,  33,  35,  37,  39,  41,  51,  53,  57,  59, 139, 161, 253, 255, 
                      263, 265, 267, 269, 271, 365, 367, 368, 371, 372, 373, 377, 378, 381, 383, 385, 387, 389, 390, 
                      393, 395, 397, 399, 401, 403, 405, 407, 409, 411, 412, 415, 416, 419, 420, 421, 422, 426, 428, 
                      430, 432, 434, 435, 438, 440, 442, 444, 446, 448, 450, 452, 453, 456, 458, 460, 462, 464, 476, 
                      478, 480, 481, 483, 486, 488, 490, 492, 494, 496, 497, 499, 501, 503, 505]

        away_fouls = [ 14,  16,  18,  20,  22,  30,  32,  34,  36,  38,  40,  50,  52,  56,  58, 138, 160, 252, 254, 
                      256, 262, 264, 266, 268, 270, 366, 369, 370, 374, 375, 376, 379, 380, 382, 384, 386, 388, 391, 
                      392, 394, 396, 398, 400, 402, 404, 406, 408, 410, 413, 414, 417, 418, 423, 424, 425, 427, 429, 
                      431, 433, 436, 437, 439, 441, 443, 445, 447, 449, 451, 454, 455, 457, 459, 461, 463, 465, 477, 
                      479, 482, 484, 487, 489, 491, 493, 495, 498, 500, 502, 504, 506]
        
        home_timeouts = [6, 27, 45, 47, 65, 572, 574, 576, 577, 580, 581, 582, 586, 587, 588, 592, 593, 597, 599]

        away_timeouts = [5, 7, 26, 44, 46, 64, 573, 575, 578, 579, 583, 584, 585, 589, 590, 591, 594, 595, 596, 598, 600]

        start_period_tokens = [84, 85, 86, 87, 88, 89, 90]
        # tokens permitidos inicio primeiro periodo
        allowed_start_tokens_period1 = [85, 86, 87]
        # tokens permitidos inicio segundo periodo
        allowed_start_tokens_period2 = [88]
        # tokens permitidos inicio terceiro periodo
        allowed_start_tokens_period3 = [89]
        # tokens permitidos inicio quarto periodo
        allowed_start_tokens_period4 = [90]
        # tokens permitidos inicio overtime
        allowed_start_tokens_period5 = [84]
        # tokens permitidos fim periodo 1
        allowed_end_tokens_period1 = [92, 97, 154, 155]
        # tokens permitidos fim periodo 2
        allowed_end_tokens_period2 = [93, 98, 156, 157]
        # tokens permitidos fim periodo 3
        allowed_end_tokens_period3 = [94, 99, 158, 159]
        # tokens permitidos fim periodo 4
        allowed_end_tokens_period4 = [95]
        # tokens permitidos fim overtime
        allowed_end_tokens_overtime = [91]
        # tokens para substituir se forem sorteados ao fim do periodo1
        change_end_tokens_period1_situation1 = [91, 93, 94, 95, 84, 85, 86, 87, 88, 89, 90]
        replace_end_token_period1_situation1 = 92
        change_end_tokens_period1_situation2 = [98, 99]
        replace_end_token_period1_situation2 = 97
        change_end_tokens_period1_situation3 = [156, 158]
        replace_end_token_period1_situation3 = 154
        change_end_tokens_period1_situation4 = [157, 159]
        replace_end_token_period1_situation4 = 155
        # tokens para substituir se forem sorteados ao fim do periodo2
        change_end_tokens_period2_situation1 = [91, 92, 94, 95, 84, 85, 86, 87, 88, 89, 90]
        replace_end_token_period2_situation1 = 93
        change_end_tokens_period2_situation2 = [97, 99]
        replace_end_token_period2_situation2 = 98
        change_end_tokens_period2_situation3 = [154, 158]
        replace_end_token_period2_situation3 = 156
        change_end_tokens_period2_situation4 = [155, 159]
        replace_end_token_period2_situation4 = 157
        # tokens para substituir se forem sorteados ao fim do periodo3
        change_end_tokens_period3_situation1 = [91, 92, 93, 95, 84, 85, 86, 87, 88, 89, 90]
        replace_end_token_period3_situation1 = 94
        change_end_tokens_period3_situation2 = [97, 98]
        replace_end_token_period3_situation2 = 99
        change_end_tokens_period3_situation3 = [154, 156]
        replace_end_token_period3_situation3 = 158
        change_end_tokens_period3_situation4 = [155, 157]
        replace_end_token_period3_situation4 = 159  
        # tokens para substituir se forem sorteados ao fim do periodo4
        change_end_tokens_period4_situation1 = [91, 92, 93, 94, 97, 98, 99, 154, 155, 156, 157, 158, 159, 84, 85, 86, 87, 88, 89, 90]
        replace_end_token_period4_situation1 = 95
        # tokens para substituir se forem sorteados ao fim do overtime
        change_end_tokens_overtime_situation1 = [92, 93, 94, 95, 97, 98, 99, 154, 155, 156, 157, 158, 159, 84, 85, 86, 87, 88, 89, 90]
        replace_end_token_overtime_situation1 = 91
        
        instant_review_token = [96]

        # se limhteam == 0, entao a mascara tem que anular: # 34 tokens
        tokens_limhteam_p = [ 21,  23,  39,  41, 267, 269, 271, 381, 383, 385, 387, 389, 390, 393, 395, 442, 444, 
                                  446, 448, 450, 452, 453, 456, 458, 460, 464, 478, 483, 488, 492, 497, 499, 501, 505]

        # se limvteam == 0, entao a mascara tem que anular:# 34 tokens
        tokens_limvteam_p = [ 20,  22,  38,  40, 266, 268, 270, 382, 384, 386, 388, 391, 392, 394, 396, 443, 445, 
                                  447, 449, 451, 454, 455, 457, 459, 461, 465, 479, 484, 489, 493, 498, 500, 502, 506]

        # se limhteam == 1, entao a mascara tem que anular: # 57  tokens
        tokens_limhteam_a = [ 15,  17,  19,  31,  33,  35,  37,  51,  53,  57,  59, 139, 161, 253, 255, 263, 265, 
                             365, 367, 368, 371, 372, 373, 377, 378, 397, 399, 401, 403, 405, 407, 409, 411, 412, 
                             415, 416, 419, 420, 421, 422, 426, 428, 430, 432, 434, 435, 438, 440, 462, 476, 480, 
                             481, 486, 490, 494, 496, 503]

        # se limvteam == 1, entao a mascara tem que anular: # 55 tokens
        tokens_limvteam_a = [ 14,  16,  18,  30,  32,  34,  36,  50,  52,  56,  58, 138, 160, 252, 254, 256, 262, 
                             264, 366, 369, 370, 374, 375, 376, 379, 380, 398, 400, 402, 404, 406, 408, 410, 413, 
                             414, 417, 418, 423, 424, 425, 427, 429, 431, 433, 436, 437, 439, 441, 463, 477, 482, 
                             487, 491, 495, 504]

        # se periodo == 1, entao a mascara tem que anular:
        tokens_avoid_period1 = [ 84,  85, 86, 87, 88,  89,  90,  91,  93,  94,  95,  98,  99, 156, 157, 158, 159]

        # se periodo == 2, entao a mascara tem que anular:
        tokens_avoid_period2 = [ 84,  85,  86,  87,  88, 89,  90,  91,  92,  94,  95,  97,  99, 154, 155, 158, 159]

        # se periodo == 3, entao a mascara tem que anular:
        tokens_avoid_period3 = [ 84,  85,  86,  87,  88,  89, 90,  91,  92,  93,  95,  97,  98, 154, 155, 156, 157]

        # se periodo == 4, entao a mascara tem que anular:
        tokens_avoid_period4 = [ 84,  85,  86,  87,  88,  89,  90, 91,  92,  93,  94,  97,  98, 99, 154, 155, 156, 157, 158, 159]

        # se periodo == overtime, entao a mascara tem que anular:
        tokens_avoid_period_ot = [84, 85,  86,  87,  88,  89,  90,  92,  93,  94,  95,  97,  98,  99, 154, 155, 156, 157, 158, 159]        
        

        dict_special_tokens = {"home_fouls_limhteam_a": tokens_limhteam_a, "home_fouls_limhteam_p": tokens_limhteam_p,
                               "away_fouls_limvteam_a": tokens_limvteam_a, "away_fouls_limvteam_p": tokens_limvteam_p,
                               "timeouts_hometeam": timeout_home_team, "timeouts_awayteam": timeout_away_team,
                               "not_allowed_period1": tokens_avoid_period1, "not_allowed_period2": tokens_avoid_period2,
                               "not_allowed_period3": tokens_avoid_period3, "not_allowed_period4": tokens_avoid_period4,
                               "not_allowed_overtime": tokens_avoid_period_ot,    
                               'end_period': end_period_tokens, 'missed_shots': missed_shot_tokens, 'shot_blocked_end': shot_blocked_tokens, 
                               'ballposs_home_p1': ball_possession_home_p1, 'ballposs_away_p1': ball_possession_away_p1, 
                               'ballposs_und_p1': ball_possession_undefined_p1, 'ballposs_und_home_p1': ball_possession_undefined_home_p1, 
                               'ballposs_und_away_p1': ball_possession_undefined_away_p1, 'ballposs_und_pot': ball_possession_undefined_pot, 
                               'ballposs_und_home_pot': ball_possession_undefined_home_pot,
                               'ballposs_und_away_pot': ball_possession_undefined_away_pot,
                               'home_pts1': home_pts_1, 'home_pts2': home_pts_2, 'home_pts3': home_pts_3, 'away_pts1': away_pts_1, 
                               'away_pts2': away_pts_2, 'away_pts3': away_pts_3, 'home_fouls': home_fouls, 'away_fouls': away_fouls,
                               'home_timeouts': home_timeouts, 'away_timeouts': away_timeouts,
                               'start_period': start_period_tokens, 'start_period1': allowed_start_tokens_period1, 
                               'start_period2': allowed_start_tokens_period2, 'start_period3': allowed_start_tokens_period3, 
                               'start_period4': allowed_start_tokens_period4, 'start_overtime': allowed_start_tokens_period5, 
                               'end_period1': allowed_end_tokens_period1, 'end_period2': allowed_end_tokens_period2, 
                               'end_period3': allowed_end_tokens_period3, 'end_period4': allowed_end_tokens_period4, 
                               'end_overtime': allowed_end_tokens_overtime,
                               'change_end_p1_s1': change_end_tokens_period1_situation1, 'sub_end_p1_s1': replace_end_token_period1_situation1,
                               'change_end_p1_s2': change_end_tokens_period1_situation2, 'sub_end_p1_s2': replace_end_token_period1_situation2,
                               'change_end_p1_s3': change_end_tokens_period1_situation3, 'sub_end_p1_s3': replace_end_token_period1_situation3,
                               'change_end_p1_s4': change_end_tokens_period1_situation4, 'sub_end_p1_s4': replace_end_token_period1_situation4,
                               'change_end_p2_s1': change_end_tokens_period2_situation1, 'sub_end_p2_s1': replace_end_token_period2_situation1,
                               'change_end_p2_s2': change_end_tokens_period2_situation2, 'sub_end_p2_s2': replace_end_token_period2_situation2,
                               'change_end_p2_s3': change_end_tokens_period2_situation3, 'sub_end_p2_s3': replace_end_token_period2_situation3,
                               'change_end_p2_s4': change_end_tokens_period2_situation4, 'sub_end_p2_s4': replace_end_token_period2_situation4,                               
                               'change_end_p3_s1': change_end_tokens_period3_situation1, 'sub_end_p3_s1': replace_end_token_period3_situation1,
                               'change_end_p3_s2': change_end_tokens_period3_situation2, 'sub_end_p3_s2': replace_end_token_period3_situation2,
                               'change_end_p3_s3': change_end_tokens_period3_situation3, 'sub_end_p3_s3': replace_end_token_period3_situation3,
                               'change_end_p3_s4': change_end_tokens_period3_situation4, 'sub_end_p3_s4': replace_end_token_period3_situation4,
                               'change_end_p4_s1': change_end_tokens_period4_situation1, 'sub_end_p4_s1': replace_end_token_period4_situation1,
                               'change_end_ot_s1': change_end_tokens_overtime_situation1,'sub_end_ot_s1': replace_end_token_overtime_situation1,
                               'instant_replay': instant_review_token}
                
        return dict_special_tokens

def define_ball_possession_vectorized(token, last_token, period, ball_possession):
    """
    Versão vetorizada de define_ball_possession.

    token           : lista/array de tokens (uma jogada por simulação)
    last_token      : escalar ou array (último token antes dessa jogada)
    period          : escalar ou array (período da jogada)
    ball_possession : escalar ou array (posse atual antes dessa jogada)
    dict_tokens     : dicionário com os grupos de tokens especiais

    Retorna:
        new_bp : array 1D com a nova posse de bola para cada simulação
    """
    def ensure_array(x):
        """
        Converte x para array 1D.
        Se for escalar, vira array com shape (1,).
        Se for lista ou array, converte para np.array.
        """
        x = np.asarray(x)
        if x.ndim == 0:
            x = x.reshape(1)
        return x
    
    dict_tokens = define_special_tokens(config = 1)

    token = np.asarray(token)
    if token.ndim != 1:
        raise ValueError("token deve ser 1D (lista ou array simples).")

    n = token.shape[0]

    # Se forem escalares, replicamos para todas as simulações
    if np.isscalar(last_token):
        last_token_arr = np.full(n, last_token)
    else:
        last_token_arr = np.asarray(last_token)
        if last_token_arr.shape[0] != n:
            raise ValueError("last_token deve ter o mesmo tamanho de token ou ser escalar.")

    if np.isscalar(period):
        period_arr = np.full(n, period)
    else:
        period_arr = np.asarray(period)
        if period_arr.shape[0] != n:
            raise ValueError("period deve ter o mesmo tamanho de token ou ser escalar.")

    if np.isscalar(ball_possession):
        bp = np.full(n, ball_possession)
    else:
        bp = np.asarray(ball_possession)
        if bp.shape[0] != n:
            raise ValueError("ball_possession deve ter o mesmo tamanho de token ou ser escalar.")

    new_bp = bp.copy()

    # --------------------------------
    # PERÍODO 1
    # --------------------------------
    mask_p1 = (period_arr == 1)

    if mask_p1.any():
        idx = np.where(mask_p1)[0]

        tok_p1  = token[idx]
        last_p1 = last_token_arr[idx]

        # posse ganha direto
        m_home = np.isin(tok_p1, dict_tokens['ballposs_home_p1'])
        new_bp[idx[m_home]] = 1

        m_away = np.isin(tok_p1, dict_tokens['ballposs_away_p1'])
        new_bp[idx[m_away]] = -1

        # posse "indeterminada" no começo do jogo
        m_last_und = np.isin(last_p1, dict_tokens['ballposs_und_p1'])
        if m_last_und.any():
            idx_und = idx[m_last_und]
            tok_und = token[idx_und]

            m_und_home = np.isin(tok_und, dict_tokens['ballposs_und_home_p1'])
            new_bp[idx_und[m_und_home]] = 1

            m_und_away = np.isin(tok_und, dict_tokens['ballposs_und_away_p1'])
            new_bp[idx_und[m_und_away]] = -1

    # --------------------------------
    # PERÍODO >= 5 (overtimes)
    # --------------------------------
    mask_ot = (period_arr >= 5)

    if mask_ot.any():
        idx = np.where(mask_ot)[0]

        tok_ot  = token[idx]
        last_ot = last_token_arr[idx]

        m_last_und_pot = np.isin(last_ot, dict_tokens['ballposs_und_pot'])
        if m_last_und_pot.any():
            idx_ot_und = idx[m_last_und_pot]
            tok_ot2    = token[idx_ot_und]

            m_home_pot = np.isin(tok_ot2, dict_tokens['ballposs_und_home_pot'])
            new_bp[idx_ot_und[m_home_pot]] = 1

            m_away_pot = np.isin(tok_ot2, dict_tokens['ballposs_und_away_pot'])
            new_bp[idx_ot_und[m_away_pot]] = -1

    return new_bp

def define_game_fouls(period):
    # Define o limite de faltas a depender do periodo
    foul_limit = 4 if period <= 4 else 3
    # Retorna uma tupla com o mesmo valor de limite de faltas para o mandante e o visitante
    return foul_limit, foul_limit

def build_numerical_vector_from_batch(game_status, period, config = 1):
    """
    Constrói o numerical_vector para TODAS as simulações de uma vez,
    usando o GameStatusBatch.

    Retorna: array (n_sims, n_features)
    """
    # número de simulações
    numerical_vector = update_numerical_vector(
        period = period, 
        remaining_time = game_status.remaining_time, 
        home_points = game_status.home_points, 
        away_points = game_status.away_points, 
        limhtimeout = game_status.limhtimeout, 
        limvtimeout = game_status.limvtimeout, 
        ball_possession = game_status.ball_possession,
        config = config)
    
    return numerical_vector

def update_numerical_vector(period, remaining_time, home_points, away_points, limhtimeout, limvtimeout, ball_possession, config = 1):
    
    # número de simulações
    n = remaining_time.shape[0]

    # -----------------------
    # 1) One-hot do período
    # -----------------------
    period_list = np.zeros((n, 5), dtype = int)
    if period > 0:
        col = min(period, 5) - 1
        period_list[:, col] = 1

    # -----------------------
    # 2) remaining_time_ratio = 1 - max(0, remaining_time / 7200)
    # -----------------------
    rt = remaining_time.astype(float)
    rt_ratio = 1 - np.clip(rt / 7200.0, 0, None)  # shape (n,)

    # -----------------------
    # 3) timeout_vars: indicadores de disponibilidade de timeouts
    # -----------------------
    timeout_home = (limhtimeout > 0).astype(int)
    timeout_away = (limvtimeout > 0).astype(int)

    # -----------------------
    # 4) points_diff normalizado em [-1, 1]
    # -----------------------
    diff = (home_points - away_points).astype(float)
    diff_30 = np.clip(diff, -30, 30) / 30
    diff_20 = np.clip(diff, -20, 20) / 10
    ind_close_3 = (np.abs(diff) <= 3).astype(int)
    ind_close_5 = (np.abs(diff) <= 5).astype(int)
    ind_close_8 = (np.abs(diff) <= 8).astype(int)
    ind_blowout_15 = (np.abs(diff) >= 15).astype(int)

    # -----------------------
    # 5) time_vars: indicadores com base em remaining_time
    # -----------------------
    t1200 = ((remaining_time <= 1200) & (remaining_time > 600)).astype(int)
    t600  = ((remaining_time <=  600) & (remaining_time > 200)).astype(int)
    t200  = ((remaining_time <=  200) & (remaining_time > 100)).astype(int)
    t100  = ((remaining_time <=  100) & (remaining_time <  50)).astype(int)
    t50   = ((remaining_time <=   50) & (remaining_time <  10)).astype(int)
    t10   = (remaining_time <=   10).astype(int)

    # -----------------------
    # 6) leading_var: quem está na frente (-1, 0, 1)
    # -----------------------
    lead_raw = home_points - away_points
    leading = np.sign(lead_raw).astype(int)
    home_leading = (lead_raw > 0).astype(int)
    away_leading = (lead_raw < 0).astype(int)

    t1200_game = ((t1200 == 1) & (period >= 4)).astype(int)
    t600_game = ((t600 == 1) & (period >= 4)).astype(int)
    t200_game = ((t200 == 1) & (period >= 4)).astype(int)
    t100_game = ((t100 == 1) & (period >= 4)).astype(int)
    t50_game = ((t50 == 1) & (period >= 4)).astype(int)

    ind_clutch_2_last_minutes = ((t1200_game == 1) & (ind_blowout_15 == 0)).astype(int) 
    ind_clutch_2_last_minutes_home = ((ind_clutch_2_last_minutes == 1) & (home_leading == 1)).astype(int) 
    ind_clutch_2_last_minutes_away = ((ind_clutch_2_last_minutes == 1) & (away_leading == 1)).astype(int) 

    ind_clutch_last_minute = ((t600_game == 1) & (ind_close_8 == 1)).astype(int) 
    ind_clutch_last_minute_home = ((ind_clutch_last_minute == 1) & (home_leading == 1)).astype(int) 
    ind_clutch_last_minute_away = ((ind_clutch_last_minute == 1) & (away_leading == 1)).astype(int) 

    ind_clutch_last_20_seconds = ((t200_game == 1) & (ind_close_5 == 1)).astype(int) 
    ind_clutch_last_20_seconds_home = ((t200_game == 1) & (home_leading == 1)).astype(int) 
    ind_clutch_last_20_seconds_away = ((t200_game == 1) & (away_leading == 1)).astype(int) 

    ind_clutch_last_10_seconds = ((t100_game  == 1) & (ind_close_3 == 1)).astype(int) 
    ind_clutch_last_10_seconds_home = ((t100_game == 1) & (home_leading == 1)).astype(int) 
    ind_clutch_last_10_seconds_away = ((t100_game == 1) & (away_leading == 1)).astype(int) 

    ind_blowout_last_minute = ((t600_game == 1) & (ind_blowout_15 == 1)).astype(int)

    # -----------------------
    # 7) ball_possession_var
    # -----------------------
    bp = ball_possession  # já é -1, 0, 1

    # -----------------------
    # 8) empilhar tudo
    # -----------------------

    if config == 1:
        # 16 Variaveis
        numerical_vector = np.column_stack([
            period_list,      # 5 colunas
            rt_ratio,         # 1
            timeout_home,     # 1
            timeout_away,     # 1
            diff_30,          # 1
            t600,             # 1
            t200,             # 1
            t100,             # 1
            t50,              # 1
            t10,              # 1
            leading,          # 1
            bp,               # 1
        ])

    elif config == 2:

        numerical_vector = np.column_stack([
            period_list,      # 5 colunas
            rt_ratio,         # 1
            timeout_home,     # 1
            timeout_away,     # 1
            diff_20,          # 1
            ind_close_3,      # 1
            ind_close_5,      # 1
            ind_close_8,      # 1
            ind_blowout_15,   # 1
            t1200,            # 1
            t600,             # 1
            t200,             # 1
            t100,             # 1
            t50,              # 1
            t10,              # 1
            home_leading,     # 1
            away_leading,     # 1
            t1200_game,       # 1
            t600_game,        # 1
            t200_game,        # 1
            t100_game,        # 1
            t50_game,         # 1
            ind_clutch_2_last_minutes,       # 1 
            ind_clutch_2_last_minutes_home,  # 1 
            ind_clutch_2_last_minutes_away,  # 1 
            ind_clutch_last_minute,          # 1 
            ind_clutch_last_minute_home,     # 1 
            ind_clutch_last_minute_away,     # 1 
            ind_clutch_last_20_seconds,      # 1 
            ind_clutch_last_20_seconds_home, # 1 
            ind_clutch_last_20_seconds_away, # 1 
            ind_clutch_last_10_seconds,      # 1 
            ind_clutch_last_10_seconds_home, # 1 
            ind_clutch_last_10_seconds_away, # 1 
            ind_blowout_last_minute,         # 1 
            bp,               # 1
        ])
    # total de features = 5 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 + 1 = 40    
    else:
        # 22 variaveis
        numerical_vector = np.column_stack([
            timeout_home,     # 1
            timeout_away,     # 1
            ind_close_3,      # 1
            ind_close_5,      # 1
            ind_close_8,      # 1
            ind_blowout_15,   # 1
            t1200,            # 1
            t600,             # 1
            t200,             # 1
            t100,             # 1
            t50,              # 1
            t10,              # 1
            home_leading,     # 1
            away_leading,     # 1
            bp,               # 1
            period_list,      # 5 colunas
            rt_ratio,         # 1
            diff_20           # 1            
        ])        


    return numerical_vector

def update_vectors(period, random_play, tokens_vector, home_points, away_points, limhtimeout, limvtimeout, remaining_time, ball_possession, indices = None, config = 1):
    
    dict_tokens = define_special_tokens(config = 1)
    if indices is None:
        random_play = np.asarray(random_play)
        bp_sub  = ball_possession
    else:
        random_play = np.asarray(random_play)[indices]
        bp_sub = ball_possession[indices] 

    # máscaras para cada tipo de jogada
    m_home1   = np.isin(random_play, dict_tokens['home_pts1'])
    m_home2   = np.isin(random_play, dict_tokens['home_pts2'])
    m_home3   = np.isin(random_play, dict_tokens['home_pts3'])
    m_htimeout = np.isin(random_play, dict_tokens['home_timeouts'])

    m_away1   = np.isin(random_play, dict_tokens['away_pts1'])
    m_away2   = np.isin(random_play, dict_tokens['away_pts2'])
    m_away3   = np.isin(random_play, dict_tokens['away_pts3'])
    m_atimeout = np.isin(random_play, dict_tokens['away_timeouts'])

    m_ball_h = np.isin(random_play, dict_tokens['ballposs_und_home_p1']) & (bp_sub == 0)
    m_ball_a = np.isin(random_play, dict_tokens['ballposs_und_away_p1']) & (bp_sub == 0)

    if indices is None:
        # --- HOME ---
        home_points[m_home1] += 1
        home_points[m_home2] += 2
        home_points[m_home3] += 3
        limhtimeout[m_htimeout] -= 1
        ball_possession[m_ball_h] = 1
        
        # --- AWAY ---
        away_points[m_away1] += 1
        away_points[m_away2] += 2
        away_points[m_away3] += 3
        limvtimeout[m_atimeout] -= 1        
        ball_possession[m_ball_a] = -1
    else:
        # --- HOME ---
        home_points[indices[m_home1]] += 1
        home_points[indices[m_home2]] += 2
        home_points[indices[m_home3]] += 3
        limhtimeout[indices[m_htimeout]] -= 1
        ball_possession[indices[m_ball_h]] = 1

        # --- AWAY ---
        away_points[indices[m_away1]] += 1
        away_points[indices[m_away2]] += 2
        away_points[indices[m_away3]] += 3
        limvtimeout[indices[m_atimeout]] -= 1      
        ball_possession[indices[m_ball_a]] = -1

    if indices is None:
        tokens = np.asarray(tokens_vector)
    else:
        tokens = np.asarray(tokens_vector)[indices]

    _tokens = np.empty_like(tokens)
    _tokens[:, :-1] = tokens[:, 1:]
    _tokens[:, -1] = random_play

    tokens_vector = _tokens

    if indices is None:
        numerical_vector = update_numerical_vector(
            period = period, 
            remaining_time = remaining_time, 
            home_points = home_points, 
            away_points = away_points, 
            limhtimeout = limhtimeout, 
            limvtimeout = limvtimeout, 
            ball_possession = ball_possession,
            config = config)
    else:
        numerical_vector = update_numerical_vector(
            period = period, 
            remaining_time = remaining_time[indices], 
            home_points = home_points[indices], 
            away_points = away_points[indices], 
            limhtimeout = limhtimeout[indices], 
            limvtimeout = limvtimeout[indices], 
            ball_possession = ball_possession[indices],
            config = config)
    
    return tokens_vector, numerical_vector

def update_game_status(game_status, random_play, period, config = 1, indices = None, inplace = False):
    """
    Atualiza pontos, limites de faltas, token_vector e numerical_vector
    de um GameStatusBatch, de forma vetorizada.

    Parâmetros
    ----------
    game_status : GameStatusBatch
        Estado atual (batelada de simulações).
    random_play : array-like (n_sims,) ou (len(indices),)
        Token sorteado da jogada para cada simulação.
        Se `indices` for None, o tamanho deve ser n_sims.
        Se `indices` não for None, o tamanho deve ser len(indices).
    period : int ou array-like
        Período atual (usado pelo build_numerical_vector_from_batch).
    indices : array-like de int, opcional (default=None)
        Lista de índices das simulações que serão atualizadas.
        - Se None  -> atualiza todas.
        - Se não None -> atualiza apenas essas posições.
    inplace : bool (default=False)
        - True  -> modifica `game_status` no lugar e o retorna.
        - False -> trabalha em uma cópia e retorna a cópia.

    Retorna
    -------
    GameStatus
        Objeto atualizado.
    """
    dict_tokens = define_special_tokens(config=1)
    random_play = np.asarray(random_play)

    # copia ou não
    g = game_status if inplace else game_status.copy()

    # ---------------------------------------------------
    # Trata indices
    # ---------------------------------------------------
    if indices is None:
        # caso antigo: todas as simulações
        if random_play.shape[0] != g.n_sims:
            raise ValueError(
                f"random_play deve ter tamanho n_sims = {g.n_sims} quando indices = None, "
                f"mas veio com {random_play.shape[0]}."
            )
        idx = None
        rp = random_play  # abreviação
        ball_poss_sub = g.ball_possession
    else:
        # atualiza apenas um subconjunto de simulações
        idx = np.asarray(indices)
        if random_play.shape[0] != idx.shape[0]:
            raise ValueError(
                f"random_play deve ter o mesmo tamanho que indices. "
                f"len(random_play)={random_play.shape[0]}, len(indices)={idx.shape[0]}."
            )
        rp = random_play  # só para deixar o nome mais curto
        ball_poss_sub = g.ball_possession[idx]

    # ---------------------------------------------------
    # Máscaras para cada tipo de jogada (sempre no vetor rp)
    # ---------------------------------------------------
    m_home1 = np.isin(rp, dict_tokens["home_pts1"])
    m_home2 = np.isin(rp, dict_tokens["home_pts2"])
    m_home3 = np.isin(rp, dict_tokens["home_pts3"])
    m_hfoul = np.isin(rp, dict_tokens["home_fouls"])
    m_htimeout = np.isin(rp, dict_tokens["home_timeouts"])

    m_away1 = np.isin(rp, dict_tokens["away_pts1"])
    m_away2 = np.isin(rp, dict_tokens["away_pts2"])
    m_away3 = np.isin(rp, dict_tokens["away_pts3"])
    m_afoul = np.isin(rp, dict_tokens["away_fouls"])
    m_atimeout = np.isin(rp, dict_tokens["away_timeouts"])

    m_ball_h = np.isin(random_play, dict_tokens['ballposs_und_home_p1']) & (ball_poss_sub == 0)
    m_ball_a = np.isin(random_play, dict_tokens['ballposs_und_away_p1']) & (ball_poss_sub == 0)

    g.period.fill(period)
    # ---------------------------------------------------
    # Atualiza HOME / AWAY
    # ---------------------------------------------------
    if idx is None:
        # Atualiza direto, como antes
        g.home_points[m_home1] += 1
        g.home_points[m_home2] += 2
        g.home_points[m_home3] += 3
        g.limhteam[m_hfoul] -= 1
        g.limhtimeout[m_htimeout] -= 1
        g.ball_possession[m_ball_h] = 1

        g.away_points[m_away1] += 1
        g.away_points[m_away2] += 2
        g.away_points[m_away3] += 3
        g.limvteam[m_afoul] -= 1
        g.limvtimeout[m_atimeout] -= 1
        g.ball_possession[m_ball_a] = -1
    else:
        # Só nas sims selecionadas (usa idx[mascara_local])
        g.home_points[idx[m_home1]] += 1
        g.home_points[idx[m_home2]] += 2
        g.home_points[idx[m_home3]] += 3
        g.limhteam[idx[m_hfoul]] -= 1
        g.limhtimeout[idx[m_htimeout]] -= 1
        g.ball_possession[idx[m_ball_h]] = 1

        g.away_points[idx[m_away1]] += 1
        g.away_points[idx[m_away2]] += 2
        g.away_points[idx[m_away3]] += 3
        g.limvteam[idx[m_afoul]] -= 1
        g.limvtimeout[idx[m_atimeout]] -= 1
        g.ball_possession[idx[m_ball_a]] = -1

    # ---------------------------------------------------
    # Atualiza token_vector (shift + joga random_play no fim)
    # ---------------------------------------------------
    if idx is None:
        # Todas as sims
        tokens = g.token_vector
        _tokens = np.empty_like(tokens)
        _tokens[:, :-1] = tokens[:, 1:]
        _tokens[:, -1] = rp
        g.token_vector = _tokens
    else:
        # Apenas as linhas selecionadas
        tokens_sub = g.token_vector[idx]          # (num_valid, window)
        _tokens_sub = np.empty_like(tokens_sub)
        _tokens_sub[:, :-1] = tokens_sub[:, 1:]
        _tokens_sub[:, -1] = rp
        g.token_vector[idx] = _tokens_sub         # grava de volta só nessas linhas

    # ---------------------------------------------------
    # Recalcula numerical_vector com base no estado atual g
    # (pode usar o próprio g para computar tudo, inclusive
    #  as sims não atualizadas continuam com seus valores antigos)
    # ---------------------------------------------------
    numerical_vector = build_numerical_vector_from_batch(g, period, config = config)
    g.numerical_vector = numerical_vector

    return g

def change_fouls_limit_two_final_minutes(game_status, valid_simulations, list_limhteam, list_limvteam):
    
    # CORRECAO: atualizando o limite de faltas.
    # Aqui a correcao nao sera feita pelo risco de se sortear alguma jogada indesejavel, mas sim pela dinamica do jogo.
    mask_two_final_minuts = game_status.remaining_time <= 1200
    idx_two_final_minutes = np.where(mask_two_final_minuts)[0]
    idx_two_final_minutes = idx_two_final_minutes[np.isin(idx_two_final_minutes, valid_simulations)]

    if len(idx_two_final_minutes) == 0:
        return

    idx_same_num_hfouls = np.where(list_limhteam == game_status.limhteam)[0]
    idx_same_num_hfouls = idx_same_num_hfouls[np.isin(idx_same_num_hfouls, idx_two_final_minutes)]

    idx_diff_num_hfouls = np.where(list_limhteam != game_status.limhteam)[0]
    idx_diff_num_hfouls = idx_diff_num_hfouls[np.isin(idx_diff_num_hfouls, idx_two_final_minutes)]

    game_status.limhteam[idx_same_num_hfouls] = np.minimum(1, game_status.limhteam[idx_same_num_hfouls])
    game_status.limhteam[idx_diff_num_hfouls] = np.minimum(0, game_status.limhteam[idx_diff_num_hfouls])

    idx_same_num_afouls = np.where(list_limvteam == game_status.limvteam)[0]
    idx_same_num_afouls = idx_same_num_afouls[np.isin(idx_same_num_afouls, idx_two_final_minutes)]

    idx_diff_num_afouls = np.where(list_limvteam != game_status.limvteam)[0]
    idx_diff_num_afouls = idx_diff_num_afouls[np.isin(idx_diff_num_afouls, idx_two_final_minutes)]

    game_status.limvteam[idx_same_num_afouls] = np.minimum(1, game_status.limvteam[idx_same_num_afouls])
    game_status.limvteam[idx_diff_num_afouls] = np.minimum(0, game_status.limvteam[idx_diff_num_afouls])

    return 

def change_timeouts_limits(game_status, valid_simulations, list_limhtimeout, list_limvtimeout):
    
    # CORRECAO: atualizando o limite de timeouts.
    # Aqui a correcao nao sera feita pelo risco de se sortear alguma jogada indesejavel, mas sim pela dinamica do jogo.
    mask_three_final_minutes_game = (game_status.remaining_time <= 1800) & (game_status.period == 4)
    idx_three_final_minutes_game = np.where(mask_three_final_minutes_game)[0]
    idx_three_final_minutes_game = idx_three_final_minutes_game[np.isin(idx_three_final_minutes_game, valid_simulations)]

    if len(idx_three_final_minutes_game) == 0:
        return

    idx_same_num_htimeouts = np.where(list_limhtimeout == game_status.limhtimeout)[0]
    idx_same_num_htimeouts = idx_same_num_htimeouts[np.isin(idx_same_num_htimeouts, idx_three_final_minutes_game)]

    idx_diff_num_htimeouts = np.where(list_limhtimeout != game_status.limhtimeout)[0]
    idx_diff_num_htimeouts = idx_diff_num_htimeouts[np.isin(idx_diff_num_htimeouts, idx_three_final_minutes_game)]

    game_status.limhtimeout[idx_same_num_htimeouts] = np.minimum(2, game_status.limhtimeout[idx_same_num_htimeouts])
    game_status.limhtimeout[idx_diff_num_htimeouts] = np.minimum(1, game_status.limhtimeout[idx_diff_num_htimeouts])

    idx_same_num_atimeouts = np.where(list_limvtimeout == game_status.limvtimeout)[0]
    idx_same_num_atimeouts = idx_same_num_atimeouts[np.isin(idx_same_num_atimeouts, idx_three_final_minutes_game)]

    idx_diff_num_atimeouts = np.where(list_limvtimeout != game_status.limvtimeout)[0]
    idx_diff_num_atimeouts = idx_diff_num_atimeouts[np.isin(idx_diff_num_atimeouts, idx_three_final_minutes_game)]

    game_status.limvtimeout[idx_same_num_atimeouts] = np.minimum(2, game_status.limvtimeout[idx_same_num_atimeouts])
    game_status.limvtimeout[idx_diff_num_atimeouts] = np.minimum(1, game_status.limvtimeout[idx_diff_num_atimeouts])

    return 

def describe_variables(arr, name):
    """
    Calcula estatísticas descritivas via NumPy (rápido)
    e retorna um DataFrame pandas exatamente no estilo do describe().
    """
    arr = np.asarray(arr)

    stats = {
        "count": arr.size,
        "mean":  arr.mean(),
        "std":   arr.std(),
        "min":   arr.min(),
        " 1%":   np.percentile(arr, 1),
        " 5%":   np.percentile(arr, 5),
        "10%":   np.percentile(arr, 10),
        "25%":   np.percentile(arr, 25),
        "50%":   np.percentile(arr, 50),
        "75%":   np.percentile(arr, 75),
        "90%":   np.percentile(arr, 90),
        "95%":   np.percentile(arr, 95),
        "99%":   np.percentile(arr, 99),
        "max":   arr.max(),
    }

    # Constrói um DataFrame de 1 coluna igual ao pandas.describe()
    df = pd.DataFrame(stats, index = [name])
    return df

def print_status(game_status, period):
    clear_output(wait = True)
    
    period_duration = 7200 if period <= 4 else 3000
    values, counts = np.unique(game_status.keep_generating, return_counts = True)
    dist = {int(v): int(c) for v, c in zip(values, counts)}
    print(f"Period: {period}\nkeep_generating vector distribution: {dist}")

    perc_played_time = np.round(100 * (period_duration - game_status.remaining_time) / period_duration, 1)

    df_stats = pd.concat([
        describe_variables(arr = game_status.home_points, name = "home_points"),
        describe_variables(arr = game_status.away_points, name = "away_points"),
        describe_variables(arr = perc_played_time, name = "perc_played_time")])

    print(df_stats)

    values = np.sign(game_status.home_points - game_status.away_points)
    values, counts = np.unique(values, return_counts = True)
    lookup = { -1: "away", 0: "tie", 1: "home" }
    labels = np.vectorize(lookup.get)(values)
    dist = {v: int(c) for v, c in zip(labels, counts)}
    print(f"Result: {dist}")
    return 


In [ ]:
def categorize_time_bins_np(period_arr, remaining_time_arr, diff_points_arr):
    period_raw = period_arr.astype(int)
    period_length = np.where(period_raw > 4, 5 * 600, 12 * 600)

    time_bin = np.zeros(period_raw.shape[0], dtype = int)

    mask0 = remaining_time_arr > (period_length - 600)
    time_bin[mask0] = 0
    mask1 = (~mask0) & (remaining_time_arr > 1200)
    time_bin[mask1] = 1
    mask2 = (~mask0) & (~mask1) & (remaining_time_arr > 600)
    time_bin[mask2] = 2
    mask3 = (~mask0) & (~mask1) & (~mask2) & (remaining_time_arr > 200)
    time_bin[mask3] = 3
    mask4 = (~mask0) & (~mask1) & (~mask2) & (~mask3)
    time_bin[mask4] = 4

    period_bin = np.where(period_raw > 4, 4, period_raw - 1)
    cat_time = period_bin * 5 + time_bin

    diff = diff_points_arr.astype(int)
    diff_bin = np.zeros(diff.shape[0], dtype = int)
    diff_bin[diff <= -20] = 0
    diff_bin[(diff > -20) & (diff < -10)] = 1
    diff_bin[(diff >= -10) & (diff < -6)] = 2
    diff_bin[(diff >= -6) & (diff < -3)] = 3
    diff_bin[(diff >= -3) & (diff <= 3)] = 4
    diff_bin[(diff > 3) & (diff <= 6)] = 5
    diff_bin[(diff > 6) & (diff <= 10)] = 6
    diff_bin[(diff > 10) & (diff < 20)] = 7
    diff_bin[diff >= 20] = 8

    return (cat_time * 9 + diff_bin).astype(np.int64)


In [ ]:
def predict_model(model, token_vector, numerical_vector, k = 0, only_consider = None, extra_info = True, type_ = 'tokens', mask_data = None):
    # Usada para sortear a proxima jogada ou o proximo tempo gasto
    # Atualizando o tipo dos vetores e a dimensao, se necessario.
    if isinstance(token_vector, list):
        token_vector = np.array(token_vector, dtype = "int32")
    if isinstance(numerical_vector, list):
        numerical_vector = np.array(numerical_vector)
    if len(token_vector.shape) == 1:
        token_vector = token_vector.reshape(1, -1)
        numerical_vector = numerical_vector.reshape(1, -1)

    if not isinstance(token_vector, torch.Tensor):
        token_vector = torch.tensor(token_vector)
    if not isinstance(numerical_vector, torch.Tensor):
        numerical_vector = torch.tensor(numerical_vector, dtype = torch.float32)
        
    # gerando os log-valores para os tokens de jogada ou tempo a partir dos vetores de input.
    with torch.inference_mode():
        logits = model(token_vector, numerical_vector)
        if mask_data is not None:
            logits = logits.masked_fill(mask_data, -1e9)

        # se for usada para sortear a proxima jogada, nao permite que o token 0 seja sorteado (token de inicializacao)
        if type_ == 'tokens': logits[:, 0] = -float("inf")

        # se quisermos sortear uma jogada apenas considerando alguns tokens, então zera a probabilidade dos que não estão sendo considerados
        if only_consider is not None:
            allowed = torch.isin(torch.arange(logits.size(1)), torch.tensor(only_consider))
            logits[:, ~allowed] = -float('inf')

        # transforma tudo em probabilidade
        probabilities = torch.softmax(logits, dim = 1)

        # Se for necessario ter uma lista das probabilidades que o modelo atribui a cada token, k precisa ser maior que 0.
        if k > 0:
            topk = torch.topk(probabilities, k, dim = 1)
            topk_preds = {
                'indices': list(topk.indices.detach().numpy()[0]), 
                'probs': list(topk.values.detach().numpy()[0])
                }
            return pd.DataFrame(topk_preds) #topk_preds 

        # se a funcao foi chamada com apenas um vetor de input entao so retorna uma jogada, caso contrario, retorna jogadas para todos os vetores de input
        sorted_items = torch.multinomial(input = probabilities, num_samples = 1).squeeze(1).detach().cpu().tolist()
        sorted_probs = probabilities[torch.arange(probabilities.size(0)), sorted_items].detach().cpu().tolist()
            # se solicitamos extra_info, entao além da jogada tambem retorna a probabilidade associada a essa jogada
        if extra_info: return sorted_items, sorted_probs
        return sorted_items

def predict_model_st(model, token_vector, numerical_vector, state_token_vector, k = 0, only_consider = None, type_ = 'tokens', extra_info = True, mask_data = None):

    if isinstance(token_vector, list):
        token_vector = np.array(token_vector, dtype = "int32")
    if isinstance(numerical_vector, list):
        numerical_vector = np.array(numerical_vector)
    if isinstance(state_token_vector, list):
        state_token_vector = np.array(state_token_vector, dtype = "int32")
        
    if len(token_vector.shape) == 1:
        token_vector = token_vector.reshape(1, -1)
        numerical_vector = numerical_vector.reshape(1, -1)
    if state_token_vector.ndim == 0:
        state_token_vector = state_token_vector.reshape(1)
    if state_token_vector.shape[0] == 1 and token_vector.shape[0] > 1:
        state_token_vector = np.full(token_vector.shape[0], state_token_vector[0])


    if not isinstance(token_vector, torch.Tensor):
        token_vector = torch.tensor(token_vector)
    if not isinstance(numerical_vector, torch.Tensor):
        numerical_vector = torch.tensor(numerical_vector, dtype = torch.float32)
    if not isinstance(state_token_vector, torch.Tensor):
        state_token_vector = torch.tensor(state_token_vector)    

    # inferência de verdade: sem grafo, mais rápido e sem vazamento de memória
    with torch.inference_mode():
        logits = model(token_vector, numerical_vector, state_token_vector)
        if mask_data is not None:
            logits = logits.masked_fill(mask_data, -1e9)

        if type_ == 'tokens': logits[:, 0] = -float("inf")

        if only_consider is not None:
            allowed = torch.isin(torch.arange(logits.size(1), device=logits.device), torch.tensor(only_consider, device=logits.device))
            logits[:, ~allowed] = -float("inf")

        probabilities = torch.softmax(logits, dim = 1)
        if k > 0:
            topk = torch.topk(probabilities, k, dim = 1)
            topk_preds = {
                'indices': list(topk.indices.detach().numpy()[0]),
                'probs':   list(topk.values.detach().numpy()[0])
                }
            return pd.DataFrame(topk_preds)
    
        sorted_items = torch.multinomial(input = probabilities, num_samples = 1).squeeze(1).detach().cpu().tolist()
        sorted_probs = probabilities[torch.arange(probabilities.size(0)), sorted_items].detach().cpu().tolist()

        if extra_info: return sorted_items, sorted_probs
        return sorted_items

def predict_model_timediff_tower(model, token_vector, numerical_vector, time_diff_features, k = 0, only_consider = None, extra_info = True, type_ = 'tokens', mask_data = None):
    
    if isinstance(token_vector, list):
        token_vector = np.array(token_vector, dtype = "int32")
    if isinstance(numerical_vector, list):
        numerical_vector = np.array(numerical_vector)
    if isinstance(time_diff_features, list):
        time_diff_features = np.array(time_diff_features)

    if len(token_vector.shape) == 1:
        token_vector = token_vector.reshape(1, -1)
        numerical_vector = numerical_vector.reshape(1, -1)
        time_diff_features = time_diff_features.reshape(1, -1)

    if not isinstance(token_vector, torch.Tensor):
        token_vector = torch.tensor(token_vector)
    if not isinstance(numerical_vector, torch.Tensor):
        numerical_vector = torch.tensor(numerical_vector, dtype=torch.float32)
    if not isinstance(time_diff_features, torch.Tensor):
        time_diff_features = torch.tensor(time_diff_features, dtype=torch.float32)

    with torch.inference_mode():
        logits = model(token_vector, time_diff_features, numerical_vector)
        if mask_data is not None:
            logits = logits.masked_fill(mask_data, -1e9)

        if type_ == 'tokens': logits[:, 0] = -float("inf")

        if only_consider is not None:
            allowed = torch.isin(torch.arange(logits.size(1)), torch.tensor(only_consider))
            logits[:, ~allowed] = -float("inf")

        probabilities = torch.softmax(logits, dim = 1)

        if k > 0:
            topk = torch.topk(probabilities, k, dim = 1)
            topk_preds = {
                'indices': list(topk.indices.detach().numpy()[0]),
                'probs':   list(topk.values.detach().numpy()[0])
                }
            return pd.DataFrame(topk_preds)

        sorted_items = torch.multinomial(input = probabilities, num_samples = 1).squeeze(1).detach().cpu().tolist()
        sorted_probs = probabilities[torch.arange(probabilities.size(0)), sorted_items].detach().cpu().tolist()
        
        if extra_info: return sorted_items, sorted_probs
        return sorted_items

def predict_model_multi_tower(model, token_vector, numerical_vector, k = 0, only_consider = None, extra_info = True, type_ = 'tokens', mask_data = None):
    
    if isinstance(token_vector, list):
        token_vector = np.array(token_vector, dtype = "int32")
    
    for idx in range(len(numerical_vector)):
        if isinstance(numerical_vector[idx], list):
            numerical_vector[idx] = np.array(numerical_vector[idx])

    if len(token_vector.shape) == 1:
        token_vector = token_vector.reshape(1, -1)
        for idx in range(len(numerical_vector)):
            numerical_vector[idx] = numerical_vector[idx].reshape(1, -1)

    if not isinstance(token_vector, torch.Tensor):
        token_vector = torch.tensor(token_vector)
    for idx in range(len(numerical_vector)):
        if not isinstance(numerical_vector[idx], torch.Tensor):
            numerical_vector[idx] = torch.tensor(numerical_vector[idx], dtype=torch.float32)    

    with torch.inference_mode():      
        logits = model(token_ids = token_vector, numerical_groups = numerical_vector)
        if mask_data is not None:
            logits = logits.masked_fill(mask_data, -1e9)


        if type_ == 'tokens': logits[:, 0] = -float("inf")

        if only_consider is not None:
            allowed = torch.isin(torch.arange(logits.size(1)), torch.tensor(only_consider))
            logits[:, ~allowed] = -float("inf")

        probabilities = torch.softmax(logits, dim = 1)

        if k > 0:
            topk = torch.topk(probabilities, k, dim = 1)
            topk_preds = {
                'indices': list(topk.indices.detach().numpy()[0]),
                'probs':   list(topk.values.detach().numpy()[0])
                }
            return pd.DataFrame(topk_preds)

        sorted_items = torch.multinomial(input = probabilities, num_samples = 1).squeeze(1).detach().cpu().tolist()
        sorted_probs = probabilities[torch.arange(probabilities.size(0)), sorted_items].detach().cpu().tolist()
        
        if extra_info: return sorted_items, sorted_probs
        return sorted_items


In [ ]:
def random_first_play(period, game_status, valid_simulations, token_model, config = 1, match_vector = None):
    # Gerando a primeira jogada do periodo
    dict_tokens = define_special_tokens(config = 1)

    token_vector = game_status.token_vector[valid_simulations]
    numerical_vector = game_status.numerical_vector[valid_simulations]  

    if match_vector is None:
        numerical_vector_ = numerical_vector
    else:
        numerical_vector_ = np.hstack([numerical_vector, match_vector])
            
    if period == 1:
        # se for a primeira jogada do jogo, entao é preciso criar o contexto do jogo também
        random_play, prob_random_play = predict_model(model = token_model, token_vector = token_vector,
                                                      numerical_vector = numerical_vector_, 
                                                      only_consider = dict_tokens['start_period1'])
        
        # game_status = [create_game_status(random_play = play) for play in random_play]
        game_status.ball_possession[valid_simulations] = define_ball_possession_vectorized(
            token = random_play, last_token = 0, period = period, ball_possession = 0)

    else:
        # se estamos no inicio de qualquer outro periodo, entao precisamos usar o contexto do jogo atual para considerar a pontuacao das equipes
        if period == 2: 
            tokens_list = dict_tokens['start_period2'] * token_vector.shape[0]
        elif period == 3: 
            tokens_list = dict_tokens['start_period3'] * token_vector.shape[0]
        elif period == 4: 
            tokens_list = dict_tokens['start_period4'] * token_vector.shape[0]
        else: 
            tokens_list = dict_tokens['start_overtime'] * token_vector.shape[0]

        # sorteia a primeira jogada de qualquer periodo que nao o primeiro
        random_play, prob_random_play = predict_model(model = token_model, token_vector = token_vector,
                                                      numerical_vector = numerical_vector_, 
                                                      only_consider = tokens_list)

        period_duration = 7200 if period <= 4 else 3000
        allowed_fouls_home, allowed_fouls_away = define_game_fouls(period)

        game_status.period[valid_simulations] = period
        game_status.limhteam[valid_simulations] = allowed_fouls_home
        game_status.limvteam[valid_simulations] = allowed_fouls_away
        game_status.remaining_time[valid_simulations] = period_duration

        if period >= 5:
            game_status.ball_possession[valid_simulations] = 0
            
            game_status.limhtimeout[valid_simulations] = 2
            game_status.limvtimeout[valid_simulations] = 2

        if period == 4:
            num_timeouts_home = np.minimum(game_status.limhtimeout[valid_simulations], 4)
            num_timeouts_away = np.minimum(game_status.limvtimeout[valid_simulations], 4)

            game_status.limhtimeout[valid_simulations] = num_timeouts_home
            game_status.limvtimeout[valid_simulations] = num_timeouts_away
    
    # a partir dos valores sorteados, atualiza o game_status
    update_game_status(game_status = game_status, random_play = random_play, period = period, 
                       indices = valid_simulations, inplace = True, config = config)

    game_status.num_plays[valid_simulations] += 1
    game_status.sorted_token[valid_simulations] = np.asarray(random_play)
    game_status.prob_token[valid_simulations] = np.asarray(prob_random_play)
    
    # sempre a primeira jogada do primeiro periodo "gasta" tempo zero
    game_status.sorted_time[valid_simulations] = 0
    game_status.prob_time[valid_simulations] = 1

    return 

def random_first_play_st(period, game_status, valid_simulations, token_model, config = 1, match_vector = None):

    dict_tokens = define_special_tokens(config = 1)

    token_vector = game_status.token_vector[valid_simulations]
    numerical_vector = game_status.numerical_vector[valid_simulations]

    # match vector (igual ao original)
    if match_vector is None:
        numerical_vector_ = numerical_vector
    else:
        numerical_vector_ = np.hstack([numerical_vector, match_vector])

    # STATE TOKEN para prever play_i (estado atual = pós (play_{i-1}, time_{i-1}))
    state_id_token = categorize_time_bins_np(
        period_arr = np.full(game_status.remaining_time[valid_simulations].shape[0], period, dtype = int),
        remaining_time_arr = game_status.remaining_time[valid_simulations].astype(int),
        diff_points_arr = (game_status.home_points[valid_simulations] - game_status.away_points[valid_simulations]).astype(int))

    if period == 1:
        # primeira jogada do jogo: restrição start_period1
        random_play, prob_random_play = predict_model_st(model = token_model, token_vector = token_vector, 
                                                         numerical_vector = numerical_vector_,
                                                         state_token_vector = state_id_token, 
                                                         only_consider = dict_tokens['start_period1'])

        # posse inicial (igual ao original)
        game_status.ball_possession[valid_simulations] = define_ball_possession_vectorized(
            token = random_play, last_token = 0, period = period, ball_possession = 0)

    else:
        # início de outros períodos: restrição por período (igual ao original)
        if period == 2:
            tokens_list = dict_tokens['start_period2'] * token_vector.shape[0]
        elif period == 3:
            tokens_list = dict_tokens['start_period3'] * token_vector.shape[0]
        elif period == 4:
            tokens_list = dict_tokens['start_period4'] * token_vector.shape[0]
        else:
            tokens_list = dict_tokens['start_overtime'] * token_vector.shape[0]

        random_play, prob_random_play = predict_model_st(model = token_model, token_vector = token_vector, 
                                                         numerical_vector = numerical_vector_,
                                                         state_token_vector = state_id_token, 
                                                         only_consider = tokens_list)
        
        # reset de relógio/faltas (igual ao original, mesmo que redundante)
        period_duration = 7200 if period <= 4 else 3000
        allowed_fouls_home, allowed_fouls_away = define_game_fouls(period)

        game_status.period[valid_simulations] = period
        game_status.limhteam[valid_simulations] = allowed_fouls_home
        game_status.limvteam[valid_simulations] = allowed_fouls_away
        game_status.remaining_time[valid_simulations] = period_duration

        if period >= 5:
            game_status.ball_possession[valid_simulations] = 0
            game_status.limhtimeout[valid_simulations] = 2
            game_status.limvtimeout[valid_simulations] = 2

        if period == 4:
            num_timeouts_home = np.minimum(game_status.limhtimeout[valid_simulations], 4)
            num_timeouts_away = np.minimum(game_status.limvtimeout[valid_simulations], 4)

            game_status.limhtimeout[valid_simulations] = num_timeouts_home
            game_status.limvtimeout[valid_simulations] = num_timeouts_away

    # a partir dos valores sorteados, atualiza o game_status
    update_game_status(game_status = game_status, random_play = random_play, period = period, 
                       indices = valid_simulations, inplace = True, config = config)

    game_status.num_plays[valid_simulations] += 1
    game_status.sorted_token[valid_simulations] = np.asarray(random_play)
    game_status.prob_token[valid_simulations] = np.asarray(prob_random_play)
    
    # sempre a primeira jogada do primeiro periodo "gasta" tempo zero
    game_status.sorted_time[valid_simulations] = 0
    game_status.prob_time[valid_simulations] = 1

    return

def random_first_play_timediff_tower(period, game_status, valid_simulations, token_model, config = 3, match_vector = None):

    dict_tokens = define_special_tokens(config = 1)

    token_vector = game_status.token_vector[valid_simulations]
    numerical_vector = game_status.numerical_vector[valid_simulations]

    if config == 3:
        num_numerical_variables = 15
    # match vector (igual ao original)
    if match_vector is None:
        numerical_vector_ = numerical_vector[:, :num_numerical_variables]
    else:
        numerical_vector_ = np.hstack([numerical_vector[:, :num_numerical_variables], match_vector])

    time_diff_vector_ = numerical_vector[:, num_numerical_variables:]

    if period == 1:
        # primeira jogada do jogo: restrição start_period1
        random_play, prob_random_play = predict_model_timediff_tower(model = token_model, token_vector = token_vector, 
                                                                     numerical_vector = numerical_vector_,
                                                                     time_diff_features = time_diff_vector_,
                                                                     only_consider = dict_tokens['start_period1'])

        # posse inicial (igual ao original)
        game_status.ball_possession[valid_simulations] = define_ball_possession_vectorized(
            token = random_play, last_token = 0, period = period, ball_possession = 0)

    else:
        # início de outros períodos: restrição por período (igual ao original)
        if period == 2:
            tokens_list = dict_tokens['start_period2'] * token_vector.shape[0]
        elif period == 3:
            tokens_list = dict_tokens['start_period3'] * token_vector.shape[0]
        elif period == 4:
            tokens_list = dict_tokens['start_period4'] * token_vector.shape[0]
        else:
            tokens_list = dict_tokens['start_overtime'] * token_vector.shape[0]

        random_play, prob_random_play = predict_model_timediff_tower(model = token_model, token_vector = token_vector, 
                                                                     numerical_vector = numerical_vector_,
                                                                     time_diff_features = time_diff_vector_,
                                                                     only_consider = tokens_list)
        
        # reset de relógio/faltas (igual ao original, mesmo que redundante)
        period_duration = 7200 if period <= 4 else 3000
        allowed_fouls_home, allowed_fouls_away = define_game_fouls(period)

        game_status.period[valid_simulations] = period
        game_status.limhteam[valid_simulations] = allowed_fouls_home
        game_status.limvteam[valid_simulations] = allowed_fouls_away
        game_status.remaining_time[valid_simulations] = period_duration

        if period >= 5:
            game_status.ball_possession[valid_simulations] = 0
            game_status.limhtimeout[valid_simulations] = 2
            game_status.limvtimeout[valid_simulations] = 2

        if period == 4:
            num_timeouts_home = np.minimum(game_status.limhtimeout[valid_simulations], 4)
            num_timeouts_away = np.minimum(game_status.limvtimeout[valid_simulations], 4)

            game_status.limhtimeout[valid_simulations] = num_timeouts_home
            game_status.limvtimeout[valid_simulations] = num_timeouts_away


    # a partir dos valores sorteados, atualiza o game_status
    update_game_status(game_status = game_status, random_play = random_play, period = period, 
                       indices = valid_simulations, inplace = True, config = config)

    game_status.num_plays[valid_simulations] += 1
    game_status.sorted_token[valid_simulations] = np.asarray(random_play)
    game_status.prob_token[valid_simulations] = np.asarray(prob_random_play)
    
    # sempre a primeira jogada do primeiro periodo "gasta" tempo zero
    game_status.sorted_time[valid_simulations] = 0
    game_status.prob_time[valid_simulations] = 1

    return

def random_first_play_multi_tower(period, game_status, valid_simulations, token_model, config = 3, match_vector = None):

    dict_tokens = define_special_tokens(config = 1)

    token_vector = game_status.token_vector[valid_simulations]
    numerical_vector = game_status.numerical_vector[valid_simulations]

    if config == 3:
        num_numerical_variables = 15
    # match vector (igual ao original)
    if match_vector is None:
        numerical_vector_ = numerical_vector[:, :num_numerical_variables]
    else:
        numerical_vector_ = np.hstack([numerical_vector[:, :num_numerical_variables], match_vector])

    time_diff_vector_ = numerical_vector[:, num_numerical_variables:]

    if period == 1:
        # primeira jogada do jogo: restrição start_period1
        random_play, prob_random_play = predict_model_multi_tower(model = token_model, token_vector = token_vector, 
                                                                  numerical_vector = [numerical_vector_, time_diff_vector_],
                                                                  only_consider = dict_tokens['start_period1'])

        # posse inicial (igual ao original)
        game_status.ball_possession[valid_simulations] = define_ball_possession_vectorized(
            token = random_play, last_token = 0, period = period, ball_possession = 0)

    else:
        # início de outros períodos: restrição por período (igual ao original)
        if period == 2:
            tokens_list = dict_tokens['start_period2'] * token_vector.shape[0]
        elif period == 3:
            tokens_list = dict_tokens['start_period3'] * token_vector.shape[0]
        elif period == 4:
            tokens_list = dict_tokens['start_period4'] * token_vector.shape[0]
        else:
            tokens_list = dict_tokens['start_overtime'] * token_vector.shape[0]

        random_play, prob_random_play = predict_model_multi_tower(model = token_model, token_vector = token_vector, 
                                                                  numerical_vector = [numerical_vector_, time_diff_vector_],
                                                                  only_consider = tokens_list)
        
        # reset de relógio/faltas (igual ao original, mesmo que redundante)
        period_duration = 7200 if period <= 4 else 3000
        allowed_fouls_home, allowed_fouls_away = define_game_fouls(period)

        game_status.period[valid_simulations] = period
        game_status.limhteam[valid_simulations] = allowed_fouls_home
        game_status.limvteam[valid_simulations] = allowed_fouls_away
        game_status.remaining_time[valid_simulations] = period_duration

        if period >= 5:
            game_status.ball_possession[valid_simulations] = 0
            game_status.limhtimeout[valid_simulations] = 2
            game_status.limvtimeout[valid_simulations] = 2

        if period == 4:
            num_timeouts_home = np.minimum(game_status.limhtimeout[valid_simulations], 4)
            num_timeouts_away = np.minimum(game_status.limvtimeout[valid_simulations], 4)

            game_status.limhtimeout[valid_simulations] = num_timeouts_home
            game_status.limvtimeout[valid_simulations] = num_timeouts_away

    # a partir dos valores sorteados, atualiza o game_status
    update_game_status(game_status = game_status, random_play = random_play, period = period, 
                       indices = valid_simulations, inplace = True, config = config)

    game_status.num_plays[valid_simulations] += 1
    game_status.sorted_token[valid_simulations] = np.asarray(random_play)
    game_status.prob_token[valid_simulations] = np.asarray(prob_random_play)
    
    # sempre a primeira jogada do primeiro periodo "gasta" tempo zero
    game_status.sorted_time[valid_simulations] = 0
    game_status.prob_time[valid_simulations] = 1

    return


In [ ]:
def random_play_loop(period, game_status, valid_simulations, token_model, time_model, config = 1, match_vector = None):
    dict_tokens = define_special_tokens(config = 1)
    # tokens que encerram o periodo
    end_period_tokens = dict_tokens['end_period']    
    # tokens especiais que fazem com que possamos sortear jogadas com o tempo restante = 0
    special_tokens = dict_tokens['special_plays_end']    
    shot_blocked_tokens = dict_tokens['shot_blocked_end']

    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector, match_vector])

    # Armazenando os valores atuais de remaining_time
    list_remaining_time = game_status.remaining_time.copy()

    # Iniciando os vetores numpy finais que irao armazenar o token e o tempo sorteado, independente de posicao
    random_token_np = np.zeros(shape = game_status.n_sims, dtype = int)
    prob_token_np = np.zeros(shape = game_status.n_sims)
    random_time_np = np.full(shape = game_status.n_sims, fill_value = -1, dtype = int)

    # FASE 1: SORTEIO DO TOKEN DE JOGADA
    # somente para as simulacoes validas, geramos o token
    random_token, prob_random_token = predict_model(model = token_model, 
                                                    token_vector = game_status.token_vector[valid_simulations], 
                                                    numerical_vector = numerical_vector_[valid_simulations])
    
    # salvando o token sorteado no vetor final
    random_token_np[valid_simulations] = random_token
    prob_token_np[valid_simulations] = prob_random_token

    # CORRECAO 1: token de encerramento sorteado porem ainda falta muito tempo a jogar.
    # verificando as simulacoes onde sera preciso sortear outro token, ja que nessas simulacoes 
    # ainda falta muito tempo a jogar, mas um dos tokens de encerramento foi sorteado para ser 
    # a proxima jogada. O criterio atual define que se falta mais de 15 segundos a jogar, o token 
    # de encerramento sorteado precisa ser substituido.

    mask_invalid_play = np.isin(random_token_np, end_period_tokens) & (list_remaining_time > 150)
    invalid_play_idx = np.where(mask_invalid_play)[0]

    while len(invalid_play_idx) > 0:
        # somente para as simulacoes que tiveram o token invalidado pelo contexto, sortearemos um novo token
        temp_random_token, temp_prob_random_token = predict_model(model = token_model, 
                                                                  token_vector = game_status.token_vector[invalid_play_idx],
                                                                  numerical_vector = numerical_vector_[invalid_play_idx])
        
        # atualizando o vetor final com o novo vetor de tokens
        replacement_dict = {idx: (temp_random_token[pos], temp_prob_random_token[pos]) for pos, idx in enumerate(invalid_play_idx)}

        for idx, new_play in replacement_dict.items():
            random_token_np[idx] = new_play[0]
            prob_token_np[idx] = new_play[1]

        # atualizando a verificacao de tokens novamente: ainda existe token de encerramento sorteado cedo demais?
        mask_invalid_play = np.isin(random_token_np, end_period_tokens) & (list_remaining_time > 150)
        invalid_play_idx = np.where(mask_invalid_play)[0]

    # # CORRECAO 2: token de encerramento sortedo porem ele se refere a outro periodo.
    # # por causa do embedding, os tokens que fazem acoes parecidas ficam com pesos parecidos e dai 
    # # pode acontecer de estarmos gerando jogada para um periodo e o token de encerramento sorteado 
    # # se referir a outro periodo. Dai forcamos que o token sorteadose refira ao periodo em questao.
    # mask_fix_last_token = np.isin(random_token_np, end_period_tokens) 
    # invalid_last_token_idx = np.where(mask_fix_last_token)[0]
    # replacement_dict = {idx: fix_last_token(token = random_token_np[idx], period = period) for idx in invalid_last_token_idx}
    # for idx, new_play in replacement_dict.items():
    #     random_token_np[idx] = new_play

    # FASE 2: SORTEIO DO TOKEN DE TEMPO GASTO.
    # com tudo corrigido para o sorteio do token, vamos ao sorteio do tempo gasto pela jogada sorteada na fase 1
    # O primeiro passo sera definir para quais simulacoes precisaremos gerar token de tempo gasto:
    # se um dos tokens de encerramento foi sorteado na fase 1, entao ele ocupa todo o tempo restante a ser jogado e 
    # dai nao precisa sortear tempo gasto para esses casos.

    mask_remaining_time_zero = np.isin(random_token_np, end_period_tokens)
    idx_zero_remaining_time = np.where(mask_remaining_time_zero)[0]
    idx_zero_remaining_time = idx_zero_remaining_time[np.isin(idx_zero_remaining_time, valid_simulations)]

    # nessas simulacoes aqui precisaremos sortear o tempo gasto
    idx_random_remaining_time = np.where(~mask_remaining_time_zero)[0]
    idx_random_remaining_time = idx_random_remaining_time[np.isin(idx_random_remaining_time, valid_simulations)]

    # ultimo passo antes de sortear o tempo: atualizar os vetores com o impacto que o token de jogada sorteado
    # causou na partida. Dai se necessario, atualiza pontos e numero de faltas. Se usa vetores temporarios aqui
    # porque a jogada sorteada nessa iteracao ainda pode ser descartada se nao houver tempo restante para coloca-la
    # no periodo. Dai nao colocamos ela no status do jogo, mas precisamos dela para gerar o tempo gasto. 
    temp_token_vector, temp_numerical_vector = update_vectors(
        period = period,
        random_play = random_token_np, 
        tokens_vector = game_status.token_vector.copy(), 
        home_points = game_status.home_points.copy(), 
        away_points = game_status.away_points.copy(), 
        limhteam = game_status.limhteam.copy(),
        limvteam = game_status.limvteam.copy(),
        limhtimeout = game_status.limhtimeout.copy(),
        limvtimeout = game_status.limvtimeout.copy(),
        remaining_time = game_status.remaining_time.copy(), 
        ball_possession = game_status.ball_possession.copy(),
        indices = idx_random_remaining_time,
        config = config)
    
    if match_vector is not None:
        temp_numerical_vector = np.hstack([temp_numerical_vector, match_vector[idx_random_remaining_time]])

    # agora:
    # temp_token_vector esta atualizado com a jogada sorteada na fase 1.
    # temp_numerical_vector esta atualizado com o impacto que a jogada sorteada na fase 1 causou no jogo, 
    # no entanto, o tempo restante nao considera essa jogada (pois depende diretamente do que vamos sortear agora)
    random_time = predict_model(model = time_model, 
                                token_vector = temp_token_vector, 
                                numerical_vector = temp_numerical_vector, 
                                type_ = 'time', extra_info = False)
    
    # # CORRECAO 3: o tempo sorteado eh maior que o tempo restante de jogo
    # # Quando isso acontece, existe duas opcoes do que fazer:
    # # 1. forcar o sorteio de um token de tempo que se enquadre no contexto do jogo, ou seja, sortear random_time 
    # #    ate que ele nao seja mais maior que o tempo restante de jogo.
    # # 2. descartar a jogada sorteada, ja que ela nao se enquadra com o contexto do jogo. 
    # # O que vamos fazer sera: vamos tentar por 3 vezes sortear um tempo valido para aqueles que tiveram problema,
    # # se nao resolver, analisaremos num momento futuro se descartaremos a jogada sorteada ou nao (dependera da jogada).
    # invalid_time_mask = np.asarray(random_time) > list_remaining_time[idx_random_remaining_time]
    # idx_invalid_time = np.where(invalid_time_mask)[0]

    # invalid_time_flag = 0
    # while len(idx_invalid_time) > 0 and invalid_time_flag < 3:
    #     # somente para as simulacoes que tiveram tempo sorteado maior que tempo restante, sortearemos um novo tempo gasto
    #     temp_random_time = predict_model(model = time_model, 
    #                                      token_vector = temp_token_vector[idx_invalid_time],
    #                                      numerical_vector = temp_numerical_vector[idx_invalid_time], 
    #                                      type_= 'time', extra_info = False)    

    #     replacement_dict = {idx: temp_random_time[pos] for pos, idx in enumerate(idx_invalid_time)}
    #     # Atualizar random_time somente nesses índices
    #     for idx, new_play in replacement_dict.items():
    #         random_time[idx] = new_play

    #     # atualizando a verificacao de tempo gasto novamente: ainda existe tempo gasto sorteado que ultrapassa o tempo restante?
    #     invalid_time_mask = np.asarray(random_time) > list_remaining_time[idx_random_remaining_time]
    #     idx_invalid_time = np.where(invalid_time_mask)[0]
    #     invalid_time_flag += 1

    # atualizando o vetor final de tempo gasto sorteado. Nas simulacoes onde um token de jogada nao foi um dos 
    # tokens de encerramento, colocamos o tempo gasto sorteado (lembrando que pode haver tempo sorteado amior 
    # que tempo restante de jogo). Nas simulacoes onde um token de jogada foi um dos tokens de encerramento, 
    # entao o tempo gasto foi o tempo restante de jogo.
    random_time_np[idx_random_remaining_time] = random_time
    random_time_np[idx_zero_remaining_time] = list_remaining_time[idx_zero_remaining_time]

    # CORRECAO 4: tempo gasto sorteado contem decimos de segundos, mas nao eh necessario.
    # Na NBA a contagem de tempo detalha apenas os minutos e segundos (mm:ss) nos 11 primeiros minutos iniciais do periodo, 
    # somente no minuto final que se detalha alem dos minutos e segundos, tambem os decimos de segundos (mm:ss:ss.f).
    # Se por acaso foi sorteado um token de tempo que envolva decimos de segundos, mas nao estamos no minuto final 
    # da partida, entao arredondamos esse tempo gasto para eliminar os decimos de segundos.

    # mask_fix_remaining_time = (list_remaining_time - random_time_np) > 600
    # idx_fix_remaining_time = np.where(mask_fix_remaining_time)[0]
    # idx_fix_remaining_time = idx_fix_remaining_time[np.isin(idx_fix_remaining_time, valid_simulations)]
    # rounded_time = (random_time_np  // 10) * 10
    # random_time_np[idx_fix_remaining_time] = rounded_time[idx_fix_remaining_time]

    # ATUALIZANDO O GAME STATUS
    # com o token de jogada e o token de tempo gasto sorteados, vamos atualizar o game status, mas antes precisamos
    # resolver o problema de tempo gasto invalido. Isso impacta diretamente se vamos continuar gerando jogadas para 
    # aquela simulacao ou nao. Dai, vamos assinalar isso no vetor keep_generating do game status.

    # keep_generating = 2 : significa que antes de sortar a jogada atual estavamos no fluxo normal de geracao de jogadas e 
    # entao um token de encerramento de periodo foi gerado. Dai assinalamos o keep_generating dessas simulacoes igual a 2.
    mask_kp = (game_status.keep_generating == 1) & np.isin(random_token_np, end_period_tokens)
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 2

    # keep_generating = 3: significa que antes de sortear a jogada atual estavamos no fluxo normal de geracao de jogadas e
    # entao um token especial foi sorteado com tempo invalido. Dai assinalamos o keep_generating dessas simulacoes igual a 3.
    # O que sera feito eh: corrigir o tempo gasto por essa jogada, adiciona-la no game_status e dai encerrar a geracao de jogadas
    # para essas simulacoes ate que venha a jogada de encerramento.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time) & (np.isin(random_token_np, special_tokens))
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 3

    # keep_generating = 5: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token especial que exige complemento
    # foi sorteado com tempo invalido, essas simulacoes foram assinaladas com keep_generating = 4 e foi permitido que elas gerem esse 
    # complemento antes de encerrar a geracao de jogadas para essas simulacoes. 
    mask_kp = (game_status.keep_generating == 4)
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 5

    # keep_generating = 4: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token especial que exige complemento foi sorteado
    # com tempo invalido, assinalamos essas simulacoes com keep_generating = 4, corrigimos seu tempo gasto sorteado e vamos permitir que gere o 
    # complemento na proxima jogada, mas depois dai so a geracao de jogadas que encerrem o periodo serao permitidas.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time) & (np.isin(random_token_np, shot_blocked_tokens))
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 4

    # keep_generating = 6: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token foi sorteado com tempo gasto invalido,
    # assinalamos essas simulacoes com keep_generating = 6 e nao permitimos que essa jogada seja colocada no game_status.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time)
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 6

    # trocando o tempo gasto sorteado para as simulacoes com keep_generating iguais 3, 4, 5, para valores validos, que deixam o tempo restante igual a 0.
    mask_change_sorted_time = np.isin(game_status.keep_generating, [3, 4, 5])
    idx_change_sorted_time = np.where(mask_change_sorted_time)[0]

    random_time_np[idx_change_sorted_time] = list_remaining_time[idx_change_sorted_time]

    idx_update_game_status = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    # atualizando o game status para aqueles casos que o token de jogada sorteado e o tempo gasto sorteado foram considerados validos.
    update_game_status(game_status = game_status, random_play = random_token_np[idx_update_game_status], 
                       period = period, indices = idx_update_game_status, inplace = True, config = config)

    game_status.num_plays[idx_update_game_status] += 1
    game_status.sorted_token[idx_update_game_status] = random_token_np[idx_update_game_status]
    game_status.prob_token[idx_update_game_status] = prob_token_np[idx_update_game_status]
    game_status.sorted_time[idx_update_game_status] = random_time_np[idx_update_game_status]

    new_remaining_time = list_remaining_time[idx_update_game_status] - random_time_np[idx_update_game_status]
    game_status.remaining_time[idx_update_game_status] = new_remaining_time

    return

def random_play_loop_st(period, game_status, valid_simulations, token_model, time_model, config = 1, match_vector = None):

    dict_tokens = define_special_tokens(config = 1)
    # tokens que encerram o periodo    
    end_period_tokens = dict_tokens['end_period']
    # tokens especiais que fazem com que possamos sortear jogadas com o tempo restante = 0    
    special_tokens = dict_tokens['special_plays_end']
    shot_blocked_tokens = dict_tokens['shot_blocked_end']

    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector, match_vector])

    list_remaining_time = game_status.remaining_time.copy()

    random_token_np = np.zeros(shape = game_status.n_sims, dtype = int)
    prob_token_np = np.zeros(shape = game_status.n_sims)
    random_time_np = np.full(shape = game_status.n_sims, fill_value= -1, dtype = int)

    # =========================================================
    # FASE 1: SORTEIO DO TOKEN (play_i)  [usa S_{i-1}]
    # =========================================================
    # somente para as simulacoes validas, geramos o token
 
    state_id_token = categorize_time_bins_np(
        period_arr = np.full(game_status.remaining_time.shape[0], period, dtype = int),
        remaining_time_arr = game_status.remaining_time.astype(int),
        diff_points_arr = (game_status.home_points - game_status.away_points).astype(int)
    )

    random_token, prob_random_token = predict_model_st(model = token_model,
                                                       token_vector = game_status.token_vector[valid_simulations],
                                                       numerical_vector = numerical_vector_[valid_simulations],
                                                       state_token_vector = state_id_token[valid_simulations])
    random_token_np[valid_simulations] = random_token
    prob_token_np[valid_simulations]   = prob_random_token

    # -------------------------
    # CORRECAO 1 (token fim cedo): token de encerramento sorteado porem ainda falta muito tempo a jogar.
    # verificando as simulacoes onde sera preciso sortear outro token, ja que nessas simulacoes 
    # ainda falta muito tempo a jogar, mas um dos tokens de encerramento foi sorteado para ser 
    # a proxima jogada. O criterio atual define que se falta mais de 15 segundos a jogar, o token 
    # de encerramento sorteado precisa ser substituido.
    # -------------------------
    mask_invalid_play = np.isin(random_token_np, end_period_tokens) & (list_remaining_time > 150)
    invalid_play_idx = np.where(mask_invalid_play)[0]

    while len(invalid_play_idx) > 0:
        # somente para as simulacoes que tiveram o token invalidado pelo contexto, sortearemos um novo token
        temp_random_token, temp_prob_random_token = predict_model_st(model = token_model,
                                                                     token_vector = game_status.token_vector[invalid_play_idx],
                                                                     numerical_vector = numerical_vector_[invalid_play_idx],
                                                                     state_token_vector = state_id_token[invalid_play_idx])


        # atualizando o vetor final com o novo vetor de tokens
        replacement_dict = {i: (temp_random_token[pos], temp_prob_random_token[pos]) for pos, i in enumerate(invalid_play_idx)}

        for idx, new_play in replacement_dict.items():
            random_token_np[idx] = new_play[0]
            prob_token_np[idx] = new_play[1]

        # atualizando a verificacao de tokens novamente: ainda existe token de encerramento sorteado cedo demais?
        mask_invalid_play = np.isin(random_token_np, end_period_tokens) & (list_remaining_time > 150)
        invalid_play_idx = np.where(mask_invalid_play)[0]

    # # -------------------------
    # # CORRECAO 2 (token fim do período errado): token de encerramento sortedo porem ele se refere a outro periodo.
    # # por causa do embedding, os tokens que fazem acoes parecidas ficam com pesos parecidos e dai 
    # # pode acontecer de estarmos gerando jogada para um periodo e o token de encerramento sorteado 
    # # se referir a outro periodo. Dai forcamos que o token sorteadose refira ao periodo em questao.
    # # -------------------------
    # mask_fix_last_token = np.isin(random_token_np, end_period_tokens)
    # invalid_last_token_idx = np.where(mask_fix_last_token)[0]
    # replacement_dict = {idx: fix_last_token(token = random_token_np[idx], period = period) for idx in invalid_last_token_idx}
    # for idx, new_play in replacement_dict.items():
    #     random_token_np[idx] = new_play

    # =========================================================
    # FASE 2: SORTEIO DO TEMPO (time_i)  [usa play_i e time_{i-1}]:     
    # com tudo corrigido para o sorteio do token, vamos ao sorteio do tempo gasto pela jogada sorteada na fase 1
    # O primeiro passo sera definir para quais simulacoes precisaremos gerar token de tempo gasto:
    # se um dos tokens de encerramento foi sorteado na fase 1, entao ele ocupa todo o tempo restante a ser jogado e 
    # dai nao precisa sortear tempo gasto para esses casos.
    # =========================================================
    mask_remaining_time_zero = np.isin(random_token_np, end_period_tokens)
    idx_zero_remaining_time = np.where(mask_remaining_time_zero)[0]
    idx_zero_remaining_time = idx_zero_remaining_time[np.isin(idx_zero_remaining_time, valid_simulations)]

    idx_random_remaining_time = np.where(~mask_remaining_time_zero)[0]
    idx_random_remaining_time = idx_random_remaining_time[np.isin(idx_random_remaining_time, valid_simulations)]

    # --- cria vetores temporários pós-play_i (sem aplicar no game_status real)
    # Aqui é o ponto que muda: precisamos também de hp/ap temporários para o temp_state_id_token.
    # Para NÃO mexer no seu update_vectors, fazemos uma cópia local e reaplicamos a lógica de pontos
    # só para diff. Isso é feio, mas é o menor impacto no resto do simulador.
    # Melhor manutenção: criar um update_vectors_st que retorna hp/ap (mas você pediu mínimo).

    temp_token_vector, temp_numerical_vector = update_vectors(
        period = period,
        random_play = random_token_np,
        tokens_vector = game_status.token_vector.copy(),
        home_points = game_status.home_points.copy(),
        away_points = game_status.away_points.copy(),
        limhteam = game_status.limhteam.copy(),
        limvteam = game_status.limvteam.copy(),
        limhtimeout = game_status.limhtimeout.copy(),
        limvtimeout = game_status.limvtimeout.copy(),
        remaining_time = game_status.remaining_time.copy(),
        ball_possession = game_status.ball_possession.copy(),
        indices = idx_random_remaining_time,
        config = config)

    if match_vector is not None:
        temp_numerical_vector = np.hstack([temp_numerical_vector, match_vector[idx_random_remaining_time]])

    # --- calcula diff pós-play_i (temporário) para temp_state_id_tokene
    # (repete mínima parte da lógica de pontos do update_vectors para não alterar suas funções antigas)
    idxr = np.asarray(idx_random_remaining_time)
    rp_sub = random_token_np[idxr]

    hp_tmp = game_status.home_points[idxr].copy()
    ap_tmp = game_status.away_points[idxr].copy()

    hp_tmp[np.isin(rp_sub, dict_tokens['home_pts1'])] += 1
    hp_tmp[np.isin(rp_sub, dict_tokens['home_pts2'])] += 2
    hp_tmp[np.isin(rp_sub, dict_tokens['home_pts3'])] += 3

    ap_tmp[np.isin(rp_sub, dict_tokens['away_pts1'])] += 1
    ap_tmp[np.isin(rp_sub, dict_tokens['away_pts2'])] += 2
    ap_tmp[np.isin(rp_sub, dict_tokens['away_pts3'])] += 3

    temp_state_id_token = categorize_time_bins_np(
        period_arr = np.full(idxr.shape[0], period, dtype = int),
        remaining_time_arr = game_status.remaining_time[idxr].astype(int),
        diff_points_arr = (hp_tmp - ap_tmp).astype(int)
    )

    # agora:
    # temp_token_vector esta atualizado com a jogada sorteada na fase 1.
    # temp_numerical_vector esta atualizado com o impacto que a jogada sorteada na fase 1 causou no jogo, 
    # no entanto, o tempo restante nao considera essa jogada (pois depende diretamente do que vamos sortear agora)
    random_time = predict_model_st(model = time_model, 
                                   token_vector = temp_token_vector, 
                                   numerical_vector = temp_numerical_vector,
                                   state_token_vector = temp_state_id_token,
                                   type_ = 'time', extra_info = False)    

    # # -------------------------
    # # CORRECAO 3 (tempo > restante):
    # # o tempo sorteado eh maior que o tempo restante de jogo
    # # Quando isso acontece, existe duas opcoes do que fazer:
    # # 1. forcar o sorteio de um token de tempo que se enquadre no contexto do jogo, ou seja, sortear random_time 
    # #    ate que ele nao seja mais maior que o tempo restante de jogo.
    # # 2. descartar a jogada sorteada, ja que ela nao se enquadra com o contexto do jogo. 
    # # O que vamos fazer sera: vamos tentar por 3 vezes sortear um tempo valido para aqueles que tiveram problema,
    # # se nao resolver, analisaremos num momento futuro se descartaremos a jogada sorteada ou nao (dependera da jogada).
    # # -------------------------
    # invalid_time_mask = np.asarray(random_time) > list_remaining_time[idx_random_remaining_time]
    # idx_invalid_time = np.where(invalid_time_mask)[0]

    # invalid_time_flag = 0
    # while len(idx_invalid_time) > 0 and invalid_time_flag < 3:

    #     # somente para as simulacoes que tiveram tempo sorteado maior que tempo restante, sortearemos um novo tempo gasto
    #     temp_random_time = predict_model_st(model = time_model,
    #         token_vector = temp_token_vector[idx_invalid_time],
    #         numerical_vector = temp_numerical_vector[idx_invalid_time],
    #         state_token_vector = temp_state_id_token[idx_invalid_time],
    #         type_ = 'time', extra_info = False)

    #     replacement_dict = {i: temp_random_time[pos] for pos, i in enumerate(idx_invalid_time)}
    #     # Atualizar random_time somente nesses índices        
    #     for idx, new_play in replacement_dict.items():
    #         random_time[idx] = new_play

    #     # atualizando a verificacao de tempo gasto novamente: ainda existe tempo gasto sorteado que ultrapassa o tempo restante?
    #     invalid_time_mask = np.asarray(random_time) > list_remaining_time[idx_random_remaining_time]
    #     idx_invalid_time = np.where(invalid_time_mask)[0]
    #     invalid_time_flag += 1

    # atualizando o vetor final de tempo gasto sorteado. Nas simulacoes onde um token de jogada nao foi um dos 
    # tokens de encerramento, colocamos o tempo gasto sorteado (lembrando que pode haver tempo sorteado amior 
    # que tempo restante de jogo). Nas simulacoes onde um token de jogada foi um dos tokens de encerramento, 
    # entao o tempo gasto foi o tempo restante de jogo.
    random_time_np[idx_random_remaining_time] = random_time
    random_time_np[idx_zero_remaining_time] = list_remaining_time[idx_zero_remaining_time]

    # CORRECAO 4: tempo gasto sorteado contem decimos de segundos, mas nao eh necessario.
    # Na NBA a contagem de tempo detalha apenas os minutos e segundos (mm:ss) nos 11 primeiros minutos iniciais do periodo, 
    # somente no minuto final que se detalha alem dos minutos e segundos, tambem os decimos de segundos (mm:ss:ss.f).
    # Se por acaso foi sorteado um token de tempo que envolva decimos de segundos, mas nao estamos no minuto final 
    # da partida, entao arredondamos esse tempo gasto para eliminar os decimos de segundos.

    # mask_fix_remaining_time = (list_remaining_time - random_time_np) > 600
    # idx_fix_remaining_time = np.where(mask_fix_remaining_time)[0]
    # idx_fix_remaining_time = idx_fix_remaining_time[np.isin(idx_fix_remaining_time, valid_simulations)]
    # rounded_time = (random_time_np  // 10) * 10
    # random_time_np[idx_fix_remaining_time] = rounded_time[idx_fix_remaining_time]

    # =========================================================
    # ATUALIZANDO GAME STATUS (IGUAL AO SEU)
    # =========================================================
    # com o token de jogada e o token de tempo gasto sorteados, vamos atualizar o game status, mas antes precisamos
    # resolver o problema de tempo gasto invalido. Isso impacta diretamente se vamos continuar gerando jogadas para 
    # aquela simulacao ou nao. Dai, vamos assinalar isso no vetor keep_generating do game status.    

    # keep_generating = 2 : significa que antes de sortar a jogada atual estavamos no fluxo normal de geracao de jogadas e 
    # entao um token de encerramento de periodo foi gerado. Dai assinalamos o keep_generating dessas simulacoes igual a 2.    
    mask_kp = (game_status.keep_generating == 1) & np.isin(random_token_np, end_period_tokens)
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 2

    # keep_generating = 3: significa que antes de sortear a jogada atual estavamos no fluxo normal de geracao de jogadas e
    # entao um token especial foi sorteado com tempo invalido. Dai assinalamos o keep_generating dessas simulacoes igual a 3.
    # O que sera feito eh: corrigir o tempo gasto por essa jogada, adiciona-la no game_status e dai encerrar a geracao de jogadas
    # para essas simulacoes ate que venha a jogada de encerramento.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time) & (np.isin(random_token_np, special_tokens))
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 3

    # keep_generating = 5: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token especial que exige complemento
    # foi sorteado com tempo invalido, essas simulacoes foram assinaladas com keep_generating = 4 e foi permitido que elas gerem esse 
    # complemento antes de encerrar a geracao de jogadas para essas simulacoes. 
    mask_kp = (game_status.keep_generating == 4)
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 5

    # keep_generating = 4: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token especial que exige complemento foi sorteado
    # com tempo invalido, assinalamos essas simulacoes com keep_generating = 4, corrigimos seu tempo gasto sorteado e vamos permitir que gere o 
    # complemento na proxima jogada, mas depois dai so a geracao de jogadas que encerrem o periodo serao permitidas.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time) & (np.isin(random_token_np, shot_blocked_tokens))
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 4

    # keep_generating = 6: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token foi sorteado com tempo gasto invalido,
    # assinalamos essas simulacoes com keep_generating = 6 e nao permitimos que essa jogada seja colocada no game_status.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time)
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 6

    # trocando o tempo gasto sorteado para as simulacoes com keep_generating iguais 3, 4, 5, para valores validos, que deixam o tempo restante igual a 0.
    mask_change_sorted_time = np.isin(game_status.keep_generating, [3, 4, 5])
    idx_change_sorted_time = np.where(mask_change_sorted_time)[0]

    random_time_np[idx_change_sorted_time] = list_remaining_time[idx_change_sorted_time]

    idx_update_game_status = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    # atualizando o game status para aqueles casos que o token de jogada sorteado e o tempo gasto sorteado foram considerados validos.
    update_game_status(game_status = game_status, random_play = random_token_np[idx_update_game_status], 
                       period = period, indices = idx_update_game_status, inplace = True, config = config)
    
    game_status.num_plays[idx_update_game_status] += 1
    game_status.sorted_token[idx_update_game_status] = random_token_np[idx_update_game_status]
    game_status.prob_token[idx_update_game_status] = prob_token_np[idx_update_game_status]
    game_status.sorted_time[idx_update_game_status] = random_time_np[idx_update_game_status]

    new_remaining_time = list_remaining_time[idx_update_game_status] - random_time_np[idx_update_game_status]
    game_status.remaining_time[idx_update_game_status] = new_remaining_time

    return

def random_play_loop_timediff_tower(period, game_status, valid_simulations, token_model, time_model, config = 3, match_vector = None):

    dict_tokens = define_special_tokens(config = 1)
    # tokens que encerram o periodo    
    end_period_tokens = dict_tokens['end_period']
    # tokens especiais que fazem com que possamos sortear jogadas com o tempo restante = 0    
    special_tokens = dict_tokens['special_plays_end']
    shot_blocked_tokens = dict_tokens['shot_blocked_end']

    if config == 3:
        num_numerical_variables = 15
    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector[:, :num_numerical_variables]
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector[:, :num_numerical_variables], match_vector])

    time_diff_vector_ = game_status.numerical_vector[:, num_numerical_variables:]

    list_remaining_time = game_status.remaining_time.copy()

    random_token_np = np.zeros(shape = game_status.n_sims, dtype = int)
    prob_token_np = np.zeros(shape = game_status.n_sims)
    random_time_np = np.full(shape = game_status.n_sims, fill_value= -1, dtype = int)

    # =========================================================
    # FASE 1: SORTEIO DO TOKEN (play_i)  [usa S_{i-1}]
    # =========================================================
    # somente para as simulacoes validas, geramos o token
 
   
    random_token, prob_random_token = predict_model_timediff_tower(model = token_model, 
                                                                   token_vector = game_status.token_vector[valid_simulations],
                                                                   numerical_vector = numerical_vector_[valid_simulations],
                                                                   time_diff_features = time_diff_vector_[valid_simulations])
    random_token_np[valid_simulations] = random_token
    prob_token_np[valid_simulations]   = prob_random_token

    # -------------------------
    # CORRECAO 1 (token fim cedo): token de encerramento sorteado porem ainda falta muito tempo a jogar.
    # verificando as simulacoes onde sera preciso sortear outro token, ja que nessas simulacoes 
    # ainda falta muito tempo a jogar, mas um dos tokens de encerramento foi sorteado para ser 
    # a proxima jogada. O criterio atual define que se falta mais de 15 segundos a jogar, o token 
    # de encerramento sorteado precisa ser substituido.
    # -------------------------
    mask_invalid_play = np.isin(random_token_np, end_period_tokens) & (list_remaining_time > 150)
    invalid_play_idx = np.where(mask_invalid_play)[0]

    while len(invalid_play_idx) > 0:
        # somente para as simulacoes que tiveram o token invalidado pelo contexto, sortearemos um novo token
        temp_random_token, temp_prob_random_token = predict_model_timediff_tower(model = token_model,
                                                                                 token_vector = game_status.token_vector[invalid_play_idx],
                                                                                 numerical_vector = numerical_vector_[invalid_play_idx],
                                                                                 time_diff_features = time_diff_vector_[invalid_play_idx])


        # atualizando o vetor final com o novo vetor de tokens
        replacement_dict = {i: (temp_random_token[pos], temp_prob_random_token[pos]) for pos, i in enumerate(invalid_play_idx)}

        for idx, new_play in replacement_dict.items():
            random_token_np[idx] = new_play[0]
            prob_token_np[idx] = new_play[1]

        # atualizando a verificacao de tokens novamente: ainda existe token de encerramento sorteado cedo demais?
        mask_invalid_play = np.isin(random_token_np, end_period_tokens) & (list_remaining_time > 150)
        invalid_play_idx = np.where(mask_invalid_play)[0]

    # # -------------------------
    # # CORRECAO 2 (token fim do período errado): token de encerramento sortedo porem ele se refere a outro periodo.
    # # por causa do embedding, os tokens que fazem acoes parecidas ficam com pesos parecidos e dai 
    # # pode acontecer de estarmos gerando jogada para um periodo e o token de encerramento sorteado 
    # # se referir a outro periodo. Dai forcamos que o token sorteadose refira ao periodo em questao.
    # # -------------------------
    # mask_fix_last_token = np.isin(random_token_np, end_period_tokens)
    # invalid_last_token_idx = np.where(mask_fix_last_token)[0]
    # replacement_dict = {idx: fix_last_token(token = random_token_np[idx], period = period) for idx in invalid_last_token_idx}
    # for idx, new_play in replacement_dict.items():
    #     random_token_np[idx] = new_play

    # =========================================================
    # FASE 2: SORTEIO DO TEMPO (time_i)  [usa play_i e time_{i-1}]:     
    # com tudo corrigido para o sorteio do token, vamos ao sorteio do tempo gasto pela jogada sorteada na fase 1
    # O primeiro passo sera definir para quais simulacoes precisaremos gerar token de tempo gasto:
    # se um dos tokens de encerramento foi sorteado na fase 1, entao ele ocupa todo o tempo restante a ser jogado e 
    # dai nao precisa sortear tempo gasto para esses casos.
    # =========================================================
    mask_remaining_time_zero = np.isin(random_token_np, end_period_tokens)
    idx_zero_remaining_time = np.where(mask_remaining_time_zero)[0]
    idx_zero_remaining_time = idx_zero_remaining_time[np.isin(idx_zero_remaining_time, valid_simulations)]

    idx_random_remaining_time = np.where(~mask_remaining_time_zero)[0]
    idx_random_remaining_time = idx_random_remaining_time[np.isin(idx_random_remaining_time, valid_simulations)]

    # --- cria vetores temporários pós-play_i (sem aplicar no game_status real)

    temp_token_vector, temp_numerical_vector = update_vectors(
        period = period,
        random_play = random_token_np,
        tokens_vector = game_status.token_vector.copy(),
        home_points = game_status.home_points.copy(),
        away_points = game_status.away_points.copy(),
        limhteam = game_status.limhteam.copy(),
        limvteam = game_status.limvteam.copy(),
        limhtimeout = game_status.limhtimeout.copy(),
        limvtimeout = game_status.limvtimeout.copy(),
        remaining_time = game_status.remaining_time.copy(),
        ball_possession = game_status.ball_possession.copy(),
        indices = idx_random_remaining_time,
        config = config)

    temp_time_diff_vector_ = temp_numerical_vector[:, num_numerical_variables:]

    if match_vector is not None:
        temp_numerical_vector = np.hstack([temp_numerical_vector[:, :num_numerical_variables], match_vector[idx_random_remaining_time]])
    else:
        temp_numerical_vector = temp_numerical_vector[:, :num_numerical_variables]

    # agora:
    # temp_token_vector esta atualizado com a jogada sorteada na fase 1.
    # temp_numerical_vector esta atualizado com o impacto que a jogada sorteada na fase 1 causou no jogo, 
    # no entanto, o tempo restante nao considera essa jogada (pois depende diretamente do que vamos sortear agora)
    random_time = predict_model_timediff_tower(model = time_model, token_vector = temp_token_vector, 
                                               numerical_vector = temp_numerical_vector,
                                               time_diff_features = temp_time_diff_vector_, 
                                               type_ = 'time', extra_info = False)    

    # # -------------------------
    # # CORRECAO 3 (tempo > restante):
    # # o tempo sorteado eh maior que o tempo restante de jogo
    # # Quando isso acontece, existe duas opcoes do que fazer:
    # # 1. forcar o sorteio de um token de tempo que se enquadre no contexto do jogo, ou seja, sortear random_time 
    # #    ate que ele nao seja mais maior que o tempo restante de jogo.
    # # 2. descartar a jogada sorteada, ja que ela nao se enquadra com o contexto do jogo. 
    # # O que vamos fazer sera: vamos tentar por 3 vezes sortear um tempo valido para aqueles que tiveram problema,
    # # se nao resolver, analisaremos num momento futuro se descartaremos a jogada sorteada ou nao (dependera da jogada).
    # # -------------------------
    # invalid_time_mask = np.asarray(random_time) > list_remaining_time[idx_random_remaining_time]
    # idx_invalid_time = np.where(invalid_time_mask)[0]

    # invalid_time_flag = 0
    # while len(idx_invalid_time) > 0 and invalid_time_flag < 3:

    #     # somente para as simulacoes que tiveram tempo sorteado maior que tempo restante, sortearemos um novo tempo gasto
    #     temp_random_time = predict_model_timediff_tower(model = time_model,
    #                                                     token_vector = temp_token_vector[idx_invalid_time],
    #                                                     numerical_vector = temp_numerical_vector[idx_invalid_time],
    #                                                     time_diff_features = temp_time_diff_vector_[idx_invalid_time], 
    #                                                     type_ = 'time', extra_info = False)

    #     replacement_dict = {i: temp_random_time[pos] for pos, i in enumerate(idx_invalid_time)}
    #     # Atualizar random_time somente nesses índices        
    #     for idx, new_play in replacement_dict.items():
    #         random_time[idx] = new_play

    #     # atualizando a verificacao de tempo gasto novamente: ainda existe tempo gasto sorteado que ultrapassa o tempo restante?
    #     invalid_time_mask = np.asarray(random_time) > list_remaining_time[idx_random_remaining_time]
    #     idx_invalid_time = np.where(invalid_time_mask)[0]
    #     invalid_time_flag += 1

    # atualizando o vetor final de tempo gasto sorteado. Nas simulacoes onde um token de jogada nao foi um dos 
    # tokens de encerramento, colocamos o tempo gasto sorteado (lembrando que pode haver tempo sorteado amior 
    # que tempo restante de jogo). Nas simulacoes onde um token de jogada foi um dos tokens de encerramento, 
    # entao o tempo gasto foi o tempo restante de jogo.
    random_time_np[idx_random_remaining_time] = random_time
    random_time_np[idx_zero_remaining_time] = list_remaining_time[idx_zero_remaining_time]

    # CORRECAO 4: tempo gasto sorteado contem decimos de segundos, mas nao eh necessario.
    # Na NBA a contagem de tempo detalha apenas os minutos e segundos (mm:ss) nos 11 primeiros minutos iniciais do periodo, 
    # somente no minuto final que se detalha alem dos minutos e segundos, tambem os decimos de segundos (mm:ss:ss.f).
    # Se por acaso foi sorteado um token de tempo que envolva decimos de segundos, mas nao estamos no minuto final 
    # da partida, entao arredondamos esse tempo gasto para eliminar os decimos de segundos.

    # mask_fix_remaining_time = (list_remaining_time - random_time_np) > 600
    # idx_fix_remaining_time = np.where(mask_fix_remaining_time)[0]
    # idx_fix_remaining_time = idx_fix_remaining_time[np.isin(idx_fix_remaining_time, valid_simulations)]
    # rounded_time = (random_time_np  // 10) * 10
    # random_time_np[idx_fix_remaining_time] = rounded_time[idx_fix_remaining_time]

    # =========================================================
    # ATUALIZANDO GAME STATUS (IGUAL AO SEU)
    # =========================================================
    # com o token de jogada e o token de tempo gasto sorteados, vamos atualizar o game status, mas antes precisamos
    # resolver o problema de tempo gasto invalido. Isso impacta diretamente se vamos continuar gerando jogadas para 
    # aquela simulacao ou nao. Dai, vamos assinalar isso no vetor keep_generating do game status.    

    # keep_generating = 2 : significa que antes de sortar a jogada atual estavamos no fluxo normal de geracao de jogadas e 
    # entao um token de encerramento de periodo foi gerado. Dai assinalamos o keep_generating dessas simulacoes igual a 2.    
    mask_kp = (game_status.keep_generating == 1) & np.isin(random_token_np, end_period_tokens)
    idx_kp = np.where(mask_kp)[0]

    game_status.keep_generating[idx_kp] = 2

    # keep_generating = 3: significa que antes de sortear a jogada atual estavamos no fluxo normal de geracao de jogadas e
    # entao um token especial foi sorteado com tempo invalido. Dai assinalamos o keep_generating dessas simulacoes igual a 3.
    # O que sera feito eh: corrigir o tempo gasto por essa jogada, adiciona-la no game_status e dai encerrar a geracao de jogadas
    # para essas simulacoes ate que venha a jogada de encerramento.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time) & (np.isin(random_token_np, special_tokens))
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 3

    # keep_generating = 5: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token especial que exige complemento
    # foi sorteado com tempo invalido, essas simulacoes foram assinaladas com keep_generating = 4 e foi permitido que elas gerem esse 
    # complemento antes de encerrar a geracao de jogadas para essas simulacoes. 
    mask_kp = (game_status.keep_generating == 4)
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 5

    # keep_generating = 4: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token especial que exige complemento foi sorteado
    # com tempo invalido, assinalamos essas simulacoes com keep_generating = 4, corrigimos seu tempo gasto sorteado e vamos permitir que gere o 
    # complemento na proxima jogada, mas depois dai so a geracao de jogadas que encerrem o periodo serao permitidas.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time) & (np.isin(random_token_np, shot_blocked_tokens))
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 4

    # keep_generating = 6: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token foi sorteado com tempo gasto invalido,
    # assinalamos essas simulacoes com keep_generating = 6 e nao permitimos que essa jogada seja colocada no game_status.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time)
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 6

    # trocando o tempo gasto sorteado para as simulacoes com keep_generating iguais 3, 4, 5, para valores validos, que deixam o tempo restante igual a 0.
    mask_change_sorted_time = np.isin(game_status.keep_generating, [3, 4, 5])
    idx_change_sorted_time = np.where(mask_change_sorted_time)[0]

    random_time_np[idx_change_sorted_time] = list_remaining_time[idx_change_sorted_time]

    idx_update_game_status = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    # atualizando o game status para aqueles casos que o token de jogada sorteado e o tempo gasto sorteado foram considerados validos.
    update_game_status(game_status = game_status, random_play = random_token_np[idx_update_game_status], 
                       period = period, indices = idx_update_game_status, inplace = True, config = config)
    
    game_status.num_plays[idx_update_game_status] += 1
    game_status.sorted_token[idx_update_game_status] = random_token_np[idx_update_game_status]
    game_status.prob_token[idx_update_game_status] = prob_token_np[idx_update_game_status]
    game_status.sorted_time[idx_update_game_status] = random_time_np[idx_update_game_status]

    new_remaining_time = list_remaining_time[idx_update_game_status] - random_time_np[idx_update_game_status]
    game_status.remaining_time[idx_update_game_status] = new_remaining_time

    return

def random_play_loop_multi_tower(period, game_status, valid_simulations, token_model, time_model, config = 3, match_vector = None):

    dict_tokens = define_special_tokens(config = 1)
    # tokens que encerram o periodo    
    end_period_tokens = dict_tokens['end_period']
    # tokens especiais que fazem com que possamos sortear jogadas com o tempo restante = 0    
    special_tokens = dict_tokens['special_plays_end']
    shot_blocked_tokens = dict_tokens['shot_blocked_end']

    if config == 3:
        num_numerical_variables = 15
    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector[:, :num_numerical_variables]
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector[:, :num_numerical_variables], match_vector])

    time_diff_vector_ = game_status.numerical_vector[:, num_numerical_variables:]

    list_remaining_time = game_status.remaining_time.copy()

    random_token_np = np.zeros(shape = game_status.n_sims, dtype = int)
    prob_token_np = np.zeros(shape = game_status.n_sims)
    random_time_np = np.full(shape = game_status.n_sims, fill_value= -1, dtype = int)

    # =========================================================
    # FASE 1: SORTEIO DO TOKEN (play_i)  [usa S_{i-1}]
    # =========================================================
    # somente para as simulacoes validas, geramos o token
 
   
    random_token, prob_random_token = predict_model_multi_tower(model = token_model, 
                                                                token_vector = game_status.token_vector[valid_simulations],
                                                                numerical_vector = [numerical_vector_[valid_simulations], time_diff_vector_[valid_simulations]])
    random_token_np[valid_simulations] = random_token
    prob_token_np[valid_simulations]   = prob_random_token

    # -------------------------
    # CORRECAO 1 (token fim cedo): token de encerramento sorteado porem ainda falta muito tempo a jogar.
    # verificando as simulacoes onde sera preciso sortear outro token, ja que nessas simulacoes 
    # ainda falta muito tempo a jogar, mas um dos tokens de encerramento foi sorteado para ser 
    # a proxima jogada. O criterio atual define que se falta mais de 15 segundos a jogar, o token 
    # de encerramento sorteado precisa ser substituido.
    # -------------------------
    mask_invalid_play = np.isin(random_token_np, end_period_tokens) & (list_remaining_time > 150)
    invalid_play_idx = np.where(mask_invalid_play)[0]

    while len(invalid_play_idx) > 0:
        # somente para as simulacoes que tiveram o token invalidado pelo contexto, sortearemos um novo token
        temp_random_token, temp_prob_random_token = predict_model_multi_tower(model = token_model,
                                                                              token_vector = game_status.token_vector[invalid_play_idx],
                                                                              numerical_vector = [numerical_vector_[invalid_play_idx], time_diff_vector_[invalid_play_idx]])


        # atualizando o vetor final com o novo vetor de tokens
        replacement_dict = {i: (temp_random_token[pos], temp_prob_random_token[pos]) for pos, i in enumerate(invalid_play_idx)}

        for idx, new_play in replacement_dict.items():
            random_token_np[idx] = new_play[0]
            prob_token_np[idx] = new_play[1]

        # atualizando a verificacao de tokens novamente: ainda existe token de encerramento sorteado cedo demais?
        mask_invalid_play = np.isin(random_token_np, end_period_tokens) & (list_remaining_time > 150)
        invalid_play_idx = np.where(mask_invalid_play)[0]

    # # -------------------------
    # # CORRECAO 2 (token fim do período errado): token de encerramento sortedo porem ele se refere a outro periodo.
    # # por causa do embedding, os tokens que fazem acoes parecidas ficam com pesos parecidos e dai 
    # # pode acontecer de estarmos gerando jogada para um periodo e o token de encerramento sorteado 
    # # se referir a outro periodo. Dai forcamos que o token sorteadose refira ao periodo em questao.
    # # -------------------------
    # mask_fix_last_token = np.isin(random_token_np, end_period_tokens)
    # invalid_last_token_idx = np.where(mask_fix_last_token)[0]
    # replacement_dict = {idx: fix_last_token(token = random_token_np[idx], period = period) for idx in invalid_last_token_idx}
    # for idx, new_play in replacement_dict.items():
    #     random_token_np[idx] = new_play

    # =========================================================
    # FASE 2: SORTEIO DO TEMPO (time_i)  [usa play_i e time_{i-1}]:     
    # com tudo corrigido para o sorteio do token, vamos ao sorteio do tempo gasto pela jogada sorteada na fase 1
    # O primeiro passo sera definir para quais simulacoes precisaremos gerar token de tempo gasto:
    # se um dos tokens de encerramento foi sorteado na fase 1, entao ele ocupa todo o tempo restante a ser jogado e 
    # dai nao precisa sortear tempo gasto para esses casos.
    # =========================================================
    mask_remaining_time_zero = np.isin(random_token_np, end_period_tokens)
    idx_zero_remaining_time = np.where(mask_remaining_time_zero)[0]
    idx_zero_remaining_time = idx_zero_remaining_time[np.isin(idx_zero_remaining_time, valid_simulations)]

    idx_random_remaining_time = np.where(~mask_remaining_time_zero)[0]
    idx_random_remaining_time = idx_random_remaining_time[np.isin(idx_random_remaining_time, valid_simulations)]

    # --- cria vetores temporários pós-play_i (sem aplicar no game_status real)

    temp_token_vector, temp_numerical_vector = update_vectors(
        period = period,
        random_play = random_token_np,
        tokens_vector = game_status.token_vector.copy(),
        home_points = game_status.home_points.copy(),
        away_points = game_status.away_points.copy(),
        limhteam = game_status.limhteam.copy(),
        limvteam = game_status.limvteam.copy(),
        limhtimeout = game_status.limhtimeout.copy(),
        limvtimeout = game_status.limvtimeout.copy(),
        remaining_time = game_status.remaining_time.copy(),
        ball_possession = game_status.ball_possession.copy(),
        indices = idx_random_remaining_time,
        config = config)

    temp_time_diff_vector_ = temp_numerical_vector[:, num_numerical_variables:]

    if match_vector is not None:
        temp_numerical_vector = np.hstack([temp_numerical_vector[:, :num_numerical_variables], match_vector[idx_random_remaining_time]])
    else:
        temp_numerical_vector = temp_numerical_vector[:, :num_numerical_variables]


    # agora:
    # temp_token_vector esta atualizado com a jogada sorteada na fase 1.
    # temp_numerical_vector esta atualizado com o impacto que a jogada sorteada na fase 1 causou no jogo, 
    # no entanto, o tempo restante nao considera essa jogada (pois depende diretamente do que vamos sortear agora)
    random_time = predict_model_multi_tower(model = time_model, token_vector = temp_token_vector, 
                                            numerical_vector = [temp_numerical_vector, temp_time_diff_vector_],
                                            type_ = 'time', extra_info = False)    

    # # -------------------------
    # # CORRECAO 3 (tempo > restante):
    # # o tempo sorteado eh maior que o tempo restante de jogo
    # # Quando isso acontece, existe duas opcoes do que fazer:
    # # 1. forcar o sorteio de um token de tempo que se enquadre no contexto do jogo, ou seja, sortear random_time 
    # #    ate que ele nao seja mais maior que o tempo restante de jogo.
    # # 2. descartar a jogada sorteada, ja que ela nao se enquadra com o contexto do jogo. 
    # # O que vamos fazer sera: vamos tentar por 3 vezes sortear um tempo valido para aqueles que tiveram problema,
    # # se nao resolver, analisaremos num momento futuro se descartaremos a jogada sorteada ou nao (dependera da jogada).
    # # -------------------------
    # invalid_time_mask = np.asarray(random_time) > list_remaining_time[idx_random_remaining_time]
    # idx_invalid_time = np.where(invalid_time_mask)[0]

    # invalid_time_flag = 0
    # while len(idx_invalid_time) > 0 and invalid_time_flag < 3:

    #     # somente para as simulacoes que tiveram tempo sorteado maior que tempo restante, sortearemos um novo tempo gasto
    #     temp_random_time = predict_model_multi_tower(model = time_model,
    #                                                  token_vector = temp_token_vector[idx_invalid_time],
    #                                                  numerical_vector = [temp_numerical_vector[idx_invalid_time], temp_time_diff_vector_[idx_invalid_time]],
    #                                                  type_ = 'time', extra_info = False)

    #     replacement_dict = {i: temp_random_time[pos] for pos, i in enumerate(idx_invalid_time)}
    #     # Atualizar random_time somente nesses índices        
    #     for idx, new_play in replacement_dict.items():
    #         random_time[idx] = new_play

    #     # atualizando a verificacao de tempo gasto novamente: ainda existe tempo gasto sorteado que ultrapassa o tempo restante?
    #     invalid_time_mask = np.asarray(random_time) > list_remaining_time[idx_random_remaining_time]
    #     idx_invalid_time = np.where(invalid_time_mask)[0]
    #     invalid_time_flag += 1

    # atualizando o vetor final de tempo gasto sorteado. Nas simulacoes onde um token de jogada nao foi um dos 
    # tokens de encerramento, colocamos o tempo gasto sorteado (lembrando que pode haver tempo sorteado amior 
    # que tempo restante de jogo). Nas simulacoes onde um token de jogada foi um dos tokens de encerramento, 
    # entao o tempo gasto foi o tempo restante de jogo.
    random_time_np[idx_random_remaining_time] = random_time
    random_time_np[idx_zero_remaining_time] = list_remaining_time[idx_zero_remaining_time]

    # CORRECAO 4: tempo gasto sorteado contem decimos de segundos, mas nao eh necessario.
    # Na NBA a contagem de tempo detalha apenas os minutos e segundos (mm:ss) nos 11 primeiros minutos iniciais do periodo, 
    # somente no minuto final que se detalha alem dos minutos e segundos, tambem os decimos de segundos (mm:ss:ss.f).
    # Se por acaso foi sorteado um token de tempo que envolva decimos de segundos, mas nao estamos no minuto final 
    # da partida, entao arredondamos esse tempo gasto para eliminar os decimos de segundos.

    # mask_fix_remaining_time = (list_remaining_time - random_time_np) > 600
    # idx_fix_remaining_time = np.where(mask_fix_remaining_time)[0]
    # idx_fix_remaining_time = idx_fix_remaining_time[np.isin(idx_fix_remaining_time, valid_simulations)]
    # rounded_time = (random_time_np  // 10) * 10
    # random_time_np[idx_fix_remaining_time] = rounded_time[idx_fix_remaining_time]

    # =========================================================
    # ATUALIZANDO GAME STATUS (IGUAL AO SEU)
    # =========================================================
    # com o token de jogada e o token de tempo gasto sorteados, vamos atualizar o game status, mas antes precisamos
    # resolver o problema de tempo gasto invalido. Isso impacta diretamente se vamos continuar gerando jogadas para 
    # aquela simulacao ou nao. Dai, vamos assinalar isso no vetor keep_generating do game status.    

    # keep_generating = 2 : significa que antes de sortar a jogada atual estavamos no fluxo normal de geracao de jogadas e 
    # entao um token de encerramento de periodo foi gerado. Dai assinalamos o keep_generating dessas simulacoes igual a 2.    
    mask_kp = (game_status.keep_generating == 1) & np.isin(random_token_np, end_period_tokens)
    idx_kp = np.where(mask_kp)[0]

    game_status.keep_generating[idx_kp] = 2

    # keep_generating = 3: significa que antes de sortear a jogada atual estavamos no fluxo normal de geracao de jogadas e
    # entao um token especial foi sorteado com tempo invalido. Dai assinalamos o keep_generating dessas simulacoes igual a 3.
    # O que sera feito eh: corrigir o tempo gasto por essa jogada, adiciona-la no game_status e dai encerrar a geracao de jogadas
    # para essas simulacoes ate que venha a jogada de encerramento.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time) & (np.isin(random_token_np, special_tokens))
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 3

    # keep_generating = 5: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token especial que exige complemento
    # foi sorteado com tempo invalido, essas simulacoes foram assinaladas com keep_generating = 4 e foi permitido que elas gerem esse 
    # complemento antes de encerrar a geracao de jogadas para essas simulacoes. 
    mask_kp = (game_status.keep_generating == 4)
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 5

    # keep_generating = 4: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token especial que exige complemento foi sorteado
    # com tempo invalido, assinalamos essas simulacoes com keep_generating = 4, corrigimos seu tempo gasto sorteado e vamos permitir que gere o 
    # complemento na proxima jogada, mas depois dai so a geracao de jogadas que encerrem o periodo serao permitidas.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time) & (np.isin(random_token_np, shot_blocked_tokens))
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 4

    # keep_generating = 6: significa que estavamos no fluxo normal de geracao de jogadas e ai, um token foi sorteado com tempo gasto invalido,
    # assinalamos essas simulacoes com keep_generating = 6 e nao permitimos que essa jogada seja colocada no game_status.
    mask_kp = (game_status.keep_generating == 1) & (random_time_np > game_status.remaining_time)
    idx_kp = np.where(mask_kp)[0]
    game_status.keep_generating[idx_kp] = 6

    # trocando o tempo gasto sorteado para as simulacoes com keep_generating iguais 3, 4, 5, para valores validos, que deixam o tempo restante igual a 0.
    mask_change_sorted_time = np.isin(game_status.keep_generating, [3, 4, 5])
    idx_change_sorted_time = np.where(mask_change_sorted_time)[0]

    random_time_np[idx_change_sorted_time] = list_remaining_time[idx_change_sorted_time]

    idx_update_game_status = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    # atualizando o game status para aqueles casos que o token de jogada sorteado e o tempo gasto sorteado foram considerados validos.
    update_game_status(game_status = game_status, random_play = random_token_np[idx_update_game_status], 
                       period = period, indices = idx_update_game_status, inplace = True, config = config)
    
    game_status.num_plays[idx_update_game_status] += 1
    game_status.sorted_token[idx_update_game_status] = random_token_np[idx_update_game_status]
    game_status.prob_token[idx_update_game_status] = prob_token_np[idx_update_game_status]
    game_status.sorted_time[idx_update_game_status] = random_time_np[idx_update_game_status]

    new_remaining_time = list_remaining_time[idx_update_game_status] - random_time_np[idx_update_game_status]
    game_status.remaining_time[idx_update_game_status] = new_remaining_time

    return


In [ ]:
def random_last_play(period, game_status, valid_simulations, token_model, config = 1, match_vector = None):
    dict_tokens = define_special_tokens(config = 1)

    # quando nao ha mais tempo restante para gerar jogadas, precisamos forcar que uma jogada que 
    # encerre o periodo seja sorteada aqui nao precisa sortear tempo gasto porque ele sera o tempo 
    # remanescente da ultima jogada
    # se o periodo é o ultimo ou qualquer um do overtime, então não precisa sortear porque só tem 
    # um token que encerra periodos nessas condiçoes.
    if period == 1: 
        only_consider_tokens = dict_tokens['end_period1']
    elif period == 2: 
        only_consider_tokens = dict_tokens['end_period2']
    elif period == 3: 
        only_consider_tokens = dict_tokens['end_period3']
    elif period == 4: 
        only_consider_tokens = dict_tokens['end_period4']
    else: 
        only_consider_tokens = dict_tokens['end_overtime']

    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector, match_vector])
        
    random_token, prob_random_token = predict_model(model = token_model, 
                                                    token_vector = game_status.token_vector[valid_simulations], 
                                                    numerical_vector = numerical_vector_[valid_simulations], 
                                                    only_consider = only_consider_tokens)
    
    # a partir dos valores sorteados, atualiza o game_status
    # a ultima jogada "gasta" o tempo que ha de jogo
    game_status.sorted_time[valid_simulations] = game_status.remaining_time[valid_simulations]
    game_status.prob_time[valid_simulations] = 1
    game_status.keep_generating[valid_simulations] = 0    
    game_status.remaining_time[valid_simulations] = 0

    update_game_status(game_status = game_status, random_play = random_token,
                       period = period, indices = valid_simulations, inplace = True, config = config)

    game_status.num_plays[valid_simulations] += 1
    game_status.sorted_token[valid_simulations] = np.asarray(random_token)
    game_status.prob_token[valid_simulations] = np.asarray(prob_random_token)

    return 

def random_last_play_st(period, game_status, valid_simulations, token_model, config = 1, match_vector = None):

    dict_tokens = define_special_tokens(config=1)

    if period == 1:
        only_consider_tokens = dict_tokens['end_period1']
    elif period == 2:
        only_consider_tokens = dict_tokens['end_period2']
    elif period == 3:
        only_consider_tokens = dict_tokens['end_period3']
    elif period == 4:
        only_consider_tokens = dict_tokens['end_period4']
    else:
        only_consider_tokens = dict_tokens['end_overtime']

    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector, match_vector])

    # STATE TOKEN para prever play_i (usa S_{i-1})
    state_id_token = categorize_time_bins_np(
        period_arr = np.full(game_status.remaining_time.shape[0], period, dtype = int),
        remaining_time_arr = game_status.remaining_time.astype(int),
        diff_points_arr = (game_status.home_points - game_status.away_points).astype(int)
    )

    random_token, prob_random_token = predict_model_st(model = token_model,
                                                       token_vector = game_status.token_vector[valid_simulations],
                                                       numerical_vector = numerical_vector_[valid_simulations],
                                                       state_token_vector = state_id_token[valid_simulations],
                                                       only_consider = only_consider_tokens)

    # a partir dos valores sorteados, atualiza o game_status
    # a ultima jogada "gasta" o tempo que ha de jogo
    game_status.sorted_time[valid_simulations] = game_status.remaining_time[valid_simulations]
    game_status.prob_time[valid_simulations] = 1
    game_status.keep_generating[valid_simulations] = 0
    game_status.remaining_time[valid_simulations] = 0

    update_game_status(game_status = game_status, random_play = random_token,
                       period = period, indices = valid_simulations, inplace = True, config = config)    

    game_status.num_plays[valid_simulations] += 1
    game_status.sorted_token[valid_simulations] = np.asarray(random_token)
    game_status.prob_token[valid_simulations] = np.asarray(prob_random_token)

    return

def random_last_play_timediff_tower(period, game_status, valid_simulations, token_model, config = 3, match_vector = None):

    dict_tokens = define_special_tokens(config=1)

    if period == 1:
        only_consider_tokens = dict_tokens['end_period1']
    elif period == 2:
        only_consider_tokens = dict_tokens['end_period2']
    elif period == 3:
        only_consider_tokens = dict_tokens['end_period3']
    elif period == 4:
        only_consider_tokens = dict_tokens['end_period4']
    else:
        only_consider_tokens = dict_tokens['end_overtime']

    if config == 3:
        num_numerical_variables = 15
    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector[:, :num_numerical_variables]
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector[:, :num_numerical_variables], match_vector])

    time_diff_vector_ = game_status.numerical_vector[:, num_numerical_variables:]


    random_token, prob_random_token = predict_model_timediff_tower(model = token_model,
                                                                   token_vector = game_status.token_vector[valid_simulations],
                                                                   numerical_vector = numerical_vector_[valid_simulations],
                                                                   time_diff_features = time_diff_vector_[valid_simulations],
                                                                   only_consider = only_consider_tokens)

    # a partir dos valores sorteados, atualiza o game_status
    # a ultima jogada "gasta" o tempo que ha de jogo
    game_status.sorted_time[valid_simulations] = game_status.remaining_time[valid_simulations]
    game_status.prob_time[valid_simulations] = 1
    game_status.keep_generating[valid_simulations] = 0
    game_status.remaining_time[valid_simulations] = 0

    update_game_status(game_status = game_status, random_play = random_token,
                       period = period, indices = valid_simulations, inplace = True, config = config)    

    game_status.num_plays[valid_simulations] += 1
    game_status.sorted_token[valid_simulations] = np.asarray(random_token)
    game_status.prob_token[valid_simulations] = np.asarray(prob_random_token)

    return

def random_last_play_multi_tower(period, game_status, valid_simulations, token_model, config = 3, match_vector = None):

    dict_tokens = define_special_tokens(config=1)

    if period == 1:
        only_consider_tokens = dict_tokens['end_period1']
    elif period == 2:
        only_consider_tokens = dict_tokens['end_period2']
    elif period == 3:
        only_consider_tokens = dict_tokens['end_period3']
    elif period == 4:
        only_consider_tokens = dict_tokens['end_period4']
    else:
        only_consider_tokens = dict_tokens['end_overtime']

    if config == 3:
        num_numerical_variables = 15
    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector[:, :num_numerical_variables]
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector[:, :num_numerical_variables], match_vector])

    time_diff_vector_ = game_status.numerical_vector[:, num_numerical_variables:]


    random_token, prob_random_token = predict_model_multi_tower(model = token_model,
                                                                token_vector = game_status.token_vector[valid_simulations],
                                                                numerical_vector = [numerical_vector_[valid_simulations], time_diff_vector_[valid_simulations]],
                                                                only_consider = only_consider_tokens)

    # a partir dos valores sorteados, atualiza o game_status
    # a ultima jogada "gasta" o tempo que ha de jogo
    game_status.sorted_time[valid_simulations] = game_status.remaining_time[valid_simulations]
    game_status.prob_time[valid_simulations] = 1
    game_status.keep_generating[valid_simulations] = 0
    game_status.remaining_time[valid_simulations] = 0

    update_game_status(game_status = game_status, random_play = random_token,
                       period = period, indices = valid_simulations, inplace = True, config = config)    

    game_status.num_plays[valid_simulations] += 1
    game_status.sorted_token[valid_simulations] = np.asarray(random_token)
    game_status.prob_token[valid_simulations] = np.asarray(prob_random_token)

    return



In [ ]:
def random_instant_replay(period, game_status, token_model, config = 1, match_vector = None):
    dict_tokens = define_special_tokens(config = 1)
    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector, match_vector])

    random_token, prob_random_token = predict_model(model = token_model, 
                                                    token_vector = game_status.token_vector, 
                                                    numerical_vector = numerical_vector_)
    
    idx_instant_review = np.where(np.isin(np.asarray(random_token), dict_tokens['instant_replay']))[0]

    if period > 4:
        idx_instant_review = np.isin(np.asarray(random_token), dict_tokens['instant_replay']) & (game_status.keep_generating > 0)
        idx_instant_review = np.where(idx_instant_review)

    update_game_status(game_status = game_status, random_play = np.asarray(random_token)[idx_instant_review],
                       period = period, indices = idx_instant_review, inplace = True, config = config)
    
    game_status.sorted_time[idx_instant_review] = 0
    game_status.prob_time[idx_instant_review] = 1

    game_status.num_plays[idx_instant_review] += 1
    game_status.sorted_token[idx_instant_review] = np.asarray(random_token)[idx_instant_review]
    game_status.prob_token[idx_instant_review] = np.asarray(prob_random_token)[idx_instant_review]

    return idx_instant_review

def random_instant_replay_st(period, game_status, token_model, config = 1, match_vector = None):
    dict_tokens = define_special_tokens(config = 1)

    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector, match_vector])

    # STATE TOKEN para prever play_i (usa S_{i-1})
    state_id_token = categorize_time_bins_np(
        period_arr = np.full(game_status.remaining_time.shape[0], period, dtype = int),
        remaining_time_arr = game_status.remaining_time.astype(int),
        diff_points_arr = (game_status.home_points - game_status.away_points).astype(int)
    )

    random_token, prob_random_token = predict_model_st(model = token_model, 
                                                       token_vector = game_status.token_vector, 
                                                       numerical_vector = numerical_vector_,
                                                       state_token_vector = state_id_token)

    idx_instant_review = np.where(np.isin(np.asarray(random_token), dict_tokens['instant_replay']))[0]

    if period > 4:
        idx_instant_review = np.isin(np.asarray(random_token), dict_tokens['instant_replay']) & (game_status.keep_generating > 0)
        idx_instant_review = np.where(idx_instant_review)[0]

    update_game_status(game_status = game_status, random_play = np.asarray(random_token)[idx_instant_review],
                       period = period, indices = idx_instant_review, inplace = True, config = config)
    

    game_status.sorted_time[idx_instant_review] = 0
    game_status.prob_time[idx_instant_review] = 1

    game_status.num_plays[idx_instant_review] += 1
    game_status.sorted_token[idx_instant_review] = np.asarray(random_token)[idx_instant_review]
    game_status.prob_token[idx_instant_review] = np.asarray(prob_random_token)[idx_instant_review]

    return idx_instant_review

def random_instant_replay_timediff_tower(period, game_status, token_model, config = 3, match_vector = None):

    dict_tokens = define_special_tokens(config = 1)

    if config == 3:
        num_numerical_variables = 15

    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector[:, :num_numerical_variables]
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector[:, :num_numerical_variables], match_vector])

    time_diff_vector_ = game_status.numerical_vector[:, num_numerical_variables:]
    
    random_token, prob_random_token = predict_model_timediff_tower(model = token_model, 
                                                                   token_vector = game_status.token_vector, 
                                                                   numerical_vector = numerical_vector_,
                                                                   time_diff_features = time_diff_vector_)

    idx_instant_review = np.where(np.isin(np.asarray(random_token), dict_tokens['instant_replay']))[0]

    if period > 4:
        idx_instant_review = np.isin(np.asarray(random_token), dict_tokens['instant_replay']) & (game_status.keep_generating > 0)
        idx_instant_review = np.where(idx_instant_review)[0]

    update_game_status(game_status = game_status, random_play = np.asarray(random_token)[idx_instant_review],
                       period = period, indices = idx_instant_review, inplace = True, config = config)
    

    game_status.sorted_time[idx_instant_review] = 0
    game_status.prob_time[idx_instant_review] = 1

    game_status.num_plays[idx_instant_review] += 1
    game_status.sorted_token[idx_instant_review] = np.asarray(random_token)[idx_instant_review]
    game_status.prob_token[idx_instant_review] = np.asarray(prob_random_token)[idx_instant_review]

    return idx_instant_review

def random_instant_replay_multi_tower(period, game_status, token_model, config = 3, match_vector = None):

    dict_tokens = define_special_tokens(config = 1)

    if config == 3:
        num_numerical_variables = 15

    if match_vector is None:
        numerical_vector_ = game_status.numerical_vector[:, :num_numerical_variables]
    else:
        numerical_vector_ = np.hstack([game_status.numerical_vector[:, :num_numerical_variables], match_vector])

    time_diff_vector_ = game_status.numerical_vector[:, num_numerical_variables:]
    
    random_token, prob_random_token = predict_model_multi_tower(model = token_model, 
                                                                token_vector = game_status.token_vector, 
                                                                numerical_vector = [numerical_vector_, time_diff_vector_])

    idx_instant_review = np.where(np.isin(np.asarray(random_token), dict_tokens['instant_replay']))[0]

    if period > 4:
        idx_instant_review = np.isin(np.asarray(random_token), dict_tokens['instant_replay']) & (game_status.keep_generating > 0)
        idx_instant_review = np.where(idx_instant_review)[0]

    update_game_status(game_status = game_status, random_play = np.asarray(random_token)[idx_instant_review],
                       period = period, indices = idx_instant_review, inplace = True, config = config)
    

    game_status.sorted_time[idx_instant_review] = 0
    game_status.prob_time[idx_instant_review] = 1

    game_status.num_plays[idx_instant_review] += 1
    game_status.sorted_token[idx_instant_review] = np.asarray(random_token)[idx_instant_review]
    game_status.prob_token[idx_instant_review] = np.asarray(prob_random_token)[idx_instant_review]

    return idx_instant_review


In [ ]:
def generate_period_plays(period, token_model, time_model, game_status, game_log, config = 1, match_vector = None):

    game_status.keep_generating[:] = 1
    
    if period > 4:
        idx_finished_games = np.where(game_status.home_points != game_status.away_points)[0]
        game_status.keep_generating[idx_finished_games] = 0
    
    print_status(game_status, period)
    valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    random_first_play(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                      token_model = token_model, config = config, match_vector = match_vector)

    while len(valid_simulations) > 0:
        clear_output(wait = True)   
        game_log.add_events(indices = valid_simulations, game_status = game_status)

        list_limhteam = game_status.limhteam.copy()
        list_limvteam = game_status.limvteam.copy()

        list_limhtimeout = game_status.limhtimeout.copy()
        list_limvtimeout = game_status.limvtimeout.copy()

        random_play_loop(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                        token_model = token_model, time_model = time_model, config = config)

        if period == 4:
            change_timeouts_limits(game_status = game_status, valid_simulations = valid_simulations, 
                                  list_limhtimeout = list_limhtimeout, list_limvtimeout = list_limhtimeout)

        change_fouls_limit_two_final_minutes(game_status = game_status, valid_simulations = valid_simulations, 
                                             list_limhteam = list_limhteam, list_limvteam = list_limvteam)

        print_status(game_status = game_status, period = period)

        # >>> FIX: salvar essas ANTES de sobrescrever keep_generating <<<
        idx_to_log = np.where(
            (game_status.keep_generating == 2) |
            (game_status.keep_generating == 3) |
            (game_status.keep_generating == 5)
        )[0]

        if len(idx_to_log) > 0:
            game_log.add_events(indices = idx_to_log, game_status = game_status)

        # keep_generating = 0: nao precisa gerar mais nada
        mask_kp = (game_status.keep_generating == 2)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 0

        # keep_generating = 6: so precisa gerar a jogada de encerramento
        mask_kp = (game_status.keep_generating == 3) | (game_status.keep_generating == 5)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 6

        valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    valid_simulations = np.where((game_status.keep_generating >= 6))[0]

    random_last_play(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                     token_model = token_model, config = config)

    game_log.add_events(indices = valid_simulations, game_status = game_status)

    # if period <= 4:
    #     idx_instant_replay = random_instant_replay(period, game_status, token_model, config = config, match_vector = None)
    #     game_log.add_events(indices = idx_instant_replay, game_status = game_status)
    
    print_status(game_status = game_status, period = period)
    return 

def generate_period_plays_2models(period, token_model1, token_model2, time_model1, time_model2, game_status, game_log, config = 1, match_vector = None):
    
    game_status.keep_generating[:] = 1
    
    if period > 4:
        idx_finished_games = np.where(game_status.home_points != game_status.away_points)[0]
        game_status.keep_generating[idx_finished_games] = 0
    
    print_status(game_status, period)
    valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    random_first_play(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                      token_model = token_model1, config = config, match_vector = match_vector)

    while len(valid_simulations) > 0:
        clear_output(wait = True)   
        game_log.add_events(indices = valid_simulations, game_status = game_status)

        list_limhteam = game_status.limhteam.copy()
        list_limvteam = game_status.limvteam.copy()

        list_limhtimeout = game_status.limhtimeout.copy()
        list_limvtimeout = game_status.limvtimeout.copy()

        random_play_loop(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                        token_model = token_model1, time_model = time_model1, config = config)

        if period == 4:
            change_timeouts_limits(game_status = game_status, valid_simulations = valid_simulations, 
                                  list_limhtimeout = list_limhtimeout, list_limvtimeout = list_limhtimeout)

        change_fouls_limit_two_final_minutes(game_status = game_status, valid_simulations = valid_simulations, 
                                             list_limhteam = list_limhteam, list_limvteam = list_limvteam)

        print_status(game_status = game_status, period = period)

        mask_two_final_minutes_game = (game_status.remaining_time <= 1200) & (game_status.period >= 4)
        idx_two_final_minutes_game = np.where(mask_two_final_minutes_game)[0]
        game_status.keep_generating[idx_two_final_minutes_game] = 7

        # >>> FIX: salvar essas ANTES de sobrescrever keep_generating <<<
        idx_to_log = np.where(
            (game_status.keep_generating == 2) |
            (game_status.keep_generating == 3) |
            (game_status.keep_generating == 5)
        )[0]

        if len(idx_to_log) > 0:
            game_log.add_events(indices = idx_to_log, game_status = game_status)

        # keep_generating = 0: nao precisa gerar mais nada
        mask_kp = (game_status.keep_generating == 2)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 0

        # keep_generating = 6: so precisa gerar a jogada de encerramento
        mask_kp = (game_status.keep_generating == 3) | (game_status.keep_generating == 5)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 6

        valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    valid_simulations = np.where((game_status.keep_generating == 7))[0]
    game_status.keep_generating[valid_simulations] = 1
    
    while len(valid_simulations) > 0:
        clear_output(wait = True)   
        game_log.add_events(indices = valid_simulations, game_status = game_status)

        list_limhteam = game_status.limhteam.copy()
        list_limvteam = game_status.limvteam.copy()

        list_limhtimeout = game_status.limhtimeout.copy()
        list_limvtimeout = game_status.limvtimeout.copy()

        random_play_loop(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                        token_model = token_model2, time_model = time_model2, config = config)

        if period == 4:
            change_timeouts_limits(game_status = game_status, valid_simulations = valid_simulations, 
                                  list_limhtimeout = list_limhtimeout, list_limvtimeout = list_limhtimeout)

        change_fouls_limit_two_final_minutes(game_status = game_status, valid_simulations = valid_simulations, 
                                            list_limhteam = list_limhteam, list_limvteam = list_limvteam)
    
        print_status(game_status = game_status, period = period)

        # >>> FIX: salvar essas ANTES de sobrescrever keep_generating <<<
        idx_to_log = np.where(
            (game_status.keep_generating == 2) |
            (game_status.keep_generating == 3) |
            (game_status.keep_generating == 5)
        )[0]

        if len(idx_to_log) > 0:
            game_log.add_events(indices = idx_to_log, game_status = game_status)

        # keep_generating = 0: nao precisa gerar mais nada
        mask_kp = (game_status.keep_generating == 2)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 0

        # keep_generating = 6: so precisa gerar a jogada de encerramento
        mask_kp = (game_status.keep_generating == 3) | (game_status.keep_generating == 5)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 6

        valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    valid_simulations = np.where((game_status.keep_generating >= 6))[0]

    if period < 4:
        random_last_play(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                         token_model = token_model1, config = config)

        game_log.add_events(indices = valid_simulations, game_status = game_status)

    else:
        random_last_play(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                         token_model = token_model2, config = config)

        game_log.add_events(indices = valid_simulations, game_status = game_status)
        
    # if period < 4:
    #     idx_instant_replay = random_instant_replay(period, game_status, token_model1, config = config, match_vector = None)
    #     game_log.add_events(indices = idx_instant_replay, game_status = game_status)

    # if period == 4:
    #     idx_instant_replay = random_instant_replay(period, game_status, token_model2, config = config, match_vector = None)
    #     game_log.add_events(indices = idx_instant_replay, game_status = game_status)
    
    print_status(game_status = game_status, period = period)
    return 

def generate_period_plays_st(period, token_model, time_model, game_status, game_log, config = 1, match_vector = None):
    
    game_status.keep_generating[:] = 1
    
    if period > 4:
        idx_finished_games = np.where(game_status.home_points != game_status.away_points)[0]
        game_status.keep_generating[idx_finished_games] = 0
    
    print_status(game_status, period)
    valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    random_first_play_st(period = period, game_status = game_status, valid_simulations = valid_simulations, token_model = token_model, 
                         config = config, match_vector = match_vector)

    while len(valid_simulations) > 0:
        clear_output(wait = True)   
        game_log.add_events(indices = valid_simulations, game_status = game_status)

        list_limhteam = game_status.limhteam.copy()
        list_limvteam = game_status.limvteam.copy()
        list_limhtimeout = game_status.limhtimeout.copy()
        list_limvtimeout = game_status.limvtimeout.copy()

        random_play_loop_st(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                            token_model = token_model, time_model = time_model, config = config)

        if period == 4:
            change_timeouts_limits(game_status = game_status, valid_simulations = valid_simulations, 
                                  list_limhtimeout = list_limhtimeout, list_limvtimeout = list_limhtimeout)

        change_fouls_limit_two_final_minutes(game_status = game_status, valid_simulations = valid_simulations, 
                                            list_limhteam = list_limhteam, list_limvteam = list_limvteam)
        
        print_status(game_status = game_status, period = period)
        # >>> FIX: salvar essas ANTES de sobrescrever keep_generating <<<
        idx_to_log = np.where(
            (game_status.keep_generating == 2) |
            (game_status.keep_generating == 3) |
            (game_status.keep_generating == 5)
        )[0]

        if len(idx_to_log) > 0:
            game_log.add_events(indices = idx_to_log, game_status = game_status)

        # keep_generating = 0: nao precisa gerar mais nada
        mask_kp = (game_status.keep_generating == 2)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 0

        # keep_generating = 6: so precisa gerar a jogada de encerramento
        mask_kp = (game_status.keep_generating == 3) | (game_status.keep_generating == 5)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 6

        valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    valid_simulations = np.where((game_status.keep_generating >= 6))[0]

    random_last_play_st(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                        token_model = token_model, config = config)

    game_log.add_events(indices = valid_simulations, game_status = game_status)

    # if period <= 4:
    #     idx_instant_replay = random_instant_replay_st(period, game_status, token_model, config = config, match_vector = None)
    #     game_log.add_events(indices = idx_instant_replay, game_status = game_status)
    
    print_status(game_status = game_status, period = period)
    return 

def generate_period_plays_timediff_tower(period, token_model, time_model, game_status, game_log, config = 3, match_vector = None):
    
    game_status.keep_generating[:] = 1
    
    if period > 4:
        idx_finished_games = np.where(game_status.home_points != game_status.away_points)[0]
        game_status.keep_generating[idx_finished_games] = 0
    
    print_status(game_status, period)
    valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    random_first_play_timediff_tower(period = period, game_status = game_status, valid_simulations = valid_simulations, token_model = token_model, 
                                     config = config, match_vector = match_vector)

    while len(valid_simulations) > 0:
        clear_output(wait = True)   
        game_log.add_events(indices = valid_simulations, game_status = game_status)

        list_limhteam = game_status.limhteam.copy()
        list_limvteam = game_status.limvteam.copy()
        list_limhtimeout = game_status.limhtimeout.copy()
        list_limvtimeout = game_status.limvtimeout.copy()

        random_play_loop_timediff_tower(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                                        token_model = token_model, time_model = time_model, config = config)

        if period == 4:
            change_timeouts_limits(game_status = game_status, valid_simulations = valid_simulations, 
                                  list_limhtimeout = list_limhtimeout, list_limvtimeout = list_limhtimeout)

        change_fouls_limit_two_final_minutes(game_status = game_status, valid_simulations = valid_simulations, 
                                             list_limhteam = list_limhteam, list_limvteam = list_limvteam)

        
        print_status(game_status = game_status, period = period)
        # >>> FIX: salvar essas ANTES de sobrescrever keep_generating <<<
        idx_to_log = np.where(
            (game_status.keep_generating == 2) |
            (game_status.keep_generating == 3) |
            (game_status.keep_generating == 5)
        )[0]

        if len(idx_to_log) > 0:
            game_log.add_events(indices = idx_to_log, game_status = game_status)

        # keep_generating = 0: nao precisa gerar mais nada
        mask_kp = (game_status.keep_generating == 2)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 0

        # keep_generating = 6: so precisa gerar a jogada de encerramento
        mask_kp = (game_status.keep_generating == 3) | (game_status.keep_generating == 5)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 6

        valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    valid_simulations = np.where((game_status.keep_generating >= 6))[0]

    random_last_play_timediff_tower(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                                    token_model = token_model, config = config)

    game_log.add_events(indices = valid_simulations, game_status = game_status)

    # if period <= 4:
    #     idx_instant_replay = random_instant_replay_timediff_tower(period, game_status, token_model, config = config, match_vector = None)
    #     game_log.add_events(indices = idx_instant_replay, game_status = game_status)
    
    print_status(game_status = game_status, period = period)
    return 

def generate_period_plays_multi_tower(period, token_model, time_model, game_status, game_log, config = 3, match_vector = None):
    
    game_status.keep_generating[:] = 1
    
    if period > 4:
        idx_finished_games = np.where(game_status.home_points != game_status.away_points)[0]
        game_status.keep_generating[idx_finished_games] = 0
    
    print_status(game_status, period)
    valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    random_first_play_multi_tower(period = period, game_status = game_status, valid_simulations = valid_simulations, token_model = token_model, 
                                  config = config, match_vector = match_vector)

    while len(valid_simulations) > 0:
        clear_output(wait = True)   
        game_log.add_events(indices = valid_simulations, game_status = game_status)

        list_limhteam = game_status.limhteam.copy()
        list_limvteam = game_status.limvteam.copy()
        list_limhtimeout = game_status.limhtimeout.copy()
        list_limvtimeout = game_status.limvtimeout.copy()

        random_play_loop_multi_tower(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                                     token_model = token_model, time_model = time_model, config = config)

        if period == 4:
            change_timeouts_limits(game_status = game_status, valid_simulations = valid_simulations, 
                                  list_limhtimeout = list_limhtimeout, list_limvtimeout = list_limhtimeout)


        change_fouls_limit_two_final_minutes(game_status = game_status, valid_simulations = valid_simulations, 
                                             list_limhteam = list_limhteam, list_limvteam = list_limvteam)

        
        print_status(game_status = game_status, period = period)
        # >>> FIX: salvar essas ANTES de sobrescrever keep_generating <<<
        idx_to_log = np.where(
            (game_status.keep_generating == 2) |
            (game_status.keep_generating == 3) |
            (game_status.keep_generating == 5)
        )[0]

        if len(idx_to_log) > 0:
            game_log.add_events(indices = idx_to_log, game_status = game_status)

        # keep_generating = 0: nao precisa gerar mais nada
        mask_kp = (game_status.keep_generating == 2)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 0

        # keep_generating = 6: so precisa gerar a jogada de encerramento
        mask_kp = (game_status.keep_generating == 3) | (game_status.keep_generating == 5)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 6

        valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    valid_simulations = np.where((game_status.keep_generating >= 6))[0]

    random_last_play_multi_tower(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                                 token_model = token_model, config = config)

    game_log.add_events(indices = valid_simulations, game_status = game_status)

    # if period <= 4:
    #     idx_instant_replay = random_instant_replay_multi_tower(period, game_status, token_model, config = config, match_vector = None)
    #     game_log.add_events(indices = idx_instant_replay, game_status = game_status)
    
    print_status(game_status = game_status, period = period)
    return 



In [ ]:
def generate_game_plays(token_model, time_model, num_simulations, token_context, num_numerical_variables, config = 1, match_vector = None):
    
    game_log = GameLog(n_sims = num_simulations)
    game_status = GameStatus.from_initial(n_sims = num_simulations, window = token_context, n_features = num_numerical_variables)

    period_duration = 7200
    period_limhteam = 4 
    period_limvteam = 4 
    game_limhtimeout = 7 
    game_limvtimeout = 7 

    for p in [1, 2, 3, 4]:
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        game_status.limhtimeout[:] = game_limhtimeout
        game_status.limvtimeout[:] = game_limvtimeout
    
        generate_period_plays(period = p, token_model = token_model, time_model = time_model, game_status = game_status, 
                              game_log = game_log, match_vector = match_vector, config = config)
            
    num_ties = sum(game_status.home_points == game_status.away_points)
    period_duration = 3000
    period_limhteam = 3 
    period_limvteam = 3 
    game_limhtimeout = 2 
    game_limvtimeout = 2 
    p = 5
    while num_ties > 0:
        
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        game_status.limhtimeout[:] = game_limhtimeout
        game_status.limvtimeout[:] = game_limvtimeout

        generate_period_plays(period = p, token_model = token_model, time_model = time_model, game_status = game_status, 
                              game_log = game_log, match_vector = match_vector, config = config)
        
        p += 1
        num_ties = sum(game_status.home_points == game_status.away_points)
    
    return game_log.to_dataframe()

def generate_game_plays_2models(token_model1, token_model2, time_model1, time_model2, num_simulations, token_context, num_numerical_variables, config = 1, match_vector = None):
    
    game_log = GameLog(n_sims = num_simulations)
    game_status = GameStatus.from_initial(n_sims = num_simulations, window = token_context, n_features = num_numerical_variables)

    period_duration = 7200
    period_limhteam = 4 
    period_limvteam = 4 
    game_limhtimeout = 7 
    game_limvtimeout = 7 

    for p in [1, 2, 3]:
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        game_status.limhtimeout[:] = game_limhtimeout
        game_status.limvtimeout[:] = game_limvtimeout
    
        generate_period_plays(period = p, token_model = token_model1, time_model = time_model1, game_status = game_status, 
                              game_log = game_log, match_vector = match_vector, config = config)

    game_status.remaining_time[:] = period_duration
    game_status.limhteam[:] = period_limhteam
    game_status.limvteam[:] = period_limvteam
    game_status.limhtimeout[:] = game_limhtimeout
    game_status.limvtimeout[:] = game_limvtimeout

    generate_period_plays_2models(period = 4, token_model1 = token_model1, time_model1 = time_model1, 
                                  token_model2 = token_model2, time_model2 = time_model2, 
                                  game_status = game_status, game_log = game_log, 
                                  match_vector = match_vector, config = config)

    num_ties = sum(game_status.home_points == game_status.away_points)
    period_duration = 3000
    period_limhteam = 3 
    period_limvteam = 3 
    game_limhtimeout = 2 
    game_limvtimeout = 2 
    p = 5
    while num_ties > 0:
        
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        game_status.limhtimeout[:] = game_limhtimeout
        game_status.limvtimeout[:] = game_limvtimeout

        generate_period_plays_2models(period = p, token_model1 = token_model1, time_model1 = time_model1, 
                                      token_model2 = token_model2, time_model2 = time_model2,  
                                      game_status = game_status, game_log = game_log, 
                                      match_vector = match_vector, config = config)
        
        p += 1
        num_ties = sum(game_status.home_points == game_status.away_points)
    
    return game_log.to_dataframe()

def generate_game_plays_st(token_model, time_model, num_simulations, token_context, num_numerical_variables, config = 1, match_vector = None):
    
    game_log = GameLog(n_sims = num_simulations)
    game_status = GameStatus.from_initial(n_sims = num_simulations, window = token_context, n_features = num_numerical_variables)

    period_duration = 7200
    period_limhteam = 4 
    period_limvteam = 4 
    game_limhtimeout = 7 
    game_limvtimeout = 7 

    for p in [1, 2, 3, 4]:
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        game_status.limhtimeout[:] = game_limhtimeout
        game_status.limvtimeout[:] = game_limvtimeout
    
        generate_period_plays_st(period = p, token_model = token_model, time_model = time_model, game_status = game_status, 
                                 game_log = game_log, match_vector = match_vector, config = config)
            
    num_ties = sum(game_status.home_points == game_status.away_points)
    period_duration = 3000
    period_limhteam = 3 
    period_limvteam = 3 
    game_limhtimeout = 2 
    game_limvtimeout = 2 

    p = 5
    while num_ties > 0:
        
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        game_status.limhtimeout[:] = game_limhtimeout
        game_status.limvtimeout[:] = game_limvtimeout

        generate_period_plays_st(period = p, token_model = token_model, time_model = time_model, game_status = game_status, 
                                 game_log = game_log, match_vector = match_vector, config = config)
        
        p += 1
        num_ties = sum(game_status.home_points == game_status.away_points)
    
    return game_log.to_dataframe()

def generate_game_plays_timediff_tower(token_model, time_model, num_simulations, token_context, num_numerical_variables, config = 3, match_vector = None):
    
    game_log = GameLog(n_sims = num_simulations)
    game_status = GameStatus.from_initial(n_sims = num_simulations, window = token_context, n_features = num_numerical_variables)

    period_duration = 7200
    period_limhteam = 4 
    period_limvteam = 4 
    game_limhtimeout = 7 
    game_limvtimeout = 7 

    for p in [1, 2, 3, 4]:
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        game_status.limhtimeout[:] = game_limhtimeout
        game_status.limvtimeout[:] = game_limvtimeout
    
        generate_period_plays_timediff_tower(period = p, token_model = token_model, time_model = time_model, game_status = game_status, 
                                             game_log = game_log, match_vector = match_vector, config = config)
            
    num_ties = sum(game_status.home_points == game_status.away_points)
    period_duration = 3000
    period_limhteam = 3 
    period_limvteam = 3 
    game_limhtimeout = 2 
    game_limvtimeout = 2 

    p = 5
    while num_ties > 0:
        
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        game_status.limhtimeout[:] = game_limhtimeout
        game_status.limvtimeout[:] = game_limvtimeout

        generate_period_plays_timediff_tower(period = p, token_model = token_model, time_model = time_model, game_status = game_status, 
                                             game_log = game_log, match_vector = match_vector, config = config)
        
        p += 1
        num_ties = sum(game_status.home_points == game_status.away_points)
    
    return game_log.to_dataframe()

def generate_game_plays_multi_tower(token_model, time_model, num_simulations, token_context, num_numerical_variables, config = 3, match_vector = None):
    
    game_log = GameLog(n_sims = num_simulations)
    game_status = GameStatus.from_initial(n_sims = num_simulations, window = token_context, n_features = num_numerical_variables)

    period_duration = 7200
    period_limhteam = 4 
    period_limvteam = 4 
    game_limhtimeout = 7 
    game_limvtimeout = 7 

    for p in [1, 2, 3, 4]:
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        game_status.limhtimeout[:] = game_limhtimeout
        game_status.limvtimeout[:] = game_limvtimeout
    
        generate_period_plays_multi_tower(period = p, token_model = token_model, time_model = time_model, game_status = game_status, 
                                          game_log = game_log, match_vector = match_vector, config = config)
            
    num_ties = sum(game_status.home_points == game_status.away_points)
    period_duration = 3000
    period_limhteam = 3 
    period_limvteam = 3 
    game_limhtimeout = 2 
    game_limvtimeout = 2 

    p = 5
    while num_ties > 0:
        
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        game_status.limhtimeout[:] = game_limhtimeout
        game_status.limvtimeout[:] = game_limvtimeout

        generate_period_plays_multi_tower(period = p, token_model = token_model, time_model = time_model, game_status = game_status, 
                                          game_log = game_log, match_vector = match_vector, config = config)
        
        p += 1
        num_ties = sum(game_status.home_points == game_status.away_points)
    
    return game_log.to_dataframe()


In [ ]:
def preprocess_vectors_to_predict(class_model, token_vector, numerical_vector, state_token_vector = None, time_diff_features = None, mask_data = None):
    
    # 1. Transformando listas em np.arrays (Sem exceções malucas para MultiTower)
    if isinstance(token_vector, list):
        token_vector = np.array(object = token_vector, dtype = "int32")

    if isinstance(numerical_vector, list):
        numerical_vector = np.array(object = numerical_vector)

    if isinstance(state_token_vector, list):
        state_token_vector = np.array(object = state_token_vector, dtype = "int32")
    
    if isinstance(time_diff_features, list):
        time_diff_features = np.array(object = time_diff_features)

    # 2. Ajuste de dimensão para casos 1D (Batch = 1)
    if len(token_vector.shape) == 1:
        token_vector = token_vector.reshape(1, -1)
        numerical_vector = numerical_vector.reshape(1, -1)

        if class_model == "StateVector" and state_token_vector is not None:
            if state_token_vector.ndim == 0:
                state_token_vector = state_token_vector.reshape(1)
            if state_token_vector.shape[0] == 1 and token_vector.shape[0] > 1:
                state_token_vector = np.full(shape = token_vector.shape[0], fill_value = state_token_vector[0])

        # Se existir time_diff_features, arruma o shape dele também
        if time_diff_features is not None and len(time_diff_features.shape) == 1:
            time_diff_features = time_diff_features.reshape(1, -1)

    # 3. Transformação em Tensores PyTorch (Direto e Reto)
    if not isinstance(token_vector, torch.Tensor):
        token_vector = torch.tensor(data = token_vector)

    if not isinstance(numerical_vector, torch.Tensor):
        numerical_vector = torch.tensor(data = numerical_vector, dtype = torch.float32)
    
    if state_token_vector is not None and not isinstance(state_token_vector, torch.Tensor) and class_model == "StateVector":
        state_token_vector = torch.tensor(data = state_token_vector)
        
    # ATENÇÃO: Tanto o TimeDiffTower quanto o MultiTower usam essa feature, então convertemos para os dois!
    if time_diff_features is not None and not isinstance(time_diff_features, torch.Tensor) and class_model in ["TimeDiffTower", "MultiTower"]:
        time_diff_features = torch.tensor(data = time_diff_features, dtype = torch.float32)

    if mask_data is not None and not isinstance(mask_data, torch.Tensor):
        mask_data = torch.tensor(mask_data, dtype = torch.bool)

    return token_vector, numerical_vector, state_token_vector, time_diff_features, mask_data

def posprocess_model_probs(logits, k, only_consider, extra_info, type_):
    logits = logits.clone()

    if only_consider is not None:
        #allowed = torch.isin(torch.arange(logits.size(1)), torch.tensor(only_consider))
        allowed = torch.isin(
            elements = torch.arange(end = logits.size(1), device = logits.device), 
            test_elements = torch.tensor(data = only_consider, device = logits.device))
            
        logits[:, ~allowed] = -float("inf")

    probabilities = torch.softmax(input = logits, dim = 1)

    if k > 0:
        topk = torch.topk(input = probabilities, k = k, dim = 1)
        topk_preds = {"indices": list(topk.indices.detach().numpy()[0]),
                      "probs":   list(topk.values.detach().numpy()[0])}
        return pd.DataFrame(topk_preds)
    
    sorted_items = torch.multinomial(input = probabilities, num_samples = 1).squeeze(1).detach().cpu().tolist()
    sorted_probs = probabilities[torch.arange(probabilities.size(0)), sorted_items].detach().cpu().tolist()

    if extra_info: 
        return sorted_items, sorted_probs
        #return sorted_items, sum_probs
    return sorted_items

def predict_model_v2(model, class_model, token_vector, numerical_vector, state_token_vector = None, time_diff_features = None, k = 0, only_consider = None, extra_info = True, type_ = "tokens", mask_data = None):

    token_vector, numerical_vector, state_token_vector, time_diff_features, mask_data = preprocess_vectors_to_predict(
        class_model, token_vector, numerical_vector, state_token_vector, time_diff_features, mask_data)

    # gerando os log-valores para os tokens de jogada ou tempo a partir dos vetores de input.
    with torch.inference_mode():
        if class_model == "ExtraInfo":
            logits = model(train_token = token_vector, train_numerical = numerical_vector)
        if class_model == "StateVector":
            logits = model(train_token = token_vector, train_numerical = numerical_vector, train_state = state_token_vector)
        if class_model == "TimeDiffTower":
            logits = model(train_token = token_vector, time_diff_features = time_diff_features, train_numerical = numerical_vector)
        if class_model == "MultiTower":
            logits = model(token_ids = token_vector, numerical_groups = [numerical_vector, time_diff_features])
        if mask_data is not None:
            logits = logits.masked_fill(mask_data, -1e9)
            
    return posprocess_model_probs(logits = logits, k = k, only_consider = only_consider, extra_info = extra_info, type_ = type_)

def prepare_model_inputs(period, game_status, class_model, config_setvariables = 0, match_vector = None):
    """
    Centraliza a preparação dos vetores de entrada (Numerical, State, TimeDiff) globais
    para qualquer arquitetura de modelo. O fatiamento [valid_simulations] deve ser feito 
    pelas funções na hora da inferência.
    """
    # 1. Define a configuração de variáveis baseada na arquitetura
    if class_model in ["ExtraInfo", "StateVector"] and config_setvariables == 0:
        config_setvariables = 1
    elif class_model in ["TimeDiffTower", "MultiTower"] and config_setvariables == 0:
        config_setvariables = 3

    # 2. Puxa a base numérica GLOBAL
    base_numerical = game_status.numerical_vector
    mv_subset = match_vector # Global

    numerical_vector_ = None
    time_diff_vector_ = None
    state_id_token = None

    # 3. Montagem para Modelos Padrão / State
    if class_model in ["ExtraInfo", "StateVector"]:
        numerical_vector_ = base_numerical if mv_subset is None else np.hstack([base_numerical, mv_subset])

    # 4. Montagem para Modelos Tower (Divide as features de tempo)
    elif class_model in ["TimeDiffTower", "MultiTower"]:
        num_numerical_variables = 15 if config_setvariables == 3 else 16 
        
        base_num_sliced = base_numerical[:, :num_numerical_variables]
        time_diff_vector_ = base_numerical[:, num_numerical_variables:]
        
        numerical_vector_ = base_num_sliced if mv_subset is None else np.hstack([base_num_sliced, mv_subset])

    # 5. Cálculo Dinâmico do State Token GLOBAL (Apenas para o modelo StateVector)
    if class_model == "StateVector":
        state_id_token = categorize_time_bins_np(
            period_arr = np.full(game_status.n_sims, period, dtype=int),
            remaining_time_arr = game_status.remaining_time.astype(int),
            diff_points_arr = (game_status.home_points - game_status.away_points).astype(int)
        )

    return config_setvariables, numerical_vector_, time_diff_vector_, state_id_token

def get_random_first_play(period, game_status, valid_simulations, token_model, class_model, config_setvariables = 0, match_vector = None):
    
    # Just some sets of the vectors depending of the class model
    dict_tokens = define_special_tokens(1)

    # 1. Chama a nossa nova central de preparação num único comando!
    config_setvariables, numerical_vector_, time_diff_vector_, state_id_token = prepare_model_inputs(
        period = period, game_status = game_status, class_model = class_model, config_setvariables = config_setvariables, match_vector = match_vector)

    token_vector = game_status.token_vector[valid_simulations]

    token_mask = build_logit_mask(game_status = game_status, valid_simulations = valid_simulations,
                                  mode = "first_play", target = "token", period = period, 
                                  output_size = 601)

    # Fatiamento no momento da inferência!
    num_vec_sliced = numerical_vector_[valid_simulations]
    
    state_sliced = state_id_token[valid_simulations] if state_id_token is not None else None
    td_sliced = time_diff_vector_[valid_simulations] if time_diff_vector_ is not None else None

    play, prob = predict_model_v2(
        model = token_model, class_model = class_model, token_vector = token_vector, 
        numerical_vector = num_vec_sliced, state_token_vector = state_sliced, 
        time_diff_features = td_sliced, mask_data = token_mask)

    ########## update some context variables in game_status ##########
    # if we are in first period, we need to sort the game's first play and then define which team got the ball possession
    if period == 1:
        game_status.ball_possession[valid_simulations] = define_ball_possession_vectorized(
            token = play, last_token = 0, period = period, ball_possession = 0)

    # if we are sorting play for any period but first, we need to reset some game status variables
    if period > 1:
        period_duration = 7200 if period <= 4 else 3000
        allowed_fouls_home, allowed_fouls_away = define_game_fouls(period)

        game_status.period[valid_simulations] = period
        game_status.limhteam[valid_simulations] = allowed_fouls_home
        game_status.limvteam[valid_simulations] = allowed_fouls_away
        game_status.remaining_time[valid_simulations] = period_duration

    # if game goes to overtime, the regular period rule for ball possession is useless and every overtime period starts with a jumpball to define ball possession.
    if period >= 5:
        game_status.ball_possession[valid_simulations] = 0
        game_status.limhtimeout[valid_simulations] = 2
        game_status.limvtimeout[valid_simulations] = 2

    if period == 4:
        num_timeouts_home = np.minimum(game_status.limhtimeout[valid_simulations], 4)
        num_timeouts_away = np.minimum(game_status.limvtimeout[valid_simulations], 4)

        game_status.limhtimeout[valid_simulations] = num_timeouts_home
        game_status.limvtimeout[valid_simulations] = num_timeouts_away

    ########## update the game_status with the sorted play ##########
    update_game_status(game_status = game_status, random_play = play, period = period, 
                       indices = valid_simulations, inplace = True, config = config_setvariables)

    game_status.num_plays[valid_simulations] += 1
    game_status.sorted_token[valid_simulations] = np.asarray(play)
    game_status.prob_token[valid_simulations] = np.asarray(prob)
    
    # The first period play "spends" time zero, by definition. So, we don't need to sort spend time by play
    game_status.sorted_time[valid_simulations] = 0
    game_status.prob_time[valid_simulations] = 1

    return 


def get_play_prob_loop_function(model, class_model, valid_simulations, game_status, numerical_vector_, state_id_token, time_diff_vector_, mask_data = None):
    if class_model == "ExtraInfo":
        play, prob = predict_model_v2(model = model, class_model = class_model, 
                                      token_vector = game_status.token_vector[valid_simulations], 
                                      numerical_vector = numerical_vector_[valid_simulations],
                                      mask_data = mask_data)
    elif class_model == "StateVector":
        play, prob = predict_model_v2(model = model, class_model = class_model, 
                                      token_vector = game_status.token_vector[valid_simulations], 
                                      numerical_vector = numerical_vector_[valid_simulations],
                                      state_token_vector = state_id_token[valid_simulations],
                                      mask_data = mask_data)
    elif class_model in ["TimeDiffTower", "MultiTower"]:
        # Agrupados! Ambos passam os argumentos separadamente. A mágica da lista do MultiTower 
        # vai acontecer lá dentro do 'with torch.inference_mode():' da predict_model_v2.
        play, prob = predict_model_v2(model = model, class_model = class_model, 
                                      token_vector = game_status.token_vector[valid_simulations], 
                                      numerical_vector = numerical_vector_[valid_simulations],
                                      time_diff_features = time_diff_vector_[valid_simulations],
                                      mask_data = mask_data)
    else:
        print("get a valid class model name")
        return None, None
    return play, prob    

def substitute_end_period_token_out_of_context(np_token_vector, np_prob_vector, list_remaining_time, model, class_model, game_status, numerical_vector_, state_id_token, time_diff_vector_, end_period_tokens, matrix_end_period_probs, max_attempts = 100):

    np_token_vector = np_token_vector.copy()
    np_prob_vector = np_prob_vector.copy()
    
    period = game_status.period[0] 
    safe_period_idx = min(period, 5) - 1

    if period < 4:
        threshold_auto_accept = 30 
    else:
        threshold_auto_accept = 50 

    # --- FUNÇÃO AUXILIAR DE VALIDAÇÃO ---
    def get_invalid_mask_hard_reject(tokens, probs, remaining_times):
        mask_is_end = np.isin(tokens, end_period_tokens)
        idx_end = np.where(mask_is_end)[0]
        
        mask_rejected = np.zeros(len(tokens), dtype = bool)

        if len(idx_end) > 0:
            rem_times_end = remaining_times[idx_end]
            probs_end = probs[idx_end] 
            
            mask_invalid_local = np.zeros(len(idx_end), dtype = bool)

            # REGRA 1: Rejeição Dura Absoluta (> 30s). Ignora a confiança da rede neural.
            mask_hard_reject = rem_times_end > 300
            mask_invalid_local[mask_hard_reject] = True

            # Alta Confiança (> 50%) ou Tempo <= Threshold continuam como False (Aceitos)
            mask_rejected[idx_end] = mask_invalid_local

        return mask_rejected

    def get_invalid_mask(tokens, probs, remaining_times):
        mask_is_end = np.isin(tokens, end_period_tokens)
        idx_end = np.where(mask_is_end)[0]
        
        mask_rejected = np.zeros(len(tokens), dtype = bool)

        if len(idx_end) > 0:
            rem_times_end = remaining_times[idx_end]
            probs_end = probs[idx_end] 
            
            mask_invalid_local = np.zeros(len(idx_end), dtype = bool)

            # REGRA 1: Rejeição Dura Absoluta (> 30s). Ignora a confiança da rede neural.
            mask_hard_reject = rem_times_end > 300
            mask_invalid_local[mask_hard_reject] = True

            # REGRA 2 & 3: Aplicadas apenas aos que passaram na Regra 1 (<= 30s) e que possuem Baixa Confiança
            mask_under_30s = ~mask_hard_reject
            mask_low_confidence = probs_end <= 0.50
            
            # Zona da Moeda Empírica (Threshold < Tempo <= 30s E Baixa Confiança)
            mask_coin = mask_under_30s & mask_low_confidence & (rem_times_end > threshold_auto_accept)
            idx_coin = np.where(mask_coin)[0]
            
            if len(idx_coin) > 0:
                eval_times = rem_times_end[idx_coin].astype(int)
                probs_sucess = matrix_end_period_probs[safe_period_idx, eval_times]
                
                confirmed_end_period = np.random.rand(len(eval_times)) <= probs_sucess
                mask_invalid_local[idx_coin] = ~confirmed_end_period

            # Alta Confiança (> 50%) ou Tempo <= Threshold continuam como False (Aceitos)
            mask_rejected[idx_end] = mask_invalid_local

        return mask_rejected
    # ------------------------------------------------

    mask_invalid_play = get_invalid_mask_hard_reject(np_token_vector, np_prob_vector, list_remaining_time)
    invalid_play_idx = np.where(mask_invalid_play)[0]

    while len(invalid_play_idx) > 0:        
        token_mask = build_logit_mask(game_status = game_status, valid_simulations = invalid_play_idx,
                                      mode = "play_loop", target = "token", period = period, output_size = 601)

        temp_play, temp_prob = get_play_prob_loop_function(
            model = model, class_model = class_model, valid_simulations = invalid_play_idx, 
            game_status = game_status, numerical_vector_ = numerical_vector_, 
            state_id_token = state_id_token, time_diff_vector_ = time_diff_vector_,
            mask_data = token_mask)

        for pos, idx in enumerate(invalid_play_idx):
            np_token_vector[idx] = temp_play[pos]
            np_prob_vector[idx]  = temp_prob[pos]

        mask_invalid_play = get_invalid_mask_hard_reject(np_token_vector, np_prob_vector, list_remaining_time)
        invalid_play_idx = np.where(mask_invalid_play)[0]


    mask_invalid_play = get_invalid_mask(np_token_vector, np_prob_vector, list_remaining_time)
    invalid_play_idx = np.where(mask_invalid_play)[0]
    
    attempts = 0

    while len(invalid_play_idx) > 0 and attempts < max_attempts:
        attempts += 1
        
        token_mask = build_logit_mask(game_status = game_status, valid_simulations = invalid_play_idx,
                                      mode = "play_loop", target = "token", period = period, output_size = 601)

        temp_play, temp_prob = get_play_prob_loop_function(
            model = model, class_model = class_model, valid_simulations = invalid_play_idx, 
            game_status = game_status, numerical_vector_ = numerical_vector_, 
            state_id_token = state_id_token, time_diff_vector_ = time_diff_vector_,
            mask_data = token_mask)

        for pos, idx in enumerate(invalid_play_idx):
            np_token_vector[idx] = temp_play[pos]
            np_prob_vector[idx]  = temp_prob[pos]

        mask_invalid_play = get_invalid_mask(np_token_vector, np_prob_vector, list_remaining_time)
        invalid_play_idx = np.where(mask_invalid_play)[0]

    return np_token_vector, np_prob_vector

def substitute_unmatched_end_period_token(np_token_vector, end_period_tokens, period):
    np_token_vector = np_token_vector.copy()
    mask_fix_last_token = np.isin(np_token_vector, end_period_tokens) 
    invalid_last_token_idx = np.where(mask_fix_last_token)[0]
    replacement_dict = {idx: fix_last_token(token = np_token_vector[idx], period = period) for idx in invalid_last_token_idx}
    for idx, new_play in replacement_dict.items():
        np_token_vector[idx] = new_play
    return np_token_vector

def get_time_loop_function(model, class_model, token_vector_, numerical_vector_, state_id_token, time_diff_vector_, mask_data = None):
    if class_model == "ExtraInfo":
        spent_time = predict_model_v2(model = model, class_model = class_model, token_vector = token_vector_, 
                                      numerical_vector = numerical_vector_, type_ = 'time', extra_info = False,
                                      mask_data = mask_data)
    elif class_model == "StateVector":
        spent_time = predict_model_v2(model = model, class_model = class_model, token_vector = token_vector_, 
                                      numerical_vector = numerical_vector_, state_token_vector = state_id_token,
                                      type_ = 'time', extra_info = False, mask_data = mask_data)
    elif class_model in ["TimeDiffTower", "MultiTower"]:
        # Agrupados! Passamos separado e a predict_model_v2 cuida de empacotar para o PyTorch.
        spent_time = predict_model_v2(model = model, class_model = class_model, token_vector = token_vector_, 
                                      numerical_vector = numerical_vector_, time_diff_features = time_diff_vector_,
                                      type_ = 'time', extra_info = False, mask_data = mask_data)
    else:
        print("get a valid class model name")
        return None
    return spent_time    

def create_temp_vectors(period, class_model, np_token_vector, game_status, idx_list, config_setvariables, match_vector, dict_tokens):
    
    temp_token_vector, temp_numerical_vector = update_vectors(
        period = period, random_play = np_token_vector, tokens_vector = game_status.token_vector.copy(), 
        home_points = game_status.home_points.copy(), away_points = game_status.away_points.copy(), 
        limhtimeout = game_status.limhtimeout.copy(), limvtimeout = game_status.limvtimeout.copy(), 
        remaining_time = game_status.remaining_time.copy(), ball_possession = game_status.ball_possession.copy(),
        indices = idx_list, config = config_setvariables)
    
    if class_model in ["ExtraInfo", "StateVector"] and config_setvariables == 0:
        config_setvariables = 1
    if class_model in ["TimeDiffTower", "MultiTower"] and config_setvariables == 0:
        config_setvariables = 3
    
    if class_model in ["ExtraInfo", "StateVector"]:
        if match_vector is None:
            numerical_vector_ = temp_numerical_vector
        else:
            numerical_vector_ = np.hstack([temp_numerical_vector, match_vector[idx_list]])

    if class_model in ["TimeDiffTower", "MultiTower"]:
        if config_setvariables == 3:
            num_numerical_variables = 15
        if match_vector is None:
            numerical_vector_ = temp_numerical_vector[:, :num_numerical_variables]
        else:
            numerical_vector_ = np.hstack([temp_numerical_vector[:, :num_numerical_variables], match_vector[idx_list]])

    if class_model in ["TimeDiffTower", "MultiTower"]:
        temp_time_diff_vector_ = temp_numerical_vector[:, num_numerical_variables:]
    else:
        temp_time_diff_vector_ = None

    if class_model == "StateVector":
        # --- get the points difference after sorted play (temp) due to calculate temp_state_id_token
        # here we repeat the minimum logic code to update point vectors
        idxr = np.asarray(idx_list)
        rp_sub = np_token_vector[idxr]

        temp_home_points = game_status.home_points[idxr].copy()
        temp_away_points = game_status.away_points[idxr].copy()

        temp_home_points[np.isin(rp_sub, dict_tokens['home_pts1'])] += 1
        temp_home_points[np.isin(rp_sub, dict_tokens['home_pts2'])] += 2
        temp_home_points[np.isin(rp_sub, dict_tokens['home_pts3'])] += 3

        temp_away_points[np.isin(rp_sub, dict_tokens['away_pts1'])] += 1
        temp_away_points[np.isin(rp_sub, dict_tokens['away_pts2'])] += 2
        temp_away_points[np.isin(rp_sub, dict_tokens['away_pts3'])] += 3

        temp_state_id_token = categorize_time_bins_np(
            period_arr = np.full(shape = idxr.shape[0], fill_value = period, dtype = int),
            remaining_time_arr = game_status.remaining_time[idxr].astype(int),
            diff_points_arr = (temp_home_points - temp_away_points).astype(int))
    else:
        temp_state_id_token = None

    return temp_token_vector, numerical_vector_, temp_state_id_token, temp_time_diff_vector_

# essa funçao vai corrigir quando uma jogada foi sorteada, mas o tempo gasto por ela sorteado é maior que o tempo restante no periodo. 
# Ai precisa ir la e decidir o que fazer com essa jogada. há duas opçoes:
# 1. forçar um tempo gasto valido para essa jogada.
# 2. descartar a jogada e forçar o sorteio de um dos tokens de encerramento de periodo.
# Para a opcao 1, esta elegivel so aqueles casos que o tempo gasto é no maximo 1 segundos maior que o tempo restante. Ai tenta por duas vezes sortear um tempo gasto valido
# Se nao der certo na opcao 1, ou ele nao estiver elegivel para a opcao 2, vai depender do tipo de token para decidir se entra ou nao
def substitute_spent_time_out_of_context(random_time, list_remaining_time, model, class_model, token_vector_, numerical_vector_, state_id_token, time_diff_vector_):
    random_time = np.asarray(random_time).copy()
    list_remaining_time = np.asarray(list_remaining_time)

    invalid_time_flag = 0
    while invalid_time_flag < 1:
        #idx_invalid = np.where(random_time > list_remaining_time)[0]
        idx_invalid = np.where( (random_time > list_remaining_time) & (random_time <= (list_remaining_time + 10)) )[0]
        if idx_invalid.size == 0:
            break

        tv = token_vector_[idx_invalid]
        nv = numerical_vector_[idx_invalid]
        sid = state_id_token[idx_invalid] if state_id_token is not None else None
        tdv = time_diff_vector_[idx_invalid] if time_diff_vector_ is not None else None

        # Only sort a new spent time for that ones where the first spent time is greater than remaining time
        temp_spent_time = get_time_loop_function(model = model, class_model = class_model, 
                                                 token_vector_ = tv, numerical_vector_ = nv, 
                                                 state_id_token = sid, time_diff_vector_ = tdv)
        
 
        random_time[idx_invalid] = temp_spent_time
        invalid_time_flag += 1

    return random_time

# Se eu nao consegui encaixar a jogada na funcao anterior, aqui a depender de qual jogada é, e se o tempo gasto for no maximo 1 segundos maior que o tempo restante, entao eu permito que 
# a jogada seja aceita no historico. Se nao for elegivel, entao a jogada vai ser descartada.
def update_keep_generating_vector(keep_generating, np_token_vector, list_remaining_time, end_period_tokens, missed_shot_tokens):
    keep_generating = keep_generating.copy()
    
    # sorteou o rebote no tempo critico e agora volta para 1 para seguir o jogo
    mask_was_4 = (keep_generating == 4)
    keep_generating[mask_was_4] = 1

    # keep_generating = 2: O fluxo normal sorteou um token oficial de encerramento de período.
    # O jogo encerra graciosamente para essas simulações.
    mask_kp = (keep_generating == 1) & np.isin(np_token_vector, end_period_tokens)
    keep_generating[mask_kp] = 2

    # 3. Arremesso Errado na ZONA CRÍTICA (Status 4)
    # Condição: status atual é 1 E tempo restante < 30s E token é missed shot
    mask_missed_crit = (keep_generating == 1) & (list_remaining_time < 300) & np.isin(np_token_vector, missed_shot_tokens)
    keep_generating[mask_missed_crit] = 4    

    return keep_generating

def resolve_critical_zone_plays(crit_idx, proposed_tokens_global, proposed_probs_global, game_status, 
                                list_remaining_time, token_model, time_model, class_model, 
                                numerical_vector_, state_id_token, time_diff_vector_, 
                                matrix_end_period_probs, end_period_tokens, config_setvariables, 
                                match_vector, dict_tokens, max_attempts = 3):
        
    # Vetor local (tamanho global) para armazenar os tempos definitivos das simulações críticas.
    # Iniciamos com -1. Ao final da função, todos os índices em crit_idx terão um tempo válido.
    final_times_global = np.full(game_status.n_sims, -1, dtype = int)
    
    # active_idx controla quem ainda está preso no interrogatório do Rejection Sampling
    active_idx = crit_idx.copy()
    
    # O período é o mesmo para todo o batch nesta etapa (seguro para a matriz)
    period = game_status.period[0]
    safe_period_idx = min(period, 5) - 1
    
    # Contador de segurança contra loops infinitos
    attempts = 0

    # O laço roda enquanto houver simulações pendentes E não estourar o limite de tentativas
    while len(active_idx) > 0 and attempts < max_attempts:
        attempts += 1
        
        # ---------------------------------------------------------
        # PASSO 0: Triagem de Fim de Período Nativos
        # ---------------------------------------------------------
        curr_tokens = proposed_tokens_global[active_idx]
        mask_is_end = np.isin(curr_tokens, end_period_tokens)
        
        idx_end_period = active_idx[mask_is_end]
        if len(idx_end_period) > 0:
            # Aprovado imediatamente! O tempo gasto é o que sobrou.
            final_times_global[idx_end_period] = list_remaining_time[idx_end_period]
            
        # Remove os aprovados da lista ativa
        active_idx = active_idx[~mask_is_end]
        
        if len(active_idx) == 0:
            break # Todos foram aprovados nesta rodada!
            
        # ---------------------------------------------------------
        # PASSO 0.5: Bypass de Dependência Física (Status 4)
        # ---------------------------------------------------------
        curr_kp_vector = game_status.keep_generating[active_idx]
        mask_is_dependent = (curr_kp_vector == 4)
        idx_dependent = active_idx[mask_is_dependent]
        
        if len(idx_dependent) > 0:
            # APROVAÇÃO AUTOMÁTICA: O rebote TEM que caber.
            # Sorteamos o tempo com a máscara de limite do relógio (hard constraint)
            time_mask_dep = build_logit_mask(game_status = game_status, valid_simulations = idx_dependent, 
                                             mode = "play_loop", target = "time", period = period, output_size = 301)
            
            t_tok, t_num, t_sta, t_tdiff = create_temp_vectors(
                period = period, class_model = class_model, np_token_vector = proposed_tokens_global, 
                game_status = game_status, idx_list = idx_dependent, config_setvariables = config_setvariables, 
                match_vector = match_vector, dict_tokens = dict_tokens)
            
            sorted_times_dep = get_time_loop_function(
                model = time_model, class_model = class_model, token_vector_ = t_tok, numerical_vector_ = t_num, 
                state_id_token = t_sta, time_diff_vector_ = t_tdiff, mask_data = time_mask_dep)
            
            final_times_global[idx_dependent] = sorted_times_dep
            
            # Removemos da lista ativa para não entrar no Rejection Sampling
            active_idx = active_idx[~mask_is_dependent]
            
        if len(active_idx) == 0:
            break

        # ---------------------------------------------------------
        # PASSO 1: SORTEIO 1 (A FÍSICA) - A jogada cabe no relógio?
        # ---------------------------------------------------------
        # Prepara os vetores temporários APENAS para os ativos
        temp_token, temp_num, temp_state, temp_time_diff = create_temp_vectors(
            period = period, class_model = class_model, np_token_vector = proposed_tokens_global, 
            game_status = game_status, idx_list = active_idx, config_setvariables = config_setvariables, 
            match_vector = match_vector, dict_tokens = dict_tokens)
        
        rem_times_ativos = list_remaining_time[active_idx]
        
        # Consulta a CDF empírica/modelo para pegar P(T_gasto <= T_restante)
        prob_fits_physics = get_cdf_time_model(
            time_model = time_model, class_model = class_model, token_vector = temp_token, 
            numerical_vector = temp_num, state_token_vector = temp_state, time_diff_features = temp_time_diff, 
            remaining_time = rem_times_ativos)
            
        # Experimento Bernoulli 1: a jogada sorteada cabe no tempo restante?
        coin1_accept = np.random.rand(len(active_idx)) <= prob_fits_physics
        
        idx_fits_physics = active_idx[coin1_accept]
        idx_fails_physics = active_idx[~coin1_accept]

        # ---------------------------------------------------------
        # PASSO 2: SUCESSO no experimento 1 (Sorteia o Tempo Oficial)
        # ---------------------------------------------------------
        if len(idx_fits_physics) > 0:
            # A jogada cabe. Constrói máscara dura e sorteia o tempo real.
            time_mask = build_logit_mask(game_status = game_status, valid_simulations = idx_fits_physics, 
                                         mode = "play_loop", target = "time", period = period, output_size = 301)
            
            t_tok, t_num, t_sta, t_tdiff = create_temp_vectors(
                period = period, class_model = class_model, np_token_vector = proposed_tokens_global, 
                game_status = game_status, idx_list = idx_fits_physics, config_setvariables = config_setvariables, 
                match_vector = match_vector, dict_tokens = dict_tokens)
            
            sorted_times = get_time_loop_function(
                model = time_model, class_model = class_model, token_vector_ = t_tok, numerical_vector_ = t_num, 
                state_id_token = t_sta, time_diff_vector_ = t_tdiff, mask_data = time_mask)
            
            # Salva o tempo definitivo e essas simulações não entram mais no active_idx
            final_times_global[idx_fits_physics] = sorted_times

        # ---------------------------------------------------------
        # PASSO 3: Experimento 2 (O ÁRBITRO) e REJEIÇÃO TOTAL
        # ---------------------------------------------------------
        idx_reject = np.array([], dtype = int) 
        
        if len(idx_fails_physics) > 0:
            rem_times_fails = list_remaining_time[idx_fails_physics].astype(int)
            
            # Busca probabilidade empírica de encerramento na matriz (Árbitro)
            prob_ends = matrix_end_period_probs[safe_period_idx, rem_times_fails]
            
            # faz o experimento bernoulli 2
            coin2_ends = np.random.rand(len(idx_fails_physics)) <= prob_ends
            
            idx_force_end = idx_fails_physics[coin2_ends]
            idx_reject = idx_fails_physics[~coin2_ends]
            
            # 3.A: Árbitro Encerrou o Jogo (Fim Forçado Válido)
            if len(idx_force_end) > 0:
                proposed_tokens_global[idx_force_end] = -1 # FLAG: Sorteie um end_period depois! 
                final_times_global[idx_force_end] = list_remaining_time[idx_force_end]

            # 3.B: Alucinação (Rejeição Total) -> Precisa de novo token!
            if len(idx_reject) > 0 and attempts < max_attempts:
                # Só resorteia se ainda tivermos tentativas sobrando
                token_mask_retry = build_logit_mask(game_status = game_status, valid_simulations = idx_reject,
                                                    mode = "play_loop", target = "token", period = period, output_size = 601)
                
                new_plays, new_probs = get_play_prob_loop_function(
                    model = token_model, class_model = class_model, valid_simulations = idx_reject, 
                    game_status = game_status, numerical_vector_ = numerical_vector_, 
                    state_id_token = state_id_token, time_diff_vector_ = time_diff_vector_, 
                    mask_data = token_mask_retry)
                
                # Sobrescreve a proposta global ruim com a nova
                proposed_tokens_global[idx_reject] = new_plays
                proposed_probs_global[idx_reject]  = new_probs

        # Atualiza a lista ativa apenas com quem falhou nas duas moedas
        active_idx = idx_reject

    # ---------------------------------------------------------
    # PASSO 4: REGRA DE SEGURANÇA (Estourou o limite de tentativas)
    # ---------------------------------------------------------
    if len(active_idx) > 0:
        # O modelo teimou 3 vezes em dar jogadas impossíveis. 
        # Forçamos o fim do período para proteger o cronômetro.
        proposed_tokens_global[active_idx] = -1 # FLAG: Sorteie um end_period depois!
        final_times_global[active_idx] = list_remaining_time[active_idx]

    # Retorna APENAS as fatias correspondentes aos índices que entraram (crit_idx)
    return proposed_tokens_global[crit_idx], proposed_probs_global[crit_idx], final_times_global[crit_idx]

def get_random_play_loop(period, game_status, valid_simulations, token_model, time_model, class_model, matrix_end_period_probs, config_setvariables = 0, match_vector = None):

    # =========================================================
    # SETUP INICIAL: Dicionários e Tokens Especiais
    # =========================================================
    dict_tokens = define_special_tokens(1)
    end_period_tokens = dict_tokens["end_period"]    
    missed_shot_tokens = dict_tokens["missed_shots"]    

    # =========================================================
    # PREPARAÇÃO DOS DADOS (A Mágica da Centralização)
    # =========================================================
    # Chamamos a nossa nova função unificadora. Ela já devolve os vetores
    # numéricos, de estado e de tempo perfeitamente fatiados para as valid_simulations!

    config_setvariables, numerical_vector_, time_diff_vector_, state_id_token = prepare_model_inputs(
        period = period, game_status = game_status, class_model = class_model, config_setvariables = config_setvariables, match_vector = match_vector)

    # O token vector também precisa ser fatiado para acompanhar
    token_vector_ = game_status.token_vector[valid_simulations]

    ## saving the current remaining_time_value
    list_remaining_time = game_status.remaining_time.copy()

    # Iniciando os vetores finais globais (tamanho total de n_sims)
    # Eles vão guardar os resultados desta iteração para posterior atualização no game_status
    random_token_np = np.zeros(shape = game_status.n_sims, dtype = int)
    prob_token_np   = np.zeros(shape = game_status.n_sims)
    random_time_np  = np.full(shape = game_status.n_sims, fill_value = -1, dtype = int)

    # ---------------------------------------------------------
    # Vetores Provisórios (Globais)
    # ---------------------------------------------------------
    # Eles têm o mesmo tamanho global, o que impede os bugs de dimensão
    # e protege as variáveis definitivas acima.
    proposed_token_np = np.zeros(shape = game_status.n_sims, dtype = int)
    proposed_prob_np  = np.zeros(shape = game_status.n_sims)

    # =========================================================
    # A NOVA ARQUITETURA: Divisão do Batch (Física vs Tática)
    # =========================================================
    # Definindo a fronteira onde o Árbitro do Relógio começa a atuar
    # 30 segundos = 300 décimo
    CRITICAL_TIME = 300 
    
    # Salva APENAS o tempo restante das simulações que ainda estão ativas
    list_remaining_time_valid_simulations = list_remaining_time[valid_simulations]
    
    # safe_mask terá tamanho 45
    safe_mask = list_remaining_time_valid_simulations > CRITICAL_TIME
    crit_mask = ~safe_mask
    
    # Mapeia as máscaras de volta para os índices globais originais
    safe_idx = valid_simulations[safe_mask]
    crit_idx = valid_simulations[crit_mask]

    ########## ok, let's start to sort plays ##########
    # This is the most complex function, because we need: 
    # 1. sort a token, 
    # 2. check if that token is allowed in that game context
    # 3. if we sorted a end period token, we need to check if the token matches period sorted
    # 4. sort a spent time
    # 5. check if spent time is allowed in that game context (sorted spent time <= remaining_time)
    # 6. update keep generating vector
    # 7. update game status variables  
    # =========================================================
    # STAGE 1: SORT PLAY TOKEN (play_i)  [use all the variables S_{i-1}]
    # =========================================================
    token_mask = build_logit_mask(game_status = game_status, valid_simulations = valid_simulations,
                                  mode = "play_loop", target = "token", period = period, output_size = 601)

    plays, probs = get_play_prob_loop_function(
        model = token_model, class_model = class_model, valid_simulations = valid_simulations, game_status = game_status, 
        numerical_vector_ = numerical_vector_, state_id_token = state_id_token, time_diff_vector_ = time_diff_vector_, 
        mask_data = token_mask)

    # Salva APENAS no rascunho global! (Resolve o bug de dimensão e protege a arquitetura)
    proposed_token_np[valid_simulations] = plays
    proposed_prob_np[valid_simulations]  = probs

    # FIXING 1: end period token sorted but we have a long remaining time.
    # description: we got an end period token, but it doesn't match game context, because we have a lot of remaining time to play.
    # current rule: if we have more than 15 seconds to play and sort any of end period tokens, we need to sub in that token.
    proposed_token_np, proposed_prob_np = substitute_end_period_token_out_of_context(
        np_token_vector = proposed_token_np, np_prob_vector = proposed_prob_np, list_remaining_time = list_remaining_time, 
        model = token_model, class_model = class_model, game_status = game_status, numerical_vector_ = numerical_vector_, 
        state_id_token = state_id_token, time_diff_vector_ = time_diff_vector_, end_period_tokens = end_period_tokens,
        matrix_end_period_probs = matrix_end_period_probs)

    # =========================================================
    # STAGE 2: SORT SPENT_TIME TOKEN (play_i)  [use all the variables S_{i-1}]
    # =========================================================
    # FIRST STEP: split the simulations into two groups:
    # 1. The ones we don't need to sort spend time because the sorted play ends the period generation
    # 2. The ones that we need to sort spend time.
    ## FIRST CASE (todos os casos que rando_token_np foi um dos tokens de encerramento de periodo):
    mask_remaining_time_zero = np.isin(proposed_token_np, end_period_tokens)
    idx_zero_remaining_time = np.where(mask_remaining_time_zero)[0]
    idx_zero_remaining_time = idx_zero_remaining_time[np.isin(idx_zero_remaining_time, valid_simulations)]

    if len(idx_zero_remaining_time) > 0:
        # Oficializa TUDO agora!
        random_token_np[idx_zero_remaining_time] = proposed_token_np[idx_zero_remaining_time]
        prob_token_np[idx_zero_remaining_time]   = proposed_prob_np[idx_zero_remaining_time]
        random_time_np[idx_zero_remaining_time]  = list_remaining_time[idx_zero_remaining_time]

    ## SECOND CASE: Jogadas normais que precisam de sorteio de tempo
    idx_random_remaining_time = np.where(~mask_remaining_time_zero)[0]
    idx_random_remaining_time = idx_random_remaining_time[np.isin(idx_random_remaining_time, valid_simulations)]

    # ---------------------------------------------------------
    # A BIFURCAÇÃO DA NOVA ARQUITETURA ACONTECE AQUI!
    # ---------------------------------------------------------
    # 1. Quem precisa sortear tempo E está na Zona Segura?
    safe_idx_random = np.intersect1d(idx_random_remaining_time, safe_idx)
    
    # 2. Quem precisa sortear tempo E está na Zona Crítica?
    crit_idx_random = np.intersect1d(idx_random_remaining_time, crit_idx)

    # =========================================================
    # CAMINHO DA ZONA SEGURA (O seu código original entra aqui intacto!)
    # =========================================================
    if len(safe_idx_random) > 0:
        # Oficializa os TOKENS agora! 
        # É obrigatório para o create_temp_vectors enxergar as jogadas corretas.
        random_token_np[safe_idx_random] = proposed_token_np[safe_idx_random]
        prob_token_np[safe_idx_random]   = proposed_prob_np[safe_idx_random]

        # O create_temp_vectors agora vai ler as jogadas reais recém-oficializadas
        temp_token_vector, temp_numerical_vector, temp_state_id_token, temp_time_diff_vector_ = create_temp_vectors(
            period = period, class_model = class_model, np_token_vector = random_token_np, game_status = game_status, 
            idx_list = safe_idx_random, config_setvariables = config_setvariables, match_vector = match_vector, dict_tokens = dict_tokens)
        
        sorted_spent_time = get_time_loop_function(
            model = time_model, class_model = class_model, token_vector_ = temp_token_vector, 
            numerical_vector_ = temp_numerical_vector, state_id_token = temp_state_id_token, 
            time_diff_vector_ = temp_time_diff_vector_)

        # Oficializa o TEMPO
        random_time_np[safe_idx_random] = sorted_spent_time    

    # =========================================================
    # CAMINHO DA ZONA CRÍTICA (AQUI ENTRARÁ O QUE O ANTIGO FIXING 3 FAZIA)
    # =========================================================
    if len(crit_idx_random) > 0:
        # Aqui o Token sorteado no Stage 1 vai ser submetido ào sorteio 1 (Física) e sorteio 2 (Árbitro).
        # Se for rejeitado, teremos um WHILE LOOP aqui dentro que vai RESORTEAR o token e testar de novo!
        # 1. Envia os rascunhos para o Rejection Sampling (com a trava de 3 tentativas)
        final_tokens, final_probs, final_times = resolve_critical_zone_plays(
            crit_idx = crit_idx_random, proposed_tokens_global = proposed_token_np, proposed_probs_global = proposed_prob_np,   
            game_status = game_status, list_remaining_time = list_remaining_time, token_model = token_model, time_model = time_model,
            class_model = class_model, numerical_vector_ = numerical_vector_, state_id_token = state_id_token, time_diff_vector_ = time_diff_vector_,
            matrix_end_period_probs = matrix_end_period_probs, end_period_tokens = end_period_tokens, config_setvariables = config_setvariables,
            match_vector = match_vector, dict_tokens = dict_tokens, max_attempts = 3)

        # ---------------------------------------------------------
        # 2. Tratamento da flag "-1" com Sorteio Contextual Tático
        # ---------------------------------------------------------
        mask_needs_end_token = final_tokens == -1
        idx_needs_end_global = crit_idx_random[mask_needs_end_token]

        if len(idx_needs_end_global) > 0:
            
            # --- VETORES GLOBAIS (Fatiamento direto, sem if/else para MultiTower!) ---
            num_vec_subset = numerical_vector_[idx_needs_end_global]
            state_id_subset  = state_id_token[idx_needs_end_global] if state_id_token is not None else None
            time_diff_subset = time_diff_vector_[idx_needs_end_global] if time_diff_vector_ is not None else None

            # --- VETOR LOCAL (Requer o mapeamento com máscara) ---
            local_mask = np.isin(valid_simulations, idx_needs_end_global)
            token_vec_subset = token_vector_[local_mask]

            if period == 1:
                token_list = dict_tokens["end_period1"]
            elif period == 2:
                token_list = dict_tokens["end_period2"]
            elif period == 3:
                token_list = dict_tokens["end_period3"]
            elif period == 4:
                token_list = dict_tokens["end_period4"]
            else:
                token_list = dict_tokens["end_overtime"]

            # Sorteio restrito aos tokens de encerramento
            drawn_end_tokens, drawn_probs = predict_model_v2(
                model = token_model, class_model = class_model, token_vector = token_vec_subset, numerical_vector = num_vec_subset, 
                state_token_vector = state_id_subset, time_diff_features = time_diff_subset, only_consider = token_list)

            # Substituição final no vetor da Zona Crítica
            final_tokens[mask_needs_end_token] = drawn_end_tokens
            final_probs[mask_needs_end_token]  = drawn_probs

        
        # 3. O CARIMBO OFICIAL: Salvando as decisões nos vetores definitivos
        random_token_np[crit_idx_random] = final_tokens
        prob_token_np[crit_idx_random]   = final_probs
        random_time_np[crit_idx_random]  = final_times

    # =========================================================
    # STAGE 3: CHECK WHERE WE UPDATE THE GAME STATUS AND DOING IT
    # =========================================================
    kp_vector = update_keep_generating_vector(keep_generating = game_status.keep_generating, np_token_vector = random_token_np, 
                                              list_remaining_time = list_remaining_time, end_period_tokens = end_period_tokens,
                                              missed_shot_tokens = missed_shot_tokens)
    
    game_status.keep_generating = kp_vector

    # index to update the game status with the sorted play and spent time
    idx_update_game_status = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    # FINALLY, updating the game status, only for simulations where we got valid pair of play and spent time
    update_game_status(game_status = game_status, random_play = random_token_np[idx_update_game_status], 
                       period = period, indices = idx_update_game_status, inplace = True, config = config_setvariables)

    game_status.num_plays[idx_update_game_status] += 1
    game_status.sorted_token[idx_update_game_status] = random_token_np[idx_update_game_status]
    game_status.prob_token[idx_update_game_status]   = prob_token_np[idx_update_game_status]
    game_status.sorted_time[idx_update_game_status]  = random_time_np[idx_update_game_status]

    new_remaining_time = list_remaining_time[idx_update_game_status] - random_time_np[idx_update_game_status]
    game_status.remaining_time[idx_update_game_status] = new_remaining_time

    return

def get_random_last_play(period, game_status, valid_simulations, token_model, class_model, config_setvariables = 0, match_vector = None):
    # Just some sets of the vectors depending of the class model
    dict_tokens = define_special_tokens(1)

    token_vector = game_status.token_vector[valid_simulations]
    numerical_vector = game_status.numerical_vector[valid_simulations]

    if class_model in ["ExtraInfo", "StateVector"] and config_setvariables == 0:
        config_setvariables = 1
    if class_model in ["TimeDiffTower", "MultiTower"] and config_setvariables == 0:
        config_setvariables = 3
    
    if class_model in ["ExtraInfo", "StateVector"]:
        if match_vector is None:
            numerical_vector_ = numerical_vector
        else:
            numerical_vector_ = np.hstack([numerical_vector, match_vector[valid_simulations]])

    if class_model in ["TimeDiffTower", "MultiTower"]:
        if config_setvariables == 3:
            num_numerical_variables = 15
        if match_vector is None:
            numerical_vector_ = numerical_vector[:, :num_numerical_variables]
        else:
            numerical_vector_ = np.hstack([numerical_vector[:, :num_numerical_variables], match_vector[valid_simulations]])

    if class_model == "StateVector":
        state_id_token = categorize_time_bins_np(
            period_arr = np.full(shape = game_status.remaining_time[valid_simulations].shape[0], fill_value = period, dtype = int),
            remaining_time_arr = game_status.remaining_time[valid_simulations].astype(int),
            diff_points_arr = (game_status.home_points[valid_simulations] - game_status.away_points[valid_simulations]).astype(int))
                    
    else:
        state_id_token = None

    if class_model in ["TimeDiffTower", "MultiTower"]:
        time_diff_vector_ = numerical_vector[:, num_numerical_variables:]
    else:
        time_diff_vector_ = None    

    # ########## ok, let's start to sort plays ##########
    # # we only give some token options for the period end token
    # if period == 1: 
    #     tokens_list = dict_tokens["end_period1"]
    # elif period == 2: 
    #     tokens_list = dict_tokens["end_period2"]
    # elif period == 3: 
    #     tokens_list = dict_tokens["end_period3"]
    # elif period == 4: 
    #     tokens_list = dict_tokens["end_period4"]
    # else: 
    #     tokens_list = dict_tokens["end_overtime"]

    token_mask = build_logit_mask(game_status = game_status, valid_simulations = valid_simulations,
                                  mode = "last_play", target = "token", period = period, output_size = 601)

    if class_model != "MultiTower":
        play, prob = predict_model_v2(
            model = token_model, class_model = class_model, token_vector = token_vector, 
            numerical_vector = numerical_vector_, state_token_vector = state_id_token, 
            time_diff_features = time_diff_vector_)
    else:
        play, prob = predict_model_v2(
            model = token_model, class_model = class_model, token_vector = token_vector, 
            numerical_vector = [numerical_vector_, time_diff_vector_])
        
    ########## update the game_status with the sorted play ##########
    game_status.sorted_time[valid_simulations] = game_status.remaining_time[valid_simulations]
    game_status.prob_time[valid_simulations] = 1
    game_status.keep_generating[valid_simulations] = 0    
    game_status.remaining_time[valid_simulations] = 0

    update_game_status(game_status = game_status, random_play = play, period = period, 
                       indices = valid_simulations, inplace = True, config = config_setvariables)

    game_status.num_plays[valid_simulations] += 1
    game_status.sorted_token[valid_simulations] = np.asarray(play)
    game_status.prob_token[valid_simulations] = np.asarray(prob)

    return

def get_random_instant_replay(period, game_status, valid_simulations, token_model, class_model, config_setvariables = 0, match_vector = None):
    
    # Just some sets of the vectors depending of the class model
    dict_tokens = define_special_tokens(1)

    token_vector = game_status.token_vector[valid_simulations]
    numerical_vector = game_status.numerical_vector[valid_simulations]

    if class_model in ["ExtraInfo", "StateVector"] and config_setvariables == 0:
        config_setvariables = 1
    if class_model in ["TimeDiffTower", "MultiTower"] and config_setvariables == 0:
        config_setvariables = 3
    
    if class_model in ["ExtraInfo", "StateVector"]:
        if match_vector is None:
            numerical_vector_ = numerical_vector
        else:
            numerical_vector_ = np.hstack([numerical_vector, match_vector[valid_simulations]])

    if class_model in ["TimeDiffTower", "MultiTower"]:
        if config_setvariables == 3:
            num_numerical_variables = 15
        if match_vector is None:
            numerical_vector_ = numerical_vector[:, :num_numerical_variables]
        else:
            numerical_vector_ = np.hstack([numerical_vector[:, :num_numerical_variables], match_vector[valid_simulations]])

    if class_model == "StateVector":
        state_id_token = categorize_time_bins_np(
            period_arr = np.full(shape = game_status.remaining_time[valid_simulations].shape[0], fill_value = period, dtype = int),
            remaining_time_arr = game_status.remaining_time[valid_simulations].astype(int),
            diff_points_arr = (game_status.home_points[valid_simulations] - game_status.away_points[valid_simulations]).astype(int))
                    
    else:
        state_id_token = None

    if class_model in ["TimeDiffTower", "MultiTower"]:
        time_diff_vector_ = numerical_vector[:, num_numerical_variables:]
    else:
        time_diff_vector_ = None    

    ########## ok, let's check if we are going to have a instant replay token ##########
    if class_model != "MultiTower":
        play, prob = predict_model_v2(
            model = token_model, class_model = class_model, token_vector = token_vector, 
            numerical_vector = numerical_vector_, state_token_vector = state_id_token, 
            time_diff_features = time_diff_vector_)
    else:
        play, prob = predict_model_v2(
            model = token_model, class_model = class_model, token_vector = token_vector, 
            numerical_vector = [numerical_vector_, time_diff_vector_])

    ########## update the game_status with the sorted play ##########
    mask_instant_replay = np.isin(np.asarray(play), dict_tokens["instant_replay"])
    idx_instant_replay = valid_simulations[mask_instant_replay]
    play_arr = np.asarray(play)[mask_instant_replay]
    prob_arr = np.asarray(prob)[mask_instant_replay]

    update_game_status(game_status = game_status, random_play = play_arr, period = period, 
                       indices = idx_instant_replay, inplace = True, config = config_setvariables)
    
    game_status.sorted_time[idx_instant_replay] = 0
    game_status.prob_time[idx_instant_replay] = 1

    game_status.num_plays[idx_instant_replay] += 1
    game_status.sorted_token[idx_instant_replay] = play_arr
    game_status.prob_token[idx_instant_replay] = prob_arr
    game_status.keep_generating[idx_instant_replay] = 6

    return idx_instant_replay

def generate_period_plays_v2(period, class_model, token_model1, time_model1, game_status, game_log, matrix_end_period_probs, token_model2 = None, time_model2 = None, config_setvariables = 0, match_vector = None):
    # Set keep generating vector as 1 (continue) for all simulations
    game_status.keep_generating[:] = 1

    # If we are simulating plays for overtime, then only tied games stay with keep generating vector valid
    if period > 4:
        idx_finished_games = np.where(game_status.home_points != game_status.away_points)[0]
        idx_unfinished_games = np.where(game_status.home_points == game_status.away_points)[0]
        game_status.keep_generating[idx_finished_games] = 0
    
    print_status(game_status, period)
    valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    get_random_first_play(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                          token_model = token_model1, class_model = class_model, config_setvariables = config_setvariables, 
                          match_vector = match_vector)

    # logica do while loop aqui:
    ## 1. atualiza o game log com o atual game status SOMENTE para as simulacoes validas
    ## 2. salva informacão atual de quantidade de faltas
    ## 3. sorteia uma jogada (play_token, spent_time)
    ## 3. atualiza o game status
    ## 4. atualiza o limite de faltas para os 2 minutos finais se necessario
    ## 5. printa o resultado (apenas estatisticas descritivas de pontuacao das equipes e percentual de tempo)
    ## 6. assinala as simulacoes com status especiais (keep_generating em [2, 3, 5])
    ## 7. salva no gamelog as simulacoes com status especiais
    ## 8. keep_generating igual a 0 para as simulacoes com keep_generating 2
    ## 9. keep_generating igual a 6 para as simulacoes com keep_generating igual a 3 ou 5
    ##10. atualizo a lista de simulacoes validas
    while len(valid_simulations) > 0:
        if token_model2 is not None and time_model2 is not None:
            # 1a.If we are in the last two minutes of 4th period or overtime, Change the keep generating status for 7 and update the valid_simulationa
            # We do that for the option to change for a specific final moments model
            mask_two_final_minutes_game = (game_status.remaining_time <= 1200) & (game_status.period >= 4) & (game_status.keep_generating == 1)
            idx_two_final_minutes_game = np.where(mask_two_final_minutes_game)[0]
            game_status.keep_generating[idx_two_final_minutes_game] = 7

            # 1b. Update the valid_simulations:
            valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

        clear_output(wait = True)
        # 1. Only for simulations with keep generating status in [1, 4], we update the game log with the current game_status.
        # That is, the sorted events are in the game history 
        game_log.add_events(indices = valid_simulations, game_status = game_status)

        # 2. save the fouls information, this will be necessary if when we need to change the fouls bonus limit at period last 2 minutes .
        list_limhteam = game_status.limhteam.copy()
        list_limvteam = game_status.limvteam.copy()
        list_limhtimeout = game_status.limhtimeout.copy()
        list_limvtimeout = game_status.limvtimeout.copy()

        # 3. In get_random_play_loop we sorted the event (play_token and spent_time) and updated the game_status
        # Only for simulations with keep_generating status at first in [1, 4].
        get_random_play_loop(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                             token_model = token_model1, time_model = time_model1, class_model = class_model, 
                             config_setvariables = config_setvariables, match_vector = match_vector, 
                             matrix_end_period_probs = matrix_end_period_probs)
        
        # Note: The keep generating status could have change depending of the event sorted in get_random_play_loop. 
        # (old keep_generating > new keep_generating): (1 > 4), (1 > 2), (4 > 1) 

        # 4. Checking how much we have in remaining time and change the fouls bonus limit if necessary
        # Maybe I have to change here.
        if period == 4:
            change_timeouts_limits(game_status = game_status, valid_simulations = valid_simulations, 
                                  list_limhtimeout = list_limhtimeout, list_limvtimeout = list_limhtimeout)

        change_fouls_limit_two_final_minutes(game_status = game_status, valid_simulations = valid_simulations, 
                                             list_limhteam = list_limhteam, list_limvteam = list_limvteam)
        
        # 5. Print stats about teams' points and spent time percentual.
        print_status(game_status = game_status, period = period)

        # 6. Assign the simulations which reach a critical status, about to end the event generation process.
        idx_about_to_end_simulations = np.where(game_status.keep_generating == 2)[0]
        
        # 7. Update game log with current game status, only for those simulations who reach about_to_end status.
        if len(idx_about_to_end_simulations) > 0:
            game_log.add_events(indices = idx_about_to_end_simulations, game_status = game_status)

        # 8. Change the keep ganerating status for that ones with status 2. These ones we already generated one of end up event naturally.
        mask_kp = (game_status.keep_generating == 2)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 0

        # 10. Update the valid simulations
        valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    if token_model2 is not None and time_model2 is not None:
        valid_simulations = np.where((game_status.keep_generating == 7))[0]
        game_status.keep_generating[valid_simulations] = 1

    while len(valid_simulations) > 0:
        clear_output(wait = True)   
        # 1. Only for simulations with keep generating status in [1, 4], we update the game log with the current game_status.
        # That is, the sorted events are in the game history 
        game_log.add_events(indices = valid_simulations, game_status = game_status)

        # 2. save the fouls information, this will be necessary if when we need to change the fouls bonus limit at period last 2 minutes .
        list_limhteam = game_status.limhteam.copy()
        list_limvteam = game_status.limvteam.copy()
        list_limhtimeout = game_status.limhtimeout.copy()
        list_limvtimeout = game_status.limvtimeout.copy()

        # 3. In get_random_play_loop we sorted the event (play_token and spent_time) and updated the game_status
        # Only for simulations with keep_generating status at first in [1, 4].
        get_random_play_loop(period = period, game_status = game_status, valid_simulations = valid_simulations, 
                             token_model = token_model2, time_model = time_model2, class_model = class_model, 
                             config_setvariables = config_setvariables, match_vector = match_vector, 
                             matrix_end_period_probs = matrix_end_period_probs)

        # 4. Checking how much we have in remaining time and change the fouls bonus limit if necessary
        # Maybe I have to change here.
        if period == 4:
            change_timeouts_limits(game_status = game_status, valid_simulations = valid_simulations, 
                                  list_limhtimeout = list_limhtimeout, list_limvtimeout = list_limhtimeout)

        change_fouls_limit_two_final_minutes(game_status = game_status, valid_simulations = valid_simulations, 
                                             list_limhteam = list_limhteam, list_limvteam = list_limvteam)
        
        # 5. Print stats about teams' points and spent time percentual.
        print_status(game_status = game_status, period = period)

        # 6. Assign the simulations which reach a critical status, about to end the event generation process.
        idx_about_to_end_simulations = np.where(game_status.keep_generating == 2)[0]
        
        # 7. Update game log with current game status, only for those simulations who reach about_to_end status.
        if len(idx_about_to_end_simulations) > 0:
            game_log.add_events(indices = idx_about_to_end_simulations, game_status = game_status)

        # 8. Change the keep ganerating status for that ones with status 2. These ones we already generated one of end up event naturally.
        mask_kp = (game_status.keep_generating == 2)
        idx_kp = np.where(mask_kp)[0]
        game_status.keep_generating[idx_kp] = 0

        # 10. Update the valid simulations
        valid_simulations = np.where((game_status.keep_generating > 0) & (game_status.keep_generating < 6))[0]

    # Update valid_simulations only for generate the end period token for simulations which this is necessary.
    valid_simulations = np.where((game_status.keep_generating >= 6))[0]

    print_status(game_status = game_status, period = period)
    return 

def generate_game_plays_v2(class_model, token_model1, time_model1, num_simulations, token_context, num_numerical_variables, matrix_end_period_probs, token_model2 = None, time_model2 = None, config_setvariables = 0, match_vector = None):
    # Initiate game history and game context
    game_log = GameLog(n_sims = num_simulations)
    game_status = GameStatus.from_initial(n_sims = num_simulations, window = token_context, n_features = num_numerical_variables)

    # Define some basic variables
    period_duration = 7200
    period_limhteam = 4
    period_limvteam = 4
    game_limhtimeout = 7
    game_limvtimeout = 7

    for p in [1, 2, 3, 4]:
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        if p == 1:
            game_status.limhtimeout[:] = game_limhtimeout
            game_status.limvtimeout[:] = game_limvtimeout

        generate_period_plays_v2(period = p, class_model = class_model, token_model1 = token_model1, time_model1 = time_model1, 
                                 game_status = game_status, game_log = game_log, token_model2 = token_model2, time_model2 = time_model2, 
                                 config_setvariables = config_setvariables, match_vector = match_vector, 
                                 matrix_end_period_probs = matrix_end_period_probs)

    num_ties = sum(game_status.home_points == game_status.away_points)
    period_duration = 3000
    period_limhteam = 3 
    period_limvteam = 3 
    game_limhtimeout = 2 
    game_limvtimeout = 2 

    p = 5
    while num_ties > 0:
        
        game_status.remaining_time[:] = period_duration
        game_status.limhteam[:] = period_limhteam
        game_status.limvteam[:] = period_limvteam
        game_status.limhtimeout[:] = game_limhtimeout
        game_status.limvtimeout[:] = game_limvtimeout

        generate_period_plays_v2(period = p, class_model = class_model, token_model1 = token_model1, time_model1 = time_model1, 
                                 game_status = game_status, game_log = game_log, token_model2 = token_model2, time_model2 = time_model2, 
                                 config_setvariables = config_setvariables, match_vector = match_vector,
                                 matrix_end_period_probs = matrix_end_period_probs)
        
        p += 1
        num_ties = sum(game_status.home_points == game_status.away_points)

    return game_log.to_dataframe()



In [ ]:
def get_cdf_time_model(time_model, class_model, token_vector, numerical_vector, remaining_time, state_token_vector = None, time_diff_features = None):
    """
    Calcula a probabilidade da jogada caber no tempo restante (CDF) para qualquer arquitetura de modelo.
    """
    
    # 1. Usa a sua esteira de pré-processamento nativa! Magia pura.
    token_vector, numerical_vector, state_token_vector, time_diff_features, _ = preprocess_vectors_to_predict(
        class_model, token_vector, numerical_vector, state_token_vector, time_diff_features, mask_data = None
    )

    remaining_time = np.asarray(remaining_time)

    # 2. Forward pass limpo, exatamente como no seu predict_model_v2
    with torch.inference_mode():
        if class_model == "ExtraInfo":
            logits = time_model(train_token = token_vector, train_numerical = numerical_vector)
        elif class_model == "StateVector":
            logits = time_model(train_token = token_vector, train_numerical = numerical_vector, train_state = state_token_vector)
        elif class_model == "TimeDiffTower":
            logits = time_model(train_token = token_vector, time_diff_features = time_diff_features, train_numerical = numerical_vector)
        elif class_model == "MultiTower":
            logits = time_model(token_ids = token_vector, numerical_groups = [numerical_vector, time_diff_features])
        else:
            print("get a valid class model name")
            return None

        # 3. Transforma em probabilidade
        probs = torch.softmax(logits, dim = 1)
        
        # 4. Mágica do Broadcasting para a máscara de tempo
        num_classes = probs.size(1)
        classes = torch.arange(num_classes, device = probs.device).unsqueeze(0) # (1, num_classes)
        rem_time_tensor = torch.tensor(remaining_time, device = probs.device).unsqueeze(1) # (batch, 1)
        
        valid_mask = classes <= rem_time_tensor # True para tempo <= remaining_time
        
        # 5. Zera as probabilidades inválidas e soma as válidas
        prob_spent_time_fit_period_demand = (probs * valid_mask).sum(dim = 1).cpu().numpy()    
        
    return prob_spent_time_fit_period_demand

In [ ]:
def build_logit_mask(game_status, valid_simulations, mode, target, period, output_size):
    idx = np.asarray(valid_simulations)
    n_valid = len(idx)

    if n_valid == 0:
        return None

    if target == "token":
        dict_tokens = define_special_tokens(config = 1)

        if mode == "first_play":
            mask_1d = np.ones(output_size, dtype = bool)

            if period == 1:
                allowed = dict_tokens["start_period1"]
            elif period == 2:
                allowed = dict_tokens["start_period2"]
            elif period == 3:
                allowed = dict_tokens["start_period3"]
            elif period == 4:
                allowed = dict_tokens["start_period4"]
            else:
                allowed = dict_tokens["start_overtime"]

            mask_1d[allowed] = False
            mask_1d[0] = True
            return np.tile(mask_1d, (n_valid, 1))

        elif mode == "last_play":
            mask_1d = np.ones(output_size, dtype = bool)

            if period == 1:
                allowed = dict_tokens["end_period1"]
            elif period == 2:
                allowed = dict_tokens["end_period2"]
            elif period == 3:
                allowed = dict_tokens["end_period3"]
            elif period == 4:
                allowed = dict_tokens["end_period4"]
            else:
                allowed = dict_tokens["end_overtime"]

            mask_1d[allowed] = False
            mask_1d[0] = True
            return np.tile(mask_1d, (n_valid, 1))

        elif mode == "play_loop":
            mask = np.zeros((n_valid, output_size), dtype = bool)
            mask[:, 0] = True

            limhteam = game_status.limhteam[idx]
            limvteam = game_status.limvteam[idx]

            limhtimeout = game_status.limhtimeout[idx]
            limvtimeout = game_status.limvtimeout[idx]
            
            rows = np.where(limhteam >= 1)[0]
            if len(rows) > 0:
                mask[np.ix_(rows, dict_tokens["home_fouls_limhteam_p"])] = True

            rows = np.where(limhteam < 1)[0]
            if len(rows) > 0:
                mask[np.ix_(rows, dict_tokens["home_fouls_limhteam_a"])] = True

            rows = np.where(limvteam >= 1)[0]
            if len(rows) > 0:
                mask[np.ix_(rows, dict_tokens["away_fouls_limvteam_p"])] = True

            rows = np.where(limvteam < 1)[0]
            if len(rows) > 0:
                mask[np.ix_(rows, dict_tokens["away_fouls_limvteam_a"])] = True

            rows = np.where(limhtimeout < 1)[0]
            if len(rows) > 0:
                mask[np.ix_(rows, dict_tokens["timeouts_hometeam"])] = True

            rows = np.where(limvtimeout < 1)[0]
            if len(rows) > 0:
                mask[np.ix_(rows, dict_tokens["timeouts_awayteam"])] = True

            if period == 1:
                not_allowed = dict_tokens["not_allowed_period1"]
                mask[:, not_allowed] = True

            elif period == 2:
                not_allowed = dict_tokens["not_allowed_period2"]
                mask[:, not_allowed] = True
             
            elif period == 3:
                not_allowed = dict_tokens["not_allowed_period3"]
                mask[:, not_allowed] = True
               
            elif period == 4:
                not_allowed = dict_tokens["not_allowed_period4"]
                mask[:, not_allowed] = True
                
            else:
                not_allowed = dict_tokens["not_allowed_overtime"]
                mask[:, not_allowed] = True

            return mask

    elif target == "time":
        remaining_time = game_status.remaining_time[idx]

        if not np.any(remaining_time < 300):
            return None

        time_positions = np.arange(output_size)
        mask = (time_positions[None, :] > remaining_time[:, None]) & (remaining_time[:, None] < 300)

        return mask

    return None